In [ ]:

import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("GPU not connected yet")


In [ ]:

from google.colab import drive

drive.mount('/content/drive')


In [ ]:

from google.colab import drive
drive.mount('/content/drive')


In [ ]:

from pathlib import Path

project_dir = Path("/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction")

project_dir.mkdir(parents=True, exist_ok=True)

print("Project folder created:", project_dir)


In [ ]:

import numpy as np
from pathlib import Path

patient_dir = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction/Patient_1"
)

for filename in ["ct.npy", "ptv_contours.npy", "dose.npy"]:
    file_path = patient_dir / filename

    if not file_path.exists():
        print(f"NOT FOUND: {filename}")
        continue

    arr = np.load(file_path, mmap_mode="r", allow_pickle=False)

    print(f"\n{filename}")
    print("Shape:", arr.shape)
    print("Data type:", arr.dtype)
    print("Minimum:", np.min(arr))
    print("Maximum:", np.max(arr))
    print("Contains NaN:", np.isnan(arr).any())


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

ct = np.load(patient_dir / "ct.npy")
ptv = np.load(patient_dir / "ptv_contours.npy")
dose = np.load(patient_dir / "dose.npy")

slice_idx = 7  # Middle of the 15 saved slices

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(ct[slice_idx, 0], cmap="gray")
axes[0].set_title("CT - Slice 8")

axes[1].imshow(ct[slice_idx, 0], cmap="gray")
axes[1].imshow(
    np.ma.masked_where(
        ptv[slice_idx, 0] <= 0,
        ptv[slice_idx, 0]
    ),
    cmap="Reds",
    alpha=0.5
)
axes[1].set_title("CT + PTV contour")

axes[2].imshow(ct[slice_idx, 0], cmap="gray")
axes[2].imshow(
    np.ma.masked_where(
        dose[slice_idx, 0] <= 0,
        dose[slice_idx, 0]
    ),
    cmap="jet",
    alpha=0.5
)
axes[2].set_title("CT + Dose")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

print("Number of CT slices:", ct.shape[0])
print("Number of PTV slices:", ptv.shape[0])
print("Number of dose slices:", dose.shape[0])

print("PTV voxels per slice:")
for i in range(ptv.shape[0]):
    print(f"Slice {i+1}: {np.count_nonzero(ptv[i])}")


In [ ]:

import h5py
import json
from pathlib import Path

raw_dir = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction/Patient_1/raw_portpy"
)

print("=== ORIGINAL CT DATA ===")
with h5py.File(raw_dir / "CT_Data.h5", "r") as f:
    def show_ct(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(name, "Shape:", obj.shape, "Type:", obj.dtype)
    f.visititems(show_ct)

print("\n=== ORIGINAL STRUCTURE DATA ===")
with h5py.File(raw_dir / "StructureSet_Data.h5", "r") as f:
    def show_structures(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(name, "Shape:", obj.shape, "Type:", obj.dtype)
    f.visititems(show_structures)

print("\n=== CT METADATA ===")
with open(raw_dir / "CT_MetaData.json") as f:
    ct_metadata = json.load(f)
print(json.dumps(ct_metadata, indent=2)[:4000])

print("\n=== STRUCTURE METADATA ===")
with open(raw_dir / "StructureSet_MetaData.json") as f:
    structure_metadata = json.load(f)
print(json.dumps(structure_metadata, indent=2)[:4000])


In [ ]:

import h5py
import numpy as np

with h5py.File(raw_dir / "CT_Data.h5", "r") as f:
    ct_shape = f["ct_hu_3d"].shape

with h5py.File(raw_dir / "StructureSet_Data.h5", "r") as f:
    ptv_shape = f["PTV"].shape
    ptv_slices = np.array([
        np.count_nonzero(f["PTV"][i])
        for i in range(ptv_shape[0])
    ])

print("Original CT shape:", ct_shape)
print("Original PTV shape:", ptv_shape)

active = np.flatnonzero(ptv_slices)

print("Slices containing PTV:", len(active))
print("First PTV slice (0-based):", active[0])
print("Last PTV slice (0-based):", active[-1])
print("Total PTV voxels:", ptv_slices.sum())


In [ ]:

import numpy as np
import h5py
from scipy.ndimage import zoom

saved_ptv = np.load(patient_dir / "ptv_contours.npy")

with h5py.File(raw_dir / "StructureSet_Data.h5", "r") as f:
    original_ptv = f["PTV"][30:45]

resized_ptv = zoom(
    original_ptv.astype(np.float32),
    (1, 128/1024, 128/1024),
    order=0
)

saved_binary = saved_ptv[:, 0] > 0.5
resized_binary = resized_ptv > 0.5

intersection = np.logical_and(saved_binary, resized_binary).sum()
denominator = saved_binary.sum() + resized_binary.sum()

dice = 2 * intersection / denominator if denominator else 1.0

print("Saved shape:", saved_binary.shape)
print("Resized original shape:", resized_binary.shape)
print("Dice similarity:", round(float(dice), 4))
print("Saved PTV pixels:", saved_binary.sum())
print("Resized original PTV pixels:", resized_binary.sum())


In [ ]:

from skimage.transform import resize
import numpy as np

original = original_ptv.astype(bool)
saved = saved_ptv[:, 0] > 0.5

methods = {
    "Nearest-neighbor resize": resize(
        original.astype(float),
        (15, 128, 128),
        order=0,
        preserve_range=True,
        anti_aliasing=False
    ) > 0.5,

    "Area-based resize": resize(
        original.astype(float),
        (15, 128, 128),
        order=1,
        preserve_range=True,
        anti_aliasing=True
    ) > 0.5
}

for name, candidate in methods.items():
    intersection = np.logical_and(saved, candidate).sum()
    total = saved.sum() + candidate.sum()
    dice = 2 * intersection / total if total else 1.0

    print(name)
    print("Dice:", round(float(dice), 4))
    print("PTV pixels:", int(candidate.sum()))
    print()


In [ ]:
%cd /content
!git clone https://github.com/UJUNWAMGBOH/FluenceFormer.git
!ls FluenceFormer

In [ ]:
from pathlib import Path

repo = Path("/content/FluenceFormer")

for folder in ["src", "scripts"]:
    print(f"\n=== {folder.upper()} ===")
    for path in sorted((repo / folder).rglob("*.py")):
        print(path.relative_to(repo))

In [ ]:
from pathlib import Path

repo = Path("/content/FluenceFormer")

for name in ["builders.py", "custom_nets.py"]:
    path = repo / "src/prostate_fluence/models" / name

    print(f"\n{'=' * 20} {name} {'=' * 20}")
    print(path.read_text())

In [ ]:
from pathlib import Path

path = Path(
    "/content/FluenceFormer/src/prostate_fluence/dataset.py"
)

print(path.read_text())

In [ ]:
from pathlib import Path

path = Path(
    "/content/FluenceFormer/src/prostate_fluence/run_train_and_infer.py"
)

print(path.read_text())

In [ ]:
from pathlib import Path

root = Path("/content/FluenceFormer/src")

keywords = [
    "stage1",
    "stage_1",
    "dose_prediction",
    "dose_model",
    "predict_dose",
    "dose_net",
]

for path in sorted(root.rglob("*.py")):
    lines = path.read_text(errors="ignore").splitlines()

    matches = [
        (i, line.strip())
        for i, line in enumerate(lines, 1)
        if any(k in line.lower() for k in keywords)
    ]

    if matches:
        print(f"\n{path.relative_to(root)}")
        for line_no, line in matches:
            print(f"  Line {line_no}: {line}")

In [ ]:
from pathlib import Path

repo = Path("/content/FluenceFormer")

for filename in ["README.md", "instruction.md"]:
    path = repo / filename
    print(f"\n{'=' * 15} {filename} {'=' * 15}")
    print(path.read_text(errors="ignore"))

In [ ]:
from IPython.display import display, Image

display(
    Image(
        filename="/content/FluenceFormer/Figures/Architecture_overview.png",
        width=900
    )
)

In [ ]:
from pathlib import Path

path = Path("/content/FluenceFormer/src/prostate_fluence/config.py")
print(path.read_text())

In [ ]:
from pathlib import Path
import json

root = Path("/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction")

for path in sorted(root.rglob("*beam*json")):
    print("\nFILE:", path)
    print(path.read_text()[:5000])

In [ ]:
from pathlib import Path
import json

beam_file = Path(
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction/"
    "Patient_1/raw_portpy/PlannerBeams.json"
)

print("File found:", beam_file.exists())

if beam_file.exists():
    with open(beam_file) as f:
        beam_data = json.load(f)

    print(json.dumps(beam_data, indent=2)[:12000])

In [ ]:
import json
from pathlib import Path

beam_file = Path(
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction/"
    "Patient_1/raw_portpy/PlannerBeams.json"
)

with open(beam_file) as f:
    beam_data = json.load(f)

beam_ids = beam_data["IDs"]

print("Number of beams:", len(beam_ids))
print("Beam IDs:", beam_ids)

print("\nBeam IDs sorted numerically:")
print(sorted(beam_ids))

In [ ]:
from pathlib import Path

raw = Path(
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction/"
    "Patient_1/raw_portpy"
)

print("Files in raw_portpy:")
for p in sorted(raw.iterdir()):
    print(p.name)

In [ ]:
from pathlib import Path
import json

folder = Path(
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction/"
    "Patient_1/raw_portpy"
)

beam_ids = [36, 44, 52, 60, 68, 4, 12, 20, 28]

for beam_id in beam_ids:
    file = folder / f"Beam_{beam_id}_MetaData.json"

    print(f"\n{'=' * 15} BEAM {beam_id} {'=' * 15}")

    if file.exists():
        with open(file) as f:
            data = json.load(f)

        print(json.dumps(data, indent=2)[:4000])
    else:
        print("FILE NOT FOUND")

In [ ]:
from pathlib import Path
import json

folder = Path(
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction/"
    "Patient_1/raw_portpy"
)

beam_ids = [36, 44, 52, 60, 68, 4, 12, 20, 28]

for beam_id in beam_ids:
    file = folder / f"Beam_{beam_id}_MetaData.json"

    print(f"\n{'=' * 15} BEAM {beam_id} {'=' * 15}")

    if file.exists():
        with open(file) as f:
            data = json.load(f)

        print(json.dumps(data, indent=2)[:4000])
    else:
        print("FILE NOT FOUND")

In [ ]:
from pathlib import Path
import json

folder = Path(
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction/"
    "Patient_1/raw_portpy"
)

beam_ids = [36, 44, 52, 60, 68, 4, 12, 20, 28]

for beam_id in beam_ids:
    file = folder / f"Beam_{beam_id}_MetaData.json"

    print(f"\n{'=' * 15} BEAM {beam_id} {'=' * 15}")

    if file.exists():
        with open(file) as f:
            data = json.load(f)

        print(json.dumps(data, indent=2)[:4000])
    else:
        print("FILE NOT FOUND")

In [ ]:
from pathlib import Path

root = Path("/content/drive/MyDrive")

print("Searching Google Drive for beam metadata...")

files = sorted(root.rglob("Beam_*_MetaData.json"))

print("Total files found:", len(files))

for file in files:
    print(file)

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

In [ ]:
from pathlib import Path

folder = Path(
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction/"
    "Patient_1/raw_portpy"
)

print("Folder exists:", folder.exists())

if folder.exists():
    print("\nFiles:")
    for file in sorted(folder.iterdir()):
        print(file.name)

In [ ]:
from pathlib import Path
import json

folder = Path(
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction/"
    "Patient_1/raw_portpy"
)

beam_ids = [36, 44, 52, 60, 68, 4, 12, 20, 28]

for beam_id in beam_ids:
    path = folder / f"Beam_{beam_id}_MetaData.json"

    with open(path) as f:
        data = json.load(f)

    print(f"\n===== BEAM {beam_id} =====")
    print(json.dumps(data, indent=2)[:3000])

In [ ]:
import json
from pathlib import Path

root = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction"
)
raw = root / "Patient_1" / "raw_portpy"

with open(raw / "PlannerBeams.json") as f:
    planner = json.load(f)

beam_mapping = []

for beam_id in planner["IDs"]:
    with open(raw / f"Beam_{beam_id}_MetaData.json") as f:
        metadata = json.load(f)

    beam_mapping.append({
        "beam_id": beam_id,
        "gantry_angle": metadata["gantry_angle"]
    })

output = root / "Patient_1" / "verified_beam_mapping.json"

with open(output, "w") as f:
    json.dump(beam_mapping, f, indent=2)

print("Saved:", output)
print(json.dumps(beam_mapping, indent=2))

In [ ]:
from pathlib import Path

root = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction"
)

print("Searching for fluence files...\n")

files = sorted(
    p for p in root.rglob("*")
    if p.is_file() and "fluence" in p.name.lower()
)

print("Number of files found:", len(files))

for p in files:
    print(p.relative_to(root))

In [ ]:
from pathlib import Path

project = Path("/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction")

print("Project folder exists:", project.exists())

if project.exists():
    print("\nPROJECT CONTENTS:")
    for item in sorted(project.iterdir()):
        print(
            "FOLDER" if item.is_dir() else "FILE",
            item.name
        )
else:
    print("Project folder not found.")

In [ ]:
from pathlib import Path

print("Google Drive mounted:", Path("/content/drive/MyDrive").exists())
print("Drive directory exists:", Path("/content/drive").exists())

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pathlib import Path

mydrive = Path("/content/drive/MyDrive")
project = mydrive / "FluenceFormer_PortPy_Reproduction"

print("Google Drive accessible:", mydrive.exists())
print("Project folder exists:", project.exists())

print("\nGoogle Drive contents:")
for item in sorted(mydrive.iterdir()):
    print("FOLDER" if item.is_dir() else "FILE", item.name)

In [ ]:
from pathlib import Path

project = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction"
)

print("FLUENCEFORMER PROJECT CONTENTS")
print("-" * 55)

for path in sorted(project.rglob("*")):
    relative = path.relative_to(project)

    if len(relative.parts) <= 3:
        kind = "FOLDER" if path.is_dir() else "FILE"
        print(f"{kind:6} {relative}")

In [ ]:
import numpy as np
from pathlib import Path

folder = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction/Patient_1"
)

for filename in ["ct.npy", "dose.npy", "ptv_contours.npy"]:
    path = folder / filename

    print(f"\n{filename}")
    print("-" * 35)

    arr = np.load(path, mmap_mode="r")

    print("Shape:", arr.shape)
    print("Data type:", arr.dtype)
    print("Minimum:", float(np.min(arr)))
    print("Maximum:", float(np.max(arr)))

In [ ]:
from pathlib import Path

repo = Path("/content/FluenceFormer")

print("Original FluenceFormer repository exists:", repo.exists())

if repo.exists():
    print("\nRepository contents:")
    for item in sorted(repo.iterdir()):
        print("FOLDER" if item.is_dir() else "FILE", item.name)

In [ ]:
!git clone https://github.com/UJUNWAMGBOH/FluenceFormer.git /content/FluenceFormer

In [ ]:
from pathlib import Path

repo = Path("/content/FluenceFormer")

print("ORIGINAL FLUENCEFORMER PYTHON FILES")
print("-" * 50)

for file in sorted(repo.rglob("*.py")):
    print(file.relative_to(repo))

In [ ]:
from pathlib import Path

file = Path(
    "/content/FluenceFormer/src/prostate_fluence/run_train_and_infer.py"
)

lines = file.read_text(encoding="utf-8").splitlines()

print("Total lines:", len(lines))
print("\nTRAINING SCRIPT — FIRST 120 LINES")
print("-" * 55)

for i, line in enumerate(lines[:120], start=1):
    print(f"{i:3}: {line}")

In [ ]:
from pathlib import Path

file = Path(
    "/content/FluenceFormer/src/prostate_fluence/dataset.py"
)

lines = file.read_text(encoding="utf-8").splitlines()

print("DATASET IMPLEMENTATION")
print("-" * 55)

for i, line in enumerate(lines[:160], start=1):
    print(f"{i:3}: {line}")

In [ ]:
from pathlib import Path

file = Path(
    "/content/FluenceFormer/src/prostate_fluence/config.py"
)

print("ORIGINAL FLUENCEFORMER CONFIGURATION")
print("-" * 55)

for i, line in enumerate(file.read_text().splitlines(), start=1):
    print(f"{i:3}: {line}")

In [ ]:
from pathlib import Path

file = Path(
    "/content/FluenceFormer/src/prostate_fluence/config.py"
)

print("ORIGINAL FLUENCEFORMER CONFIGURATION")
print("-" * 55)

for i, line in enumerate(file.read_text().splitlines(), start=1):
    print(f"{i:3}: {line}")

In [ ]:
from pathlib import Path

repo = Path("/content/FluenceFormer")

print("Repository exists:", repo.exists())
print("Source folder exists:", (repo / "src").exists())
print("Config file exists:", (repo / "src/prostate_fluence/config.py").exists())

if repo.exists():
    print("\nRepository contents:")
    for item in repo.iterdir():
        print(item.name)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

project = Path("/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction")

print("Google Drive mounted:", Path("/content/drive/MyDrive").exists())
print("FluenceFormer project found:", project.exists())

In [ ]:
!git clone https://github.com/UJUNWAMGBOH/FluenceFormer.git /content/FluenceFormer

from pathlib import Path

config = Path("/content/FluenceFormer/src/prostate_fluence/config.py")

print("\nOriginal configuration file found:", config.exists())

In [ ]:
from pathlib import Path

file = Path(
    "/content/FluenceFormer/src/prostate_fluence/config.py"
)

print("ORIGINAL FLUENCEFORMER CONFIGURATION")
print("-" * 55)

for i, line in enumerate(file.read_text().splitlines(), start=1):
    print(f"{i:3}: {line}")

In [ ]:
import json
from pathlib import Path

mapping_file = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction/"
    "Patient_1/verified_beam_mapping.json"
)

print("Mapping file exists:", mapping_file.exists())

if mapping_file.exists():
    with open(mapping_file, "r") as f:
        mapping = json.load(f)

    print("\nVERIFIED PATIENT 1 BEAM MAPPING")
    print("-" * 50)
    print(json.dumps(mapping, indent=2))

In [ ]:
from pathlib import Path

project = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction"
)

print("SEARCHING FOR OPTIMIZED FLUENCE AND DOSE FILES")
print("-" * 55)

for file in sorted(project.rglob("*")):
    if file.is_file():
        name = file.name.lower()

        if (
            "fluence" in name
            or "optimal_intensity" in name
            or "optimized_dose" in name
        ):
            print(file.relative_to(project))

In [ ]:
from pathlib import Path
import numpy as np

folder = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction/"
    "Patient_1/optimized_portpy"
)

beam_ids = [36, 44, 52, 60, 68, 4, 12, 20, 28]

files = [
    "optimal_intensity.npy",
    "optimized_dose_1d.npy",
    "optimized_dose_3d.npy",
] + [f"beam_{bid}_fluence.npy" for bid in beam_ids]

print("Folder exists:", folder.exists())
print("-" * 60)

verified = 0

for name in files:
    path = folder / name

    if not path.exists():
        print(f"MISSING: {name}")
        continue

    arr = np.load(path, mmap_mode="r")
    print(f"OK: {name} | Shape: {arr.shape}")
    verified += 1

print("-" * 60)
print(f"Files verified: {verified}/{len(files)}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import numpy as np

folder = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction/"
    "Patient_1/optimized_portpy"
)

beam_ids = [36, 44, 52, 60, 68, 4, 12, 20, 28]

files = [
    "optimal_intensity.npy",
    "optimized_dose_1d.npy",
    "optimized_dose_3d.npy",
] + [f"beam_{bid}_fluence.npy" for bid in beam_ids]

print("Folder exists:", folder.exists())
print("-" * 60)

verified = 0

for name in files:
    path = folder / name

    if not path.exists():
        print(f"MISSING: {name}")
        continue

    arr = np.load(path, mmap_mode="r")
    print(f"OK: {name} | Shape: {arr.shape}")
    verified += 1

print("-" * 60)
print(f"Files verified: {verified}/{len(files)}")

In [ ]:
import numpy as np
from pathlib import Path

patient = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction/Patient_1"
)

for name in ["ct.npy", "dose.npy", "ptv_contours.npy"]:
    a = np.load(patient / name)
    print(name, "shape:", a.shape, "finite:", np.isfinite(a).all())

print("\nPTV voxels per slice:")
ptv = np.load(patient / "ptv_contours.npy")

for i in range(ptv.shape[0]):
    print(f"Slice {i:2d}: {np.count_nonzero(ptv[i])} PTV pixels")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

patient = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction/Patient_1"
)

ct = np.load(patient / "ct.npy")
dose = np.load(patient / "dose.npy")
ptv = np.load(patient / "ptv_contours.npy")

slice_index = 7

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(ct[slice_index, 0], cmap="gray")
axes[0].set_title("CT Scan")

axes[1].imshow(ct[slice_index, 0], cmap="gray")
axes[1].contour(
    ptv[slice_index, 0],
    levels=[0.5],
    colors="red"
)
axes[1].set_title("CT + PTV Contour")

im = axes[2].imshow(dose[slice_index, 0], cmap="jet")
axes[2].set_title("Radiation Dose")
fig.colorbar(im, ax=axes[2], label="Gy per fraction")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
from pathlib import Path

patient = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction/Patient_1"
)

dose = np.load(patient / "dose.npy")
ptv = np.load(patient / "ptv_contours.npy")

inside = dose[ptv > 0.5]
outside = dose[ptv <= 0.5]

print("PTV pixels:", inside.size)
print("Mean dose inside PTV:", inside.mean(), "Gy")
print("Mean dose outside PTV:", outside.mean(), "Gy")
print("Minimum dose inside PTV:", inside.min(), "Gy")
print("Maximum dose inside PTV:", inside.max(), "Gy")

In [ ]:
import h5py
from pathlib import Path

file = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction/"
    "Patient_1/raw_portpy/StructureSet_Data.h5"
)

print("Structure file exists:", file.exists())

if file.exists():
    with h5py.File(file, "r") as f:
        def show_item(name, obj):
            if isinstance(obj, h5py.Dataset):
                print(f"DATASET: {name} | Shape: {obj.shape}")
            else:
                print(f"GROUP: {name}")

        f.visititems(show_item)

In [ ]:
import h5py
import numpy as np
from pathlib import Path

file = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction/"
    "Patient_1/raw_portpy/StructureSet_Data.h5"
)

with h5py.File(file, "r") as f:
    ptv = f["PTV"]

    print("Original PTV shape:", ptv.shape)
    print("\nSlices containing PTV:")

    for i in range(ptv.shape[0]):
        count = np.count_nonzero(ptv[i])

        if count > 0:
            print(f"Slice {i}: {count} pixels")

In [ ]:
import h5py
import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path

patient = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction/Patient_1"
)

processed = np.load(patient / "ptv_contours.npy")[:, 0] > 0.5

raw_file = patient / "raw_portpy/StructureSet_Data.h5"

with h5py.File(raw_file, "r") as f:
    raw = np.asarray(f["PTV"][30:45]) > 0

raw_tensor = torch.from_numpy(raw.astype(np.float32))[:, None]

resized = F.interpolate(
    raw_tensor,
    size=(128, 128),
    mode="nearest"
)[:, 0].numpy() > 0.5

print("Slice | Dice score")
print("-" * 25)

scores = []

for i in range(15):
    a = processed[i]
    b = resized[i]

    intersection = np.logical_and(a, b).sum()
    dice = 2 * intersection / (a.sum() + b.sum())

    scores.append(dice)
    print(f"{i:5d} | {dice:.4f}")

print("-" * 25)
print("Mean Dice:", round(float(np.mean(scores)), 4))

In [ ]:
import h5py
import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path

patient = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction/Patient_1"
)

processed = np.load(patient / "ptv_contours.npy")[:, 0] > 0.5

with h5py.File(patient / "raw_portpy/StructureSet_Data.h5", "r") as f:
    raw = np.asarray(f["PTV"][30:45], dtype=np.float32)

raw_tensor = torch.from_numpy(raw[:, None])

print("RESIZING METHOD COMPARISON")
print("-" * 40)

for method in ["nearest", "nearest-exact"]:
    resized = F.interpolate(
        raw_tensor,
        size=(128, 128),
        mode=method
    )[:, 0].numpy() > 0.5

    scores = []

    for i in range(15):
        a = processed[i]
        b = resized[i]

        intersection = np.logical_and(a, b).sum()
        denominator = a.sum() + b.sum()

        dice = 2 * intersection / denominator if denominator else 1.0
        scores.append(dice)

    print(f"{method}: Mean Dice = {np.mean(scores):.4f}")

In [ ]:
import numpy as np
import h5py
import torch
import torch.nn.functional as F
from pathlib import Path

patient = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction/Patient_1"
)

processed = np.load(patient / "ptv_contours.npy")[7, 0] > 0.5

with h5py.File(patient / "raw_portpy/StructureSet_Data.h5", "r") as f:
    raw = np.asarray(f["PTV"][37], dtype=np.float32)

raw_tensor = torch.from_numpy(raw)[None, None]

resized = F.interpolate(
    raw_tensor,
    size=(128, 128),
    mode="nearest-exact"
)[0, 0].numpy() > 0.5

def mask_info(mask):
    y, x = np.where(mask)

    return {
        "pixels": len(x),
        "center_x": round(float(x.mean()), 2),
        "center_y": round(float(y.mean()), 2),
        "x_range": (int(x.min()), int(x.max())),
        "y_range": (int(y.min()), int(y.max()))
    }

print("PREPROCESSED PTV:")
print(mask_info(processed))

print("\nORIGINAL PTV RESIZED:")
print(mask_info(resized))

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np

# Use the masks from our previous cell
original = torch.tensor(resized.astype(np.float32))[None, None]

print("PTV MASK EXPANSION TEST")
print("-" * 40)

for kernel in [1, 3, 5, 7]:
    expanded = F.max_pool2d(
        original,
        kernel_size=kernel,
        stride=1,
        padding=kernel // 2
    )[0, 0].numpy() > 0.5

    intersection = np.logical_and(processed, expanded).sum()
    denominator = processed.sum() + expanded.sum()

    dice = 2 * intersection / denominator

    print(
        f"Kernel {kernel}x{kernel}: "
        f"Pixels = {expanded.sum()}, "
        f"Dice = {dice:.4f}"
    )

In [ ]:
import numpy as np
import h5py
import torch
import torch.nn.functional as F
from pathlib import Path

patient = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction/Patient_1"
)

processed = np.load(patient / "ptv_contours.npy")[:, 0] > 0.5

with h5py.File(patient / "raw_portpy/StructureSet_Data.h5", "r") as f:
    raw = np.asarray(f["PTV"][30:45], dtype=np.float32)

raw_tensor = torch.from_numpy(raw[:, None])

resized = F.interpolate(
    raw_tensor,
    size=(128, 128),
    mode="nearest-exact"
)

print("KERNEL | MEAN DICE | MIN DICE | MAX DICE")
print("-" * 45)

for kernel in [1, 3, 5, 7]:
    expanded = F.max_pool2d(
        resized,
        kernel_size=kernel,
        stride=1,
        padding=kernel // 2
    )[:, 0].numpy() > 0.5

    scores = []

    for i in range(15):
        a = processed[i]
        b = expanded[i]

        intersection = np.logical_and(a, b).sum()
        denominator = a.sum() + b.sum()
        dice = 2 * intersection / denominator if denominator else 1.0

        scores.append(dice)

    print(
        f"{kernel}x{kernel: <4} | "
        f"{np.mean(scores):.4f}    | "
        f"{np.min(scores):.4f}   | "
        f"{np.max(scores):.4f}"
    )

In [ ]:
import numpy as np
import h5py
import torch
import torch.nn.functional as F
from pathlib import Path

patient = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction/Patient_1"
)

processed = np.load(patient / "ptv_contours.npy")[:, 0] > 0.5

with h5py.File(patient / "raw_portpy/StructureSet_Data.h5", "r") as f:
    raw = np.asarray(f["PTV"][30:45], dtype=np.float32)

raw_tensor = torch.from_numpy(raw[:, None])

bilinear = F.interpolate(
    raw_tensor,
    size=(128, 128),
    mode="bilinear",
    align_corners=False
)[:, 0].numpy()

print("THRESHOLD | MEAN DICE | MIN DICE")
print("-" * 40)

for threshold in [0.01, 0.1, 0.25, 0.5]:
    resized = bilinear > threshold

    scores = []

    for i in range(15):
        a = processed[i]
        b = resized[i]

        intersection = np.logical_and(a, b).sum()
        denominator = a.sum() + b.sum()

        dice = 2 * intersection / denominator if denominator else 1.0
        scores.append(dice)

    print(
        f"{threshold:9.2f} | "
        f"{np.mean(scores):.4f}    | "
        f"{np.min(scores):.4f}"
    )

In [ ]:
import numpy as np
import h5py
from scipy.ndimage import zoom
from pathlib import Path

patient = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction/Patient_1"
)

processed = np.load(patient / "ptv_contours.npy")[:, 0] > 0.5

with h5py.File(patient / "raw_portpy/StructureSet_Data.h5", "r") as f:
    raw = np.asarray(f["PTV"][30:45]) > 0

resized = np.stack([
    zoom(mask.astype(np.float32), (128/1024, 128/1024), order=0) > 0.5
    for mask in raw
])

scores = []

for i in range(15):
    a = processed[i]
    b = resized[i]

    intersection = np.logical_and(a, b).sum()
    denominator = a.sum() + b.sum()

    dice = 2 * intersection / denominator if denominator else 1.0
    scores.append(dice)

print("SciPy zoom mean Dice:", round(float(np.mean(scores)), 4))
print("Minimum Dice:", round(float(np.min(scores)), 4))
print("Maximum Dice:", round(float(np.max(scores)), 4))
print("Slice 7 resized PTV pixels:", int(resized[7].sum()))
print("Slice 7 processed PTV pixels:", int(processed[7].sum()))

In [ ]:
import numpy as np
from pathlib import Path

patient = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction/Patient_1"
)

ptv = np.load(patient / "ptv_contours.npy")

print("PTV shape:", ptv.shape)
print("PTV data type:", ptv.dtype)
print("Minimum value:", ptv.min())
print("Maximum value:", ptv.max())
print("Unique values:", np.unique(ptv))

In [ ]:
import json
from pathlib import Path

patient = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction/Patient_1"
)

metadata_path = (
    patient / "raw_portpy" / "StructureSet_MetaData.json"
)

with open(metadata_path, "r") as f:
    metadata = json.load(f)

print("Metadata type:", type(metadata).__name__)

if isinstance(metadata, dict):
    print("\nTop-level keys:")
    for key in metadata:
        print("-", key)

    print("\nEntries containing PTV:")
    for key, value in metadata.items():
        if "PTV" in str(key).upper():
            print(key, ":", str(value)[:1000])
else:
    print(str(metadata)[:2000])

In [ ]:
import h5py
import numpy as np
from pathlib import Path

patient = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction/Patient_1"
)

raw_file = patient / "raw_portpy/StructureSet_Data.h5"

with h5py.File(raw_file, "r") as f:
    ptv = np.asarray(f["PTV"][37]) > 0
    ctv = np.asarray(f["CTV"][37]) > 0

print("ORIGINAL PORTPY — SLICE 37")
print("-" * 40)
print("PTV pixels:", int(ptv.sum()))
print("CTV pixels:", int(ctv.sum()))
print("PTV / CTV size ratio:", round(ptv.sum() / ctv.sum(), 3))
print("CTV pixels outside PTV:", int(np.logical_and(ctv, ~ptv).sum()))

In [ ]:
import h5py
from pathlib import Path

patient = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction/Patient_1"
)

ct_file = patient / "raw_portpy/CT_Data.h5"

with h5py.File(ct_file, "r") as f:
    print("ORIGINAL CT DATA")
    print("-" * 40)

    for name, dataset in f.items():
        print("Dataset:", name)
        print("Shape:", dataset.shape)
        print("Data type:", dataset.dtype)
        print()

In [ ]:
import json
from pathlib import Path

patient = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction/Patient_1"
)

metadata_path = patient / "raw_portpy/CT_MetaData.json"

with open(metadata_path, "r") as f:
    metadata = json.load(f)

print("CT METADATA")
print("-" * 45)
print("Metadata type:", type(metadata).__name__)

if isinstance(metadata, dict):
    for key, value in metadata.items():
        print(f"{key}: {str(value)[:500]}")
else:
    for item in metadata[:10]:
        print(item)

In [ ]:
import json
from pathlib import Path

patient = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction/Patient_1"
)

metadata_path = patient / "raw_portpy/CT_MetaData.json"

with open(metadata_path, "r") as f:
    ct_metadata = json.load(f)

print("CT SPATIAL INFORMATION")
print("-" * 40)

for key in [
    "origin_xyz_mm",
    "resolution_xyz_mm",
    "size_xyz_mm",
    "direction"
]:
    print(f"{key}: {ct_metadata.get(key)}")

print("\nNEXT: We need to verify how PortPy maps")
print("the optimized dose and PTV to this CT grid.")

In [ ]:
import numpy as np
import h5py
from pathlib import Path

patient = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction/Patient_1"
)

dose = np.load(
    patient / "optimized_portpy/optimized_dose_3d.npy",
    mmap_mode="r"
)

with h5py.File(patient / "raw_portpy/StructureSet_Data.h5", "r") as f:
    ptv = np.asarray(f["PTV"][37]) > 0

dose_slice = np.asarray(dose[37])

print("ORIGINAL SLICE 37")
print("-" * 40)
print("Dose shape:", dose_slice.shape)
print("PTV shape:", ptv.shape)
print("Mean dose inside PTV:", round(float(dose_slice[ptv].mean()), 4))
print("Mean dose outside PTV:", round(float(dose_slice[~ptv].mean()), 4))
print("Maximum dose:", round(float(dose_slice.max()), 4))

In [ ]:
import h5py
import numpy as np
from pathlib import Path

patient = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction/Patient_1"
)

with h5py.File(
    patient / "raw_portpy/StructureSet_Data.h5", "r"
) as f:
    ptv = np.asarray(f["PTV"][37]) > 0
    body = np.asarray(f["BODY"][37]) > 0

print("ANATOMICAL CONSISTENCY — SLICE 37")
print("-" * 45)

print("PTV pixels:", int(ptv.sum()))
print("BODY pixels:", int(body.sum()))

outside = np.logical_and(ptv, ~body).sum()

print("PTV pixels outside BODY:", int(outside))
print(
    "Percentage of PTV inside BODY:",
    round(100 * (1 - outside / ptv.sum()), 2)
)

In [ ]:
import h5py
import numpy as np
from pathlib import Path

patient = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction/Patient_1"
)

with h5py.File(
    patient / "raw_portpy/StructureSet_Data.h5", "r"
) as f:
    ptv = f["PTV"]

    counts = np.array([
        np.count_nonzero(ptv[i])
        for i in range(ptv.shape[0])
    ])

active_slices = np.where(counts > 0)[0]

print("PTV SLICE COVERAGE")
print("-" * 40)
print("First PTV slice:", active_slices.min())
print("Last PTV slice:", active_slices.max())
print("Number of PTV slices:", len(active_slices))

selected = np.arange(30, 45)

print("Selected slices:", selected[0], "to", selected[-1])
print(
    "All PTV slices included:",
    np.array_equal(active_slices, selected)
)

In [ ]:
from pathlib import Path

project = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction"
)

stage1_dir = project / "Stage1_Verified_Preprocessing"

stage1_dir.mkdir(parents=True, exist_ok=True)

print("Stage 1 preprocessing folder:")
print(stage1_dir)
print("Folder exists:", stage1_dir.exists())

In [ ]:
import numpy as np
import h5py
from pathlib import Path

patient = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction/Patient_1"
)

dose_path = patient / "optimized_portpy/optimized_dose_3d.npy"

with h5py.File(patient / "raw_portpy/CT_Data.h5", "r") as ct_file, \
     h5py.File(patient / "raw_portpy/StructureSet_Data.h5", "r") as structure_file:

    ct = ct_file["ct_hu_3d"]
    ptv = structure_file["PTV"]
    dose = np.load(dose_path, mmap_mode="r")

    print("ORIGINAL DATA VERIFICATION")
    print("-" * 45)

    print("CT shape:", ct.shape)
    print("PTV shape:", ptv.shape)
    print("Dose shape:", dose.shape)

    print("\nAll shapes match:", ct.shape == ptv.shape == dose.shape)

    print("\nSLICE 37 VALUES")
    print("CT range:", np.min(ct[37]), "to", np.max(ct[37]))
    print("PTV unique values:", np.unique(ptv[37]))
    print("Dose range:", np.min(dose[37]), "to", np.max(dose[37]))

In [ ]:
import numpy as np
import h5py
import torch
import torch.nn.functional as F
from pathlib import Path

patient = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction/Patient_1"
)

# Load the same original anatomical slice
with h5py.File(patient / "raw_portpy/CT_Data.h5", "r") as f:
    ct = np.asarray(f["ct_hu_3d"][37], dtype=np.float32)

with h5py.File(patient / "raw_portpy/StructureSet_Data.h5", "r") as f:
    ptv = np.asarray(f["PTV"][37], dtype=np.float32)

dose = np.load(
    patient / "optimized_portpy/optimized_dose_3d.npy",
    mmap_mode="r"
)

dose_slice = np.asarray(dose[37], dtype=np.float32)

def resize_slice(image, mode):
    tensor = torch.from_numpy(image.copy())[None, None]

    kwargs = {"align_corners": False} if mode == "bilinear" else {}

    result = F.interpolate(
        tensor,
        size=(128, 128),
        mode=mode,
        **kwargs
    )

    return result[0, 0].numpy()

ct_128 = resize_slice(ct, "bilinear")
ptv_128 = resize_slice(ptv, "nearest-exact")
dose_128 = resize_slice(dose_slice, "bilinear")

print("RESIZED SLICE 37")
print("-" * 40)
print("CT:", ct_128.shape)
print("PTV:", ptv_128.shape)
print("Dose:", dose_128.shape)

print("\nPTV unique values:", np.unique(ptv_128))
print("PTV pixels:", int(ptv_128.sum()))

print("\nCT range:", ct_128.min(), "to", ct_128.max())
print("Dose range:", dose_128.min(), "to", dose_128.max())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# CT with PTV outline
axes[0].imshow(ct_128, cmap="gray")
axes[0].contour(ptv_128, levels=[0.5], colors="red")
axes[0].set_title("CT + PTV Outline")

# PTV mask
axes[1].imshow(ptv_128, cmap="gray")
axes[1].set_title("PTV Mask")

# Radiation dose with PTV outline
im = axes[2].imshow(dose_128, cmap="inferno")
axes[2].contour(ptv_128, levels=[0.5], colors="cyan")
axes[2].set_title("Radiation Dose + PTV")

for ax in axes:
    ax.axis("off")

fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

inside = dose_128[ptv_128 > 0.5]
outside = dose_128[ptv_128 <= 0.5]

print("RESIZED SLICE 37 — DOSE CHECK")
print("-" * 40)

print("Mean dose inside PTV:", round(float(inside.mean()), 4))
print("Mean dose outside PTV:", round(float(outside.mean()), 4))
print("Minimum dose inside PTV:", round(float(inside.min()), 4))
print("Maximum dose inside PTV:", round(float(inside.max()), 4))
print("PTV pixels:", len(inside))

In [ ]:
import numpy as np
import h5py
import torch
import torch.nn.functional as F
from pathlib import Path

patient = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction/Patient_1"
)

output_dir = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction/"
    "Stage1_Verified_Preprocessing"
)

output_dir.mkdir(parents=True, exist_ok=True)

dose_volume = np.load(
    patient / "optimized_portpy/optimized_dose_3d.npy",
    mmap_mode="r"
)

def resize_image(image, mode):
    tensor = torch.from_numpy(
        np.asarray(image, dtype=np.float32).copy()
    )[None, None]

    kwargs = {"align_corners": False} if mode == "bilinear" else {}

    resized = F.interpolate(
        tensor,
        size=(128, 128),
        mode=mode,
        **kwargs
    )

    return resized[0, 0].numpy()

ct_slices = []
ptv_slices = []
dose_slices = []

with h5py.File(patient / "raw_portpy/CT_Data.h5", "r") as ct_file, \
     h5py.File(patient / "raw_portpy/StructureSet_Data.h5", "r") as structure_file:

    for slice_index in range(30, 45):

        ct = ct_file["ct_hu_3d"][slice_index]
        ptv = structure_file["PTV"][slice_index]
        dose = dose_volume[slice_index]

        ct_slices.append(resize_image(ct, "bilinear"))
        ptv_slices.append(resize_image(ptv, "nearest-exact"))
        dose_slices.append(resize_image(dose, "bilinear"))

ct_array = np.stack(ct_slices)[:, None]
ptv_array = np.stack(ptv_slices)[:, None]
dose_array = np.stack(dose_slices)[:, None]

assert ct_array.shape == (15, 1, 128, 128)
assert ptv_array.shape == (15, 1, 128, 128)
assert dose_array.shape == (15, 1, 128, 128)
assert np.all(np.isin(ptv_array, [0, 1]))

np.save(output_dir / "ct_stage1.npy", ct_array)
np.save(output_dir / "ptv_stage1.npy", ptv_array)
np.save(output_dir / "dose_stage1.npy", dose_array)

print("STAGE 1 DATASET CREATED")
print("-" * 45)
print("CT:", ct_array.shape)
print("PTV:", ptv_array.shape)
print("Dose:", dose_array.shape)
print("Saved to:", output_dir)

In [ ]:
import numpy as np
from pathlib import Path

folder = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction/"
    "Stage1_Verified_Preprocessing"
)

print("SAVED STAGE 1 FILE VERIFICATION")
print("-" * 45)

for filename in [
    "ct_stage1.npy",
    "ptv_stage1.npy",
    "dose_stage1.npy"
]:
    path = folder / filename

    data = np.load(path)

    print("\nFile:", filename)
    print("Exists:", path.exists())
    print("Shape:", data.shape)
    print("Data type:", data.dtype)
    print("Minimum:", float(data.min()))
    print("Maximum:", float(data.max()))
    print("Contains NaN:", bool(np.isnan(data).any()))
    print("Contains infinity:", bool(np.isinf(data).any()))

    assert data.shape == (15, 1, 128, 128)
    assert np.isfinite(data).all()

print("\nALL SAVED FILES VERIFIED SUCCESSFULLY")

In [ ]:
from pathlib import Path

repo = Path("/content/FluenceFormer")

print("ORIGINAL FLUENCEFORMER REPOSITORY")
print("-" * 45)

print("Repository exists:", repo.exists())

if repo.exists():
    print("\nTop-level contents:")
    for item in sorted(repo.iterdir()):
        print("-", item.name)
else:
    print("\nRepository needs to be cloned again.")

In [ ]:
!git clone https://github.com/UJUNWAMGBOH/FluenceFormer.git /content/FluenceFormer

In [ ]:
from pathlib import Path

repo = Path("/content/FluenceFormer")

print("FLUENCEFORMER PYTHON FILES")
print("-" * 55)

for file in sorted(repo.rglob("*.py")):
    print(file.relative_to(repo))

In [ ]:
from pathlib import Path

file = Path(
    "/content/FluenceFormer/src/prostate_fluence/models/builders.py"
)

print("ORIGINAL FLUENCEFORMER MODEL BUILDERS")
print("=" * 60)

print(file.read_text())

In [ ]:
from pathlib import Path

file = Path(
    "/content/FluenceFormer/src/prostate_fluence/dataset.py"
)

print("ORIGINAL FLUENCEFORMER DATASET")
print("=" * 60)

print(file.read_text())

In [ ]:
from pathlib import Path

file = Path(
    "/content/FluenceFormer/src/prostate_fluence/run_train_and_infer.py"
)

print("ORIGINAL FLUENCEFORMER TRAINING SCRIPT")
print("=" * 60)

for number, line in enumerate(
    file.read_text().splitlines(), start=1
):
    print(f"{number:4}: {line}")

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import numpy as np
from pathlib import Path

folder = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction/"
    "Stage1_Verified_Preprocessing"
)

print("FLUENCEFORMER STAGE 1 DATA")
print("-" * 45)

for filename in [
    "ct_stage1.npy",
    "ptv_stage1.npy",
    "dose_stage1.npy"
]:
    path = folder / filename

    if path.exists():
        data = np.load(path)
        print(f"{filename}: {data.shape} - READY")
    else:
        print(f"{filename}: MISSING")

In [ ]:
import torch

print("STAGE 1 GPU CHECK")
print("-" * 40)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1e9,
            2
        ),
        "GB"
    )
else:
    print("No GPU detected.")
    print("Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
import numpy as np
import torch
from pathlib import Path

folder = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction/"
    "Stage1_Verified_Preprocessing"
)

# Load verified data
ct = np.load(folder / "ct_stage1.npy")
ptv = np.load(folder / "ptv_stage1.npy")
dose = np.load(folder / "dose_stage1.npy")

# Normalize CT using the original repository's HU range
ct_normalized = np.clip((ct + 1000) / 4000, 0, 1)

# PTV is already binary
ptv_normalized = ptv

# Normalize PortPy dose using a documented 3 Gy/fraction scale
dose_normalized = dose / 3.0

# Combine CT and PTV into two input channels
X = np.concatenate(
    [ct_normalized, ptv_normalized],
    axis=1
)

# Ground-truth dose is the training target
Y = dose_normalized

# Convert to PyTorch tensors
X = torch.tensor(X, dtype=torch.float32)
Y = torch.tensor(Y, dtype=torch.float32)

print("STAGE 1 TRAINING DATA")
print("-" * 45)
print("Input X:", X.shape)
print("Target Y:", Y.shape)
print("CT normalized range:",
      float(X[:, 0].min()), float(X[:, 0].max()))
print("PTV range:",
      float(X[:, 1].min()), float(X[:, 1].max()))
print("Dose normalized range:",
      float(Y.min()), float(Y.max()))

In [ ]:
import importlib.util

print("STAGE 1 MODEL DEPENDENCY CHECK")
print("-" * 45)

if importlib.util.find_spec("monai") is not None:
    import monai
    print("MONAI installed:", True)
    print("MONAI version:", monai.__version__)
else:
    print("MONAI installed:", False)
    print("MONAI needs to be installed before building SwinUNETR.")

In [ ]:
%pip install -q "monai==1.5.1"

In [ ]:
import monai
from monai.networks.nets import SwinUNETR

print("MONAI INSTALLATION CHECK")
print("-" * 40)
print("MONAI version:", monai.__version__)
print("SwinUNETR imported successfully!")

In [ ]:
import torch
from monai.networks.nets import SwinUNETR

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

stage1_model = SwinUNETR(
    in_channels=2,
    out_channels=1,
    feature_size=24,
    spatial_dims=2,
    use_checkpoint=True
).to(device)

print("STAGE 1 SWINUNETR MODEL")
print("-" * 45)
print("Device:", device)
print("Input channels: 2 (CT + PTV)")
print("Output channels: 1 (Dose)")
print("Feature size: 24")
print(
    "Trainable parameters:",
    f"{sum(p.numel() for p in stage1_model.parameters() if p.requires_grad):,}"
)

In [ ]:
import torch

print("STAGE 1 MODEL FORWARD-PASS TEST")
print("-" * 45)

stage1_model.eval()

# Take one CT + PTV sample
sample_input = X[0:1].to(device)

with torch.no_grad():
    sample_prediction = torch.relu(
        stage1_model(sample_input)
    )

print("Input shape:", sample_input.shape)
print("Prediction shape:", sample_prediction.shape)
print("Minimum predicted dose:",
      float(sample_prediction.min()))
print("Maximum predicted dose:",
      float(sample_prediction.max()))

assert sample_prediction.shape == (1, 1, 128, 128)

print("\nFORWARD PASS SUCCESSFUL!")

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# Prepare our 15 training samples
train_dataset = TensorDataset(X, Y)

train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True
)

# Mean Squared Error: difference between predicted and actual dose
criterion = torch.nn.MSELoss()

# Adam optimizer adjusts the model's parameters
optimizer = torch.optim.Adam(
    stage1_model.parameters(),
    lr=1e-4
)

print("STAGE 1 TRAINING SETUP")
print("-" * 45)
print("Training samples:", len(train_dataset))
print("Batch size:", train_loader.batch_size)
print("Training batches:", len(train_loader))
print("Loss function: MSE")
print("Optimizer: Adam")
print("Learning rate: 0.0001")
print("Device:", device)

assert len(train_dataset) == 15
assert len(train_loader) == 8

print("\nTRAINING SETUP SUCCESSFUL!")

In [ ]:
import torch
import torch.nn.functional as F

print("FLUENCEFORMER STAGE 1 — FIRST TRAINING RUN")
print("=" * 55)

stage1_model.train()

training_losses = []

for epoch in range(5):

    total_loss = 0.0

    for batch_X, batch_Y in train_loader:

        batch_X = batch_X.to(device)
        batch_Y = batch_Y.to(device)

        optimizer.zero_grad()

        # Predict normalized radiation dose
        predictions = F.relu(stage1_model(batch_X))

        # Compare prediction with actual normalized dose
        loss = criterion(predictions, batch_Y)

        # Update the model's parameters
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)
    training_losses.append(average_loss)

    print(
        f"Epoch {epoch + 1}/5 "
        f"| Training MSE: {average_loss:.6f}",
        flush=True
    )

print("\nFIRST STAGE 1 TRAINING RUN COMPLETED!")

In [ ]:
import torch
from pathlib import Path

save_dir = Path(
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction/"
    "Stage1_Checkpoints"
)

save_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = save_dir / "swinunetr_stage1_patient1_epoch5.pth"

torch.save(
    {
        "epoch": 5,
        "model_state_dict": stage1_model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "training_losses": training_losses,
        "architecture": "SwinUNETR",
        "input_channels": 2,
        "output_channels": 1,
        "feature_size": 24,
        "dose_scale_gy": 3.0,
        "learning_rate": 1e-4,
        "training_patient": "Patient_1",
    },
    checkpoint_path
)

print("STAGE 1 CHECKPOINT SAVED")
print("-" * 45)
print("File:", checkpoint_path.name)
print("Exists:", checkpoint_path.exists())
print(
    "File size:",
    round(checkpoint_path.stat().st_size / (1024**2), 2),
    "MB"
)
print("Epochs completed:", 5)
print("Final training MSE:", training_losses[-1])

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

stage1_model.eval()

# Select the middle slice (slice 37 in original CT)
slice_index = 7

with torch.no_grad():
    prediction = torch.relu(
        stage1_model(X[slice_index:slice_index+1].to(device))
    )

# Convert normalized doses back to Gy
predicted_dose = prediction[0, 0].cpu().numpy() * 3.0
actual_dose = Y[slice_index, 0].numpy() * 3.0

# Recover CT values in Hounsfield units
ct_image = X[slice_index, 0].numpy() * 4000 - 1000
ptv_mask = X[slice_index, 1].numpy()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(ct_image, cmap="gray", vmin=-1000, vmax=1000)
axes[0].contour(ptv_mask, levels=[0.5], colors="red")
axes[0].set_title("CT + PTV Target")

axes[1].imshow(actual_dose, cmap="jet", vmin=0, vmax=3)
axes[1].set_title("Actual PortPy Dose (Gy)")

axes[2].imshow(predicted_dose, cmap="jet", vmin=0, vmax=3)
axes[2].set_title("Predicted Stage 1 Dose (Gy)")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

mse = np.mean((predicted_dose - actual_dose) ** 2)
mae = np.mean(np.abs(predicted_dose - actual_dose))

print("SLICE 37 RESULTS")
print("-" * 40)
print("MSE (Gy²):", round(float(mse), 6))
print("MAE (Gy):", round(float(mae), 6))
print("Actual maximum dose (Gy):", round(float(actual_dose.max()), 4))
print("Predicted maximum dose (Gy):", round(float(predicted_dose.max()), 4))

In [ ]:
import numpy as np

inside = ptv_mask > 0.5

actual_inside = actual_dose[inside]
predicted_inside = predicted_dose[inside]

print("STAGE 1 — PTV DOSE COMPARISON")
print("-" * 45)

print("Actual mean PTV dose (Gy):",
      round(float(actual_inside.mean()), 4))

print("Predicted mean PTV dose (Gy):",
      round(float(predicted_inside.mean()), 4))

print("Actual minimum PTV dose (Gy):",
      round(float(actual_inside.min()), 4))

print("Predicted minimum PTV dose (Gy):",
      round(float(predicted_inside.min()), 4))

print("PTV mean absolute error (Gy):",
      round(float(np.mean(
          np.abs(actual_inside - predicted_inside)
      )), 4))

print("PTV pixels:", int(inside.sum()))

In [ ]:
import torch
import torch.nn.functional as F

stage1_model.eval()

with torch.no_grad():
    raw_output = stage1_model(
        X[7:8].to(device)
    )

    relu_output = F.relu(raw_output)

ptv = X[7, 1].to(device) > 0.5

raw_ptv = raw_output[0, 0][ptv]
relu_ptv = relu_output[0, 0][ptv]

print("STAGE 1 — RELU INVESTIGATION")
print("-" * 45)

print("Raw minimum inside PTV:",
      round(float(raw_ptv.min()), 6))

print("Raw maximum inside PTV:",
      round(float(raw_ptv.max()), 6))

print("Raw mean inside PTV:",
      round(float(raw_ptv.mean()), 6))

print("PTV pixels with negative raw predictions:",
      int((raw_ptv < 0).sum().item()))

print("Total PTV pixels:",
      int(ptv.sum().item()))

print("Mean after ReLU:",
      round(float(relu_ptv.mean()), 6))

In [ ]:
import torch

stage1_model.train()

batch_x = X[7:8].to(device)
batch_y = Y[7:8].to(device)

optimizer.zero_grad()

raw = stage1_model(batch_x)
raw.retain_grad()

prediction = torch.relu(raw)
loss = criterion(prediction, batch_y)

loss.backward()

ptv_pixels = batch_x[0, 1] > 0.5
ptv_gradients = raw.grad[0, 0][ptv_pixels]

print("STAGE 1 — PTV GRADIENT CHECK")
print("-" * 45)
print("Loss:", round(float(loss.item()), 6))
print("PTV pixels:", int(ptv_pixels.sum()))
print("PTV pixels with nonzero output gradient:",
      int((ptv_gradients != 0).sum()))
print("Maximum absolute PTV gradient:",
      float(ptv_gradients.abs().max()))

optimizer.zero_grad()

In [ ]:
print("STAGE 1 — OUTPUT HEAD INSPECTION")
print("-" * 50)

print("Output head:")
print(stage1_model.out)

print("\nOutput head parameters:")
for name, param in stage1_model.out.named_parameters():
    print(name, tuple(param.shape))

print("\nTraining checkpoint remains unchanged.")

In [ ]:
import torch

print("STAGE 1 — OUTPUT LAYER WEIGHTS")
print("-" * 50)

output_conv = stage1_model.out.conv.conv

with torch.no_grad():
    weights = output_conv.weight
    bias = output_conv.bias

    print("Weight minimum:", float(weights.min()))
    print("Weight maximum:", float(weights.max()))
    print("Weight mean:", float(weights.mean()))
    print("Weight standard deviation:",
          float(weights.std()))

    print("\nBias:", float(bias.item()))

    print("\nAny NaN weights:",
          bool(torch.isnan(weights).any()))

    print("Any infinite weights:",
          bool(torch.isinf(weights).any()))

In [ ]:
print("STAGE 1 — ARCHITECTURE VERIFICATION")
print("-" * 50)

print("Model:", type(stage1_model).__name__)

print("\nInput convolution:")
print(stage1_model.swinViT.patch_embed.proj)

print("\nOutput convolution:")
print(stage1_model.out)

print("\nNumber of trainable parameters:")
print(sum(
    p.numel()
    for p in stage1_model.parameters()
    if p.requires_grad
))

print("\nCurrent training configuration:")
print("Loss:", type(criterion).__name__)
print("Optimizer:", type(optimizer).__name__)
print("Learning rate:", optimizer.param_groups[0]["lr"])
print("Completed epochs:", len(training_losses))

In [ ]:
import torch

print("STAGE 1 — TRAINING DATA VERIFICATION")
print("-" * 50)

print("Input shape:", tuple(X.shape))
print("Target shape:", tuple(Y.shape))

print("\nCT channel:")
print("Min:", float(X[:, 0].min()))
print("Max:", float(X[:, 0].max()))

print("\nPTV channel:")
print("Min:", float(X[:, 1].min()))
print("Max:", float(X[:, 1].max()))

print("\nDose target (normalized):")
print("Min:", float(Y.min()))
print("Max:", float(Y.max()))

ptv_all = X[:, 1] > 0.5
dose_all = Y[:, 0]

print("\nDose inside PTV:")
print("Mean:", float(dose_all[ptv_all].mean()))
print("Min:", float(dose_all[ptv_all].min()))
print("Max:", float(dose_all[ptv_all].max()))

print("\nFinite inputs:", bool(torch.isfinite(X).all()))
print("Finite targets:", bool(torch.isfinite(Y).all()))

In [ ]:
print("STAGE 1 — SWIN TRANSFORMER CONFIGURATION")
print("-" * 50)

swin = stage1_model.swinViT

print("Patch embedding:")
print(swin.patch_embed)

print("\nTransformer layers:")
for name in ["layers1", "layers2", "layers3", "layers4"]:
    layers = getattr(swin, name)
    print(f"\n{name}:")
    print("Number of blocks:", len(layers[0].blocks))

    first_block = layers[0].blocks[0]
    print("Attention heads:", first_block.attn.num_heads)
    print("Window size:", first_block.window_size)

print("\nModel feature size: 24")

In [ ]:
print("STAGE 1 — SWIN TRANSFORMER CONFIGURATION")
print("-" * 50)

swin = stage1_model.swinViT

print("Patch embedding:")
print(swin.patch_embed)

print("\nTransformer layers:")
for name in ["layers1", "layers2", "layers3", "layers4"]:
    layers = getattr(swin, name)
    print(f"\n{name}:")
    print("Number of blocks:", len(layers[0].blocks))

    first_block = layers[0].blocks[0]
    print("Attention heads:", first_block.attn.num_heads)
    print("Window size:", first_block.window_size)

print("\nModel feature size: 24")

In [ ]:
import subprocess
from pathlib import Path

repo = Path("/content/FluenceFormer_reference")

if not repo.exists():
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/UJUNWAMGBOH/FluenceFormer.git",
            str(repo)
        ],
        check=True
    )

print("STEP 108 — ORIGINAL SWINUNETR CODE")
print("-" * 55)

for file in sorted(repo.rglob("*.py")):
    content = file.read_text(errors="replace")
    if "SwinUNETR(" in content:
        print("\nFILE:", file.relative_to(repo))

        lines = content.splitlines()
        for i, line in enumerate(lines):
            if "SwinUNETR(" in line:
                start = max(0, i - 4)
                end = min(len(lines), i + 16)

                for j in range(start, end):
                    print(f"{j + 1}: {lines[j]}")

In [ ]:
from pathlib import Path

repo = Path("/content/FluenceFormer_reference")

print("STEP 109 — SEARCH FOR STAGE 1 IMPLEMENTATION")
print("-" * 60)

keywords = [
    "stage1",
    "stage_1",
    "dose_prediction",
    "dose_predictor",
    "in_channels=2",
    "in_channels = 2",
]

for file in sorted(repo.rglob("*.py")):
    content = file.read_text(errors="replace")
    lines = content.splitlines()

    matches = [
        (i + 1, line.strip())
        for i, line in enumerate(lines)
        if any(word in line.lower() for word in keywords)
    ]

    if matches:
        print("\nFILE:", file.relative_to(repo))

        for line_number, line in matches[:20]:
            print(f"Line {line_number}: {line}")

In [ ]:
from pathlib import Path

repo = Path("/content/FluenceFormer_reference")

print("STEP 110 — REFERENCE TRAINING SCRIPTS")
print("-" * 55)

for file in sorted(repo.rglob("*.py")):
    name = file.name.lower()

    if any(word in name for word in [
        "train", "run", "main", "dataset", "config"
    ]):
        print(file.relative_to(repo))

In [ ]:
from pathlib import Path

file = Path(
    "/content/FluenceFormer_reference/"
    "src/prostate_fluence/run_train_and_infer.py"
)

print("STEP 111 — ORIGINAL TRAINING ENTRY POINT")
print("-" * 60)

lines = file.read_text(errors="replace").splitlines()

print("Total lines:", len(lines))
print()

for i, line in enumerate(lines[:180], start=1):
    print(f"{i:03d}: {line}")

In [ ]:
from pathlib import Path

file = Path(
    "/content/FluenceFormer_reference/"
    "src/prostate_fluence/config.py"
)

print("STEP 112 — ORIGINAL EXPERIMENT CONFIGURATION")
print("-" * 60)

for i, line in enumerate(
    file.read_text(errors="replace").splitlines(),
    start=1
):
    print(f"{i:03d}: {line}")

In [ ]:
from pathlib import Path

file = Path(
    "/content/FluenceFormer_reference/"
    "src/prostate_fluence/dataset.py"
)

print("STEP 113 — ORIGINAL DATASET IMPLEMENTATION")
print("-" * 60)

lines = file.read_text(errors="replace").splitlines()

print("Total lines:", len(lines))
print()

for i, line in enumerate(lines[:180], start=1):
    print(f"{i:03d}: {line}")

In [ ]:
from pathlib import Path

file = Path(
    "/content/FluenceFormer_reference/"
    "src/prostate_fluence/utils.py"
)

print("STEP 114 — REFERENCE PREPROCESSING FUNCTIONS")
print("-" * 60)

lines = file.read_text(errors="replace").splitlines()

keywords = [
    "def normalize_global",
    "def resize_D1HW_numpy",
    "def ensure_D1HW",
]

for i, line in enumerate(lines):
    if any(keyword in line for keyword in keywords):
        print(f"\nFUNCTION AT LINE {i + 1}:")
        for j in range(i, min(i + 20, len(lines))):
            print(f"{j + 1:03d}: {lines[j]}")

In [ ]:
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("STEP 115 — REPRODUCIBILITY SETUP")
print("-" * 45)
print("Random seed:", SEED)
print("PyTorch version:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0)
      if torch.cuda.is_available() else "CPU")
print("Deterministic cuDNN:", torch.backends.cudnn.deterministic)
print("Existing Stage 1 checkpoint: unchanged")

In [ ]:
import torch
from monai.networks.nets import SwinUNETR

print("STEP 116 — NEW STAGE 1 MODEL")
print("-" * 50)

stage1_model_v2 = SwinUNETR(
    in_channels=2,
    out_channels=1,
    feature_size=48,
    spatial_dims=2,
    use_checkpoint=True
).to(device)

parameters = sum(
    p.numel()
    for p in stage1_model_v2.parameters()
    if p.requires_grad
)

print("Architecture: SwinUNETR")
print("Input channels: 2")
print("Output channels: 1")
print("Feature size: 48")
print("Trainable parameters:", f"{parameters:,}")
print("Device:", device)

print("\nImportant:")
print("This is a newly initialized model.")
print("No training has started.")
print("The original checkpoint remains unchanged.")

In [ ]:
import torch
import torch.nn.functional as F

print("STEP 117 — NEW MODEL INITIALIZATION CHECK")
print("-" * 55)

stage1_model_v2.train()
stage1_model_v2.zero_grad(set_to_none=True)

sample_x = X[7:8].to(device)
sample_y = Y[7:8].to(device)

raw_output = stage1_model_v2(sample_x)
raw_output.retain_grad()

prediction = F.relu(raw_output)
loss = F.mse_loss(prediction, sample_y)

loss.backward()

ptv = sample_x[0, 1] > 0.5

raw_ptv = raw_output[0, 0][ptv]
ptv_grad = raw_output.grad[0, 0][ptv]

print("Raw PTV prediction minimum:",
      round(float(raw_ptv.min().item()), 6))

print("Raw PTV prediction maximum:",
      round(float(raw_ptv.max().item()), 6))

print("Raw PTV prediction mean:",
      round(float(raw_ptv.mean().item()), 6))

print("\nPTV pixels:", int(ptv.sum().item()))

print("PTV pixels with positive raw predictions:",
      int((raw_ptv > 0).sum().item()))

print("PTV pixels receiving nonzero gradients:",
      int((ptv_grad != 0).sum().item()))

print("\nInitial MSE loss:",
      round(float(loss.item()), 6))

stage1_model_v2.zero_grad(set_to_none=True)

In [ ]:
import torch

print("STEP 118 — OUTPUT DISTRIBUTION CHECK")
print("-" * 50)

stage1_model_v2.eval()

with torch.no_grad():
    raw = stage1_model_v2(X[7:8].to(device))[0, 0]

    ptv = X[7, 1].to(device) > 0.5

    print("Entire image:")
    print("Mean raw output:", round(float(raw.mean()), 4))
    print("Positive pixels:", int((raw > 0).sum()))
    print("Total pixels:", raw.numel())

    print("\nInside PTV:")
    print("Mean raw output:", round(float(raw[ptv].mean()), 4))
    print("Positive pixels:", int((raw[ptv] > 0).sum()))
    print("Total pixels:", int(ptv.sum()))

    print("\nOutside PTV:")
    print("Mean raw output:", round(float(raw[~ptv].mean()), 4))
    print("Positive pixels:", int((raw[~ptv] > 0).sum()))
    print("Total pixels:", int((~ptv).sum()))

In [ ]:
import torch

print("STEP 119 — NEW MODEL OUTPUT HEAD")
print("-" * 50)

output_layer = stage1_model_v2.out.conv.conv

with torch.no_grad():
    weights = output_layer.weight
    bias = output_layer.bias

    print("Weight minimum:", round(float(weights.min()), 6))
    print("Weight maximum:", round(float(weights.max()), 6))
    print("Weight mean:", round(float(weights.mean()), 6))
    print("Weight standard deviation:",
          round(float(weights.std()), 6))

    print("\nBias:", round(float(bias.item()), 6))

    print("\nWeights finite:",
          bool(torch.isfinite(weights).all()))

    print("Bias finite:",
          bool(torch.isfinite(bias).all()))

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

print("STEP 120 — STAGE 1 CONTROLLED TRAINING")
print("-" * 55)

train_loader_v2 = DataLoader(
    TensorDataset(X, Y),
    batch_size=2,
    shuffle=True
)

optimizer_v2 = torch.optim.Adam(
    stage1_model_v2.parameters(),
    lr=1e-4
)

stage1_losses_v2 = []

for epoch in range(2):
    stage1_model_v2.train()
    total_loss = 0.0

    for batch_x, batch_y in train_loader_v2:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer_v2.zero_grad(set_to_none=True)

        prediction = F.relu(stage1_model_v2(batch_x))
        loss = F.mse_loss(prediction, batch_y)

        loss.backward()
        optimizer_v2.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader_v2)
    stage1_losses_v2.append(average_loss)

    print(
        f"Epoch {epoch + 1}/2 | "
        f"Training MSE: {average_loss:.6f}"
    )

print("\nTwo-epoch diagnostic training completed.")
print("Previous checkpoint remains unchanged.")

In [ ]:
import torch
import torch.nn.functional as F

print("STEP 121 — PTV DOSE AFTER TWO EPOCHS")
print("-" * 55)

stage1_model_v2.eval()

with torch.no_grad():
    raw = stage1_model_v2(
        X[7:8].to(device)
    )

    predicted = F.relu(raw)

ptv = X[7, 1].to(device) > 0.5

actual_ptv = Y[7, 0].to(device)[ptv] * 3.0
predicted_ptv = predicted[0, 0][ptv] * 3.0

print("Actual mean PTV dose (Gy):",
      round(float(actual_ptv.mean()), 4))

print("Predicted mean PTV dose (Gy):",
      round(float(predicted_ptv.mean()), 4))

print("Predicted minimum PTV dose (Gy):",
      round(float(predicted_ptv.min()), 4))

print("Predicted maximum PTV dose (Gy):",
      round(float(predicted_ptv.max()), 4))

print("PTV mean absolute error (Gy):",
      round(float(
          (predicted_ptv - actual_ptv).abs().mean()
      ), 4))

print("PTV pixels with positive predictions:",
      int((predicted_ptv > 0).sum().item()))

print("Total PTV pixels:", int(ptv.sum().item()))

In [ ]:
import torch
import torch.nn.functional as F

print("STEP 122 — PTV VS BACKGROUND ERROR")
print("-" * 55)

stage1_model_v2.eval()

with torch.no_grad():
    sample_x = X[7:8].to(device)
    sample_y = Y[7:8].to(device)

    prediction = F.relu(stage1_model_v2(sample_x))

    squared_error = (prediction - sample_y) ** 2

    ptv_mask = sample_x[:, 1:2] > 0.5
    outside_mask = ~ptv_mask

    total_mse = squared_error.mean()
    ptv_mse = squared_error[ptv_mask].mean()
    outside_mse = squared_error[outside_mask].mean()

    ptv_fraction = ptv_mask.float().mean()

    ptv_contribution = (
        squared_error[ptv_mask].sum()
        / squared_error.numel()
    )

    outside_contribution = (
        squared_error[outside_mask].sum()
        / squared_error.numel()
    )

print("PTV fraction of image:",
      f"{ptv_fraction.item() * 100:.2f}%")

print("\nTotal image MSE:",
      round(total_mse.item(), 6))

print("PTV-only MSE:",
      round(ptv_mse.item(), 6))

print("Outside-PTV MSE:",
      round(outside_mse.item(), 6))

print("\nContribution to total MSE:")

print("PTV:",
      round(ptv_contribution.item(), 6))

print("Outside PTV:",
      round(outside_contribution.item(), 6))

print("\nCheck — contributions sum:",
      round(
          (ptv_contribution + outside_contribution).item(),
          6
      ))

In [ ]:
import torch
import torch.nn.functional as F

print("STEP 123 — PTV GRADIENT CHECK AFTER TRAINING")
print("-" * 55)

stage1_model_v2.train()
stage1_model_v2.zero_grad(set_to_none=True)

sample_x = X[7:8].to(device)
sample_y = Y[7:8].to(device)

raw = stage1_model_v2(sample_x)
raw.retain_grad()

prediction = F.relu(raw)
loss = F.mse_loss(prediction, sample_y)

loss.backward()

ptv = sample_x[:, 1:2] > 0.5

ptv_raw = raw[ptv]
ptv_grad = raw.grad[ptv]

print("PTV pixels:", ptv_raw.numel())

print("Positive raw PTV predictions:",
      int((ptv_raw > 0).sum().item()))

print("PTV pixels with nonzero output gradients:",
      int((ptv_grad != 0).sum().item()))

print("PTV pixels with zero output gradients:",
      int((ptv_grad == 0).sum().item()))

print("\nMean raw PTV output:",
      round(float(ptv_raw.mean().item()), 6))

print("PTV output gradient absolute sum:",
      round(float(ptv_grad.abs().sum().item()), 8))

print("MSE loss:",
      round(float(loss.item()), 6))

stage1_model_v2.zero_grad(set_to_none=True)
stage1_model_v2.eval()

print("\nNo optimizer step performed.")
print("Model weights unchanged.")

In [ ]:
import os
import torch

print("STEP 124 — SAVE TWO-EPOCH DIAGNOSTIC CHECKPOINT")
print("-" * 55)

checkpoint_dir = (
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction/"
    "Stage1_Checkpoints"
)

os.makedirs(checkpoint_dir, exist_ok=True)

checkpoint_path = os.path.join(
    checkpoint_dir,
    "swinunetr_stage1_patient1_feature48_epoch2_diagnostic.pth"
)

assert len(stage1_losses_v2) == 2
assert not os.path.exists(checkpoint_path), (
    "Checkpoint already exists. Refusing to overwrite it."
)

torch.save({
    "epoch": 2,
    "model_state_dict": stage1_model_v2.state_dict(),
    "optimizer_state_dict": optimizer_v2.state_dict(),
    "training_losses": stage1_losses_v2,
    "architecture": "SwinUNETR",
    "feature_size": 48,
    "in_channels": 2,
    "out_channels": 1,
    "learning_rate": 1e-4,
    "patient": "Patient_1",
    "experiment_type": "preliminary_diagnostic",
    "notes": (
        "Two-epoch Stage 1 diagnostic. "
        "PTV output gradients blocked at 198/199 pixels "
        "on evaluated slice. Not a validated reproduction."
    )
}, checkpoint_path)

print("Checkpoint saved:", checkpoint_path)
print("File size (MB):",
      round(os.path.getsize(checkpoint_path) / 1e6, 2))

print("\nTraining losses:", stage1_losses_v2)
print("Existing checkpoints were not overwritten.")

In [ ]:
import numpy as np
import torch

print("STEP 125 — STAGE 1 DOSE NORMALIZATION AUDIT")
print("-" * 55)

dose_path = (
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction/"
    "Stage1_Verified_Preprocessing/"
    "dose_stage1.npy"
)

raw_dose = np.load(dose_path)

print("Processed PortPy dose array:")
print("Shape:", raw_dose.shape)
print("Minimum (Gy):", float(raw_dose.min()))
print("Maximum (Gy):", float(raw_dose.max()))
print("Mean (Gy):", float(raw_dose.mean()))

print("\nTraining target Y:")
print("Shape:", tuple(Y.shape))
print("Minimum:", float(Y.min()))
print("Maximum:", float(Y.max()))
print("Mean:", float(Y.mean()))

expected_Y = torch.tensor(
    raw_dose / 3.0,
    dtype=Y.dtype
)

print("\nNormalization verification:")
print("Y equals dose / 3:",
      bool(torch.allclose(Y.cpu(), expected_Y)))

print("Maximum absolute difference:",
      float((Y.cpu() - expected_Y).abs().max()))

ptv_mask = X[:, 1:2] > 0.5

print("\nInside PTV:")
print("Mean dose in Gy:",
      float(torch.tensor(raw_dose)[ptv_mask].mean()))

print("Mean normalized target:",
      float(Y[ptv_mask].mean()))

print("\nNo model weights or files modified.")

In [ ]:
from pathlib import Path

print("STEP 126 — REFERENCE DOSE PREPROCESSING")
print("-" * 60)

repo = Path("/content/FluenceFormer_reference")

files_to_check = [
    repo / "src/prostate_fluence/dataset.py",
    repo / "src/prostate_fluence/config.py",
    repo / "src/prostate_fluence/run_train_and_infer.py"
]

keywords = [
    "dose",
    "normaliz",
    "scal",
    "relu",
    "mse",
    "loss"
]

for path in files_to_check:
    print(f"\nFILE: {path.relative_to(repo)}")
    print("-" * 60)

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    lines = path.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()

    matches = [
        i for i, line in enumerate(lines)
        if any(
            word in line.lower()
            for word in keywords
        )
    ]

    shown = set()

    for i in matches:
        for j in range(
            max(0, i - 2),
            min(len(lines), i + 3)
        ):
            if j not in shown:
                print(f"{j + 1:4}: {lines[j]}")
                shown.add(j)

print("\nReference inspection completed.")
print("No model weights or data modified.")

In [ ]:
from pathlib import Path

print("STEP 127 — REPOSITORY-WIDE STAGE 1 SEARCH")
print("-" * 60)

repo = Path("/content/FluenceFormer_reference")

terms = [
    "stage1",
    "stage_1",
    "stage 1",
    "dose prediction",
    "dose_prediction",
    "epochs_s1"
]

matches = []

for path in repo.rglob("*"):
    if not path.is_file():
        continue

    if ".git" in path.parts:
        continue

    if path.suffix.lower() not in {
        ".py", ".md", ".yaml", ".yml", ".json", ".txt"
    }:
        continue

    try:
        lines = path.read_text(
            encoding="utf-8",
            errors="replace"
        ).splitlines()
    except OSError:
        continue

    for number, line in enumerate(lines, start=1):
        if any(term in line.lower() for term in terms):
            matches.append((
                str(path.relative_to(repo)),
                number,
                line.strip()
            ))

print("Total matches:", len(matches))

for filename, number, line in matches[:100]:
    print(f"{filename}:{number}: {line[:180]}")

if len(matches) > 100:
    print(f"\nAdditional matches omitted: {len(matches)-100}")

if not matches:
    print("No Stage 1 references found in searched text files.")

print("\nNo model weights or data modified.")

In [ ]:
from pathlib import Path

print("STEP 128 — REFERENCE README METHODOLOGY")
print("-" * 60)

readme_path = Path(
    "/content/FluenceFormer_reference/README.md"
)

lines = readme_path.read_text(
    encoding="utf-8",
    errors="replace"
).splitlines()

for i, line in enumerate(lines[:110], start=1):
    print(f"{i:3}: {line}")

print("\nREADME inspection completed.")
print("No model weights or data modified.")

In [ ]:
from pathlib import Path

print("STEP 129 — ORIGINAL TRAINING INSTRUCTIONS")
print("-" * 60)

repo = Path("/content/FluenceFormer_reference")

instruction_files = [
    p for p in repo.rglob("*")
    if p.is_file()
    and "instruction" in p.name.lower()
    and ".git" not in p.parts
]

print("Instruction files found:", len(instruction_files))

for path in instruction_files:
    print("\nFILE:", path.relative_to(repo))
    print("-" * 60)

    lines = path.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()

    for number, line in enumerate(lines, start=1):
        print(f"{number:4}: {line}")

if not instruction_files:
    print(
        "\nNo instruction file found in the cloned repository."
    )

print("\nInspection completed.")
print("No model weights or data modified.")

In [ ]:
import requests
import fitz

print("STEP 130 — ORIGINAL PAPER STAGE 1 SEARCH")
print("-" * 60)

paper_url = "https://arxiv.org/pdf/2512.22425"

response = requests.get(paper_url, timeout=60)
response.raise_for_status()

pdf = fitz.open(stream=response.content, filetype="pdf")

keywords = [
    "stage 1",
    "stage-1",
    "dose prediction",
    "dose prior",
    "mean squared error",
    "relu",
    "normalization",
    "preprocessing"
]

for page_number, page in enumerate(pdf, start=1):
    text = page.get_text()

    lines = text.splitlines()

    matches = [
        i for i, line in enumerate(lines)
        if any(
            keyword in line.lower()
            for keyword in keywords
        )
    ]

    if not matches:
        continue

    print(f"\nPAGE {page_number}")
    print("-" * 60)

    shown = set()

    for i in matches:
        for j in range(
            max(0, i - 3),
            min(len(lines), i + 5)
        ):
            if j not in shown:
                print(lines[j])
                shown.add(j)

pdf.close()

print("\nPaper search completed.")
print("No model weights or data modified.")

In [ ]:
%pip install -q PyMuPDF

In [ ]:
import requests
import fitz

print("STEP 130 — ORIGINAL PAPER STAGE 1 SEARCH")
print("-" * 60)

paper_url = "https://arxiv.org/pdf/2512.22425"

response = requests.get(paper_url, timeout=60)
response.raise_for_status()

pdf = fitz.open(stream=response.content, filetype="pdf")

keywords = [
    "stage 1",
    "stage-1",
    "dose prediction",
    "dose prior",
    "mean squared error",
    "relu",
    "normalization",
    "preprocessing"
]

for page_number, page in enumerate(pdf, start=1):
    text = page.get_text()

    lines = text.splitlines()

    matches = [
        i for i, line in enumerate(lines)
        if any(
            keyword in line.lower()
            for keyword in keywords
        )
    ]

    if not matches:
        continue

    print(f"\nPAGE {page_number}")
    print("-" * 60)

    shown = set()

    for i in matches:
        for j in range(
            max(0, i - 3),
            min(len(lines), i + 5)
        ):
            if j not in shown:
                print(lines[j])
                shown.add(j)

pdf.close()

print("\nPaper search completed.")
print("No model weights or data modified.")

In [ ]:
import h5py
from pathlib import Path

print("STEP 131 — PORTPY ANATOMICAL STRUCTURE AUDIT")
print("-" * 60)

structure_path = Path(
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction/"
    "Patient_1/raw_portpy/StructureSet_Data.h5"
)

print("File exists:", structure_path.exists())

if not structure_path.exists():
    raise FileNotFoundError(structure_path)

with h5py.File(structure_path, "r") as f:

    def show_structure(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(
                f"{name} | "
                f"shape={obj.shape} | "
                f"dtype={obj.dtype}"
            )

    f.visititems(show_structure)

print("\nAudit completed.")
print("No model weights or data modified.")

In [ ]:
from pathlib import Path

print("STEP 131A — LOCATE PORTPY STRUCTURE FILE")
print("-" * 60)

drive_root = Path("/content/drive/MyDrive")

print("Google Drive accessible:", drive_root.exists())

if not drive_root.exists():
    from google.colab import drive
    drive.mount("/content/drive")

project = drive_root / "FluenceFormer_PortPy_Reproduction"

print("Project folder exists:", project.exists())

if project.exists():
    matches = list(
        project.rglob("StructureSet_Data.h5")
    )

    print("\nMatching structure files:", len(matches))

    for path in matches[:20]:
        print(path)

    if not matches:
        print("\nNo matching file found in project folder.")

print("\nNo files or model weights modified.")

In [ ]:
import h5py
from pathlib import Path

print("STEP 131B — PORTPY ANATOMICAL STRUCTURES")
print("-" * 60)

structure_path = Path(
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction/"
    "Patient_1/raw_portpy/StructureSet_Data.h5"
)

with h5py.File(structure_path, "r") as f:

    print("Available anatomical datasets:\n")

    def show_structure(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(
                f"{name} | "
                f"shape={obj.shape} | "
                f"dtype={obj.dtype}"
            )

    f.visititems(show_structure)

print("\nStructure audit completed.")
print("No files or model weights modified.")

In [ ]:
import h5py
import numpy as np

print("STEP 132 — ANATOMICAL STRUCTURES IN SELECTED SLICES")
print("-" * 65)

structure_path = (
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction/"
    "Patient_1/raw_portpy/StructureSet_Data.h5"
)

selected_slices = range(30, 45)

with h5py.File(structure_path, "r") as f:

    for name in sorted(f.keys()):
        dataset = f[name]

        if not isinstance(dataset, h5py.Dataset):
            continue

        masks = dataset[30:45]

        pixels = np.count_nonzero(masks)

        slices_present = np.count_nonzero(
            np.any(masks > 0, axis=(1, 2))
        )

        print(
            f"{name:10} | "
            f"Slices present: {slices_present:2}/15 | "
            f"Nonzero pixels: {pixels:,}"
        )

print("\nNo preprocessing or model weights modified.")

In [ ]:
import requests
import pymupdf

print("STEP 133 — EXACT STAGE 1 INPUT SPECIFICATION")
print("-" * 60)

url = "https://arxiv.org/pdf/2512.22425"

response = requests.get(url, timeout=60)
response.raise_for_status()

pdf = pymupdf.open(
    stream=response.content,
    filetype="pdf"
)

for page_number in [2, 3, 5]:
    print(f"\n{'=' * 25}")
    print(f"PDF PAGE {page_number + 1}")
    print(f"{'=' * 25}")

    page = pdf[page_number]
    text = page.get_text(sort=True)

    print(text[:14000])

pdf.close()

print("\nPaper inspection completed.")
print("No model weights or datasets modified.")

In [ ]:
from pathlib import Path

print("STEP 134 — ORIGINAL CONTOUR REPRESENTATION")
print("-" * 60)

repo = Path("/content/FluenceFormer_reference")

for relative_path in [
    "src/prostate_fluence/dataset.py",
    "src/prostate_fluence/utils.py"
]:
    path = repo / relative_path

    print(f"\nFILE: {relative_path}")
    print("-" * 60)

    if not path.exists():
        print("File not found")
        continue

    lines = path.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()

    for number, line in enumerate(lines, start=1):
        print(f"{number:4}: {line}")

print("\nInspection completed.")
print("No model weights or datasets modified.")

In [ ]:
from pathlib import Path

print("STEP 134A — LOCATE ORIGINAL FLUENCEFORMER REPOSITORY")
print("-" * 65)

locations = [
    Path("/content/FluenceFormer_reference"),
    Path("/content/FluenceFormer"),
    Path("/content/drive/MyDrive/FluenceFormer_reference"),
    Path("/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction")
]

for location in locations:
    print(f"\nLocation: {location}")
    print("Exists:", location.exists())

    if location.exists():
        print("Is directory:", location.is_dir())

        if location.is_dir():
            print("Contents:")
            for item in list(location.iterdir())[:15]:
                print("  ", item.name)

print("\nSearching /content for repository dataset.py files:")

matches = list(Path("/content").glob(
    "*/src/prostate_fluence/dataset.py"
))

for match in matches:
    print("FOUND:", match)

if not matches:
    print("No matching repository dataset.py found.")

print("\nNo files or model weights modified.")

In [ ]:
import subprocess
from pathlib import Path

print("STEP 134B — RESTORE ORIGINAL FLUENCEFORMER REPOSITORY")
print("-" * 65)

repo_url = "https://github.com/UJUNWAMGBOH/FluenceFormer.git"
repo_path = Path("/content/FluenceFormer_reference")

if not repo_path.exists():
    result = subprocess.run(
        ["git", "clone", repo_url, str(repo_path)],
        capture_output=True,
        text=True
    )

    print("Git clone return code:", result.returncode)
    print(result.stdout[-2000:])
    print(result.stderr[-2000:])

    if result.returncode != 0:
        raise RuntimeError("Repository clone failed.")
else:
    print("Repository already exists.")

print("\nRepository exists:", repo_path.exists())

if repo_path.exists():
    commit = subprocess.run(
        ["git", "-C", str(repo_path), "rev-parse", "HEAD"],
        capture_output=True,
        text=True
    )

    print("Repository commit:", commit.stdout.strip())

    dataset_path = (
        repo_path / "src/prostate_fluence/dataset.py"
    )

    print("Dataset implementation exists:", dataset_path.exists())

print("\nNo patient data or model checkpoints modified.")

In [ ]:
from pathlib import Path

print("STEP 135 — ORIGINAL CONTOUR HANDLING")
print("-" * 65)

repo = Path("/content/FluenceFormer_reference")
dataset_file = repo / "src/prostate_fluence/dataset.py"

print("File exists:", dataset_file.exists())

if not dataset_file.exists():
    raise FileNotFoundError(dataset_file)

lines = dataset_file.read_text(
    encoding="utf-8",
    errors="replace"
).splitlines()

keywords = [
    "contour",
    "mask",
    "channel",
    "stack",
    "concatenate",
    "cat(",
    "load(",
    "in_channels"
]

matching_lines = set()

for i, line in enumerate(lines):
    if any(word in line.lower() for word in keywords):
        start = max(0, i - 4)
        end = min(len(lines), i + 6)

        matching_lines.update(range(start, end))

print("\nRelevant code sections:\n")

previous = -2

for i in sorted(matching_lines):
    if i > previous + 1:
        print("\n" + "-" * 45)

    print(f"{i + 1:4}: {lines[i]}")
    previous = i

print("\nTotal lines in dataset.py:", len(lines))
print("No patient data or model weights modified.")

In [ ]:
from pathlib import Path

print("STEP 136 — INSPECT CONTOUR ARRAY HELPERS")
print("-" * 65)

repo = Path("/content/FluenceFormer_reference")

targets = [
    "ensure_D1HW",
    "resize_D1HW_numpy"
]

found = False

for path in sorted((repo / "src").rglob("*.py")):
    lines = path.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()

    for i, line in enumerate(lines):
        if any(
            line.lstrip().startswith(f"def {name}(")
            for name in targets
        ):
            found = True

            print(f"\nFILE: {path.relative_to(repo)}")
            print(f"FUNCTION AT LINE {i + 1}")
            print("-" * 50)

            for j in range(i, min(i + 45, len(lines))):
                if j > i and lines[j].startswith(
                    ("def ", "class ", "@")
                ):
                    break

                print(f"{j + 1:4}: {lines[j]}")

if not found:
    print("Helper definitions not found in src/.")

print("\nNo datasets or model weights modified.")

In [ ]:
from pathlib import Path
import numpy as np

print("STEP 137 — VERIFY SAVED STAGE 1 DATA")
print("-" * 65)

data_dir = Path(
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction/"
    "Stage1_Verified_Preprocessing"
)

files = {
    "CT": "ct_stage1.npy",
    "PTV": "ptv_stage1.npy",
    "Dose": "dose_stage1.npy"
}

arrays = {}

for name, filename in files.items():
    path = data_dir / filename

    print(f"\n{name}")
    print("Exists:", path.exists())

    if not path.exists():
        continue

    arr = np.load(path, mmap_mode="r")
    arrays[name] = arr

    print("Shape:", arr.shape)
    print("Data type:", arr.dtype)
    print("Minimum:", float(np.min(arr)))
    print("Maximum:", float(np.max(arr)))
    print("Mean:", float(np.mean(arr)))

print("\nAll three arrays available:", len(arrays) == 3)

if len(arrays) == 3:
    print(
        "Shapes match:",
        len(set(a.shape for a in arrays.values())) == 1
    )

print("\nNo files or model weights modified.")

In [ ]:
import numpy as np
import torch
from pathlib import Path

print("STEP 138 — RECONSTRUCT STAGE 1 INPUTS")
print("-" * 65)

data_dir = Path(
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction/"
    "Stage1_Verified_Preprocessing"
)

ct = np.load(data_dir / "ct_stage1.npy")
ptv = np.load(data_dir / "ptv_stage1.npy")
dose = np.load(data_dir / "dose_stage1.npy")

# Fixed preprocessing used in our earlier experiment
ct_normalized = np.clip(
    (ct + 1000.0) / 4000.0,
    0.0,
    1.0
).astype(np.float32)

ptv_normalized = ptv.astype(np.float32)

dose_normalized = (
    dose / 3.0
).astype(np.float32)

# Stage 1: CT + contour -> dose
X = torch.from_numpy(
    np.concatenate(
        [ct_normalized, ptv_normalized],
        axis=1
    )
)

Y = torch.from_numpy(dose_normalized)

print("Input X shape:", tuple(X.shape))
print("Target Y shape:", tuple(Y.shape))

print("\nInput channel 1 (CT):")
print("  Min:", float(X[:, 0].min()))
print("  Max:", float(X[:, 0].max()))

print("\nInput channel 2 (PTV):")
print("  Min:", float(X[:, 1].min()))
print("  Max:", float(X[:, 1].max()))

print("\nTarget dose:")
print("  Min:", float(Y.min()))
print("  Max:", float(Y.max()))

print("\nFinite values:", bool(
    torch.isfinite(X).all()
    and torch.isfinite(Y).all()
))

print("Input channels:", X.shape[1])
print("Samples:", X.shape[0])

print("\nNo files or model weights modified.")

In [ ]:
import torch
from pathlib import Path

print("STEP 139 — VERIFY SAVED STAGE 1 CHECKPOINT")
print("-" * 65)

checkpoint_path = Path(
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction/"
    "Stage1_Checkpoints/"
    "swinunetr_stage1_patient1_feature48_epoch2_diagnostic.pth"
)

print("Checkpoint exists:", checkpoint_path.exists())

if not checkpoint_path.exists():
    raise FileNotFoundError(checkpoint_path)

print(
    "Checkpoint size (MB):",
    round(checkpoint_path.stat().st_size / (1024**2), 2)
)

# Inspect checkpoint on CPU without loading it into the GPU.
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

print("\nCheckpoint contents:")

if isinstance(checkpoint, dict):
    for key, value in checkpoint.items():
        if isinstance(value, dict):
            print(f"  {key}: dictionary ({len(value)} entries)")
        elif isinstance(value, (int, float, str, bool)):
            print(f"  {key}: {value}")
        else:
            print(f"  {key}: {type(value).__name__}")
else:
    print("Checkpoint type:", type(checkpoint).__name__)

print("\nNo model training performed.")
print("No files or model weights modified.")

In [ ]:
import torch
from monai.networks.nets import SwinUNETR
from pathlib import Path

print("STEP 140 — RESTORE SAVED STAGE 1 MODEL")
print("-" * 65)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

checkpoint_path = Path(
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction/"
    "Stage1_Checkpoints/"
    "swinunetr_stage1_patient1_feature48_epoch2_diagnostic.pth"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

# Reconstruct the architecture used in the diagnostic
stage1_model = SwinUNETR(
    in_channels=2,
    out_channels=1,
    feature_size=48,
    spatial_dims=2,
    use_checkpoint=True
).to(device)

stage1_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

stage1_model.eval()

parameter_count = sum(
    p.numel() for p in stage1_model.parameters()
)

print("Model parameters:", f"{parameter_count:,}")
print("Checkpoint epoch:", checkpoint["epoch"])

# Test one of the previously prepared slices
with torch.no_grad():
    test_input = X[7:8].to(device)

    raw_output = stage1_model(test_input)

    # Paper-specified nonnegative dose output
    prediction = torch.relu(raw_output)

print("\nInput shape:", tuple(test_input.shape))
print("Output shape:", tuple(prediction.shape))

print(
    "Prediction minimum:",
    float(prediction.min().item())
)

print(
    "Prediction maximum:",
    float(prediction.max().item())
)

print(
    "Prediction mean:",
    float(prediction.mean().item())
)

print(
    "Finite predictions:",
    bool(torch.isfinite(prediction).all())
)

print("\nModel restored successfully.")
print("No training performed.")
print("No checkpoint files modified.")

In [ ]:
%pip install -q monai==1.5.1

import torch
import monai
from monai.networks.nets import SwinUNETR

print("STEP 140A — VERIFY MONAI INSTALLATION")
print("-" * 60)

print("MONAI version:", monai.__version__)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nSwinUNETR import successful.")
print("No patient data or checkpoints modified.")

In [ ]:
import torch
from pathlib import Path

print("STEP 140C — VERIFY CPU ENVIRONMENT")
print("-" * 60)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

project = Path(
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction"
)

print("Google Drive project accessible:", project.exists())

checkpoint = (
    project / "Stage1_Checkpoints" /
    "swinunetr_stage1_patient1_feature48_epoch2_diagnostic.pth"
)

print("Saved checkpoint accessible:", checkpoint.exists())

print("\nNo training performed.")
print("No files modified.")

In [ ]:
from pathlib import Path

print("STEP 141 — PORTPY PATIENT INVENTORY")
print("-" * 65)

project = Path(
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction"
)

patient_folders = sorted(
    [
        p for p in project.iterdir()
        if p.is_dir() and p.name.startswith("Patient_")
    ],
    key=lambda p: (
        int(p.name.split("_")[-1])
        if p.name.split("_")[-1].isdigit()
        else float("inf")
    )
)

print("Patient folders found:", len(patient_folders))

for patient in patient_folders:
    print(f"\n{patient.name}")

    raw = patient / "raw_portpy"
    optimized = patient / "optimized_portpy"

    print("  Raw folder exists:", raw.exists())
    print("  Optimized folder exists:", optimized.exists())

    print(
        "  CT available:",
        (raw / "CT_Data.h5").exists()
    )

    print(
        "  Structures available:",
        (raw / "StructureSet_Data.h5").exists()
    )

    print(
        "  Optimized dose available:",
        (optimized / "optimized_dose_3d.npy").exists()
    )

print("\nNo files modified.")
print("No model training performed.")

In [ ]:
from google.colab import drive
from pathlib import Path

print("STEP 142 — MOUNT GOOGLE DRIVE")
print("-" * 60)

drive.mount("/content/drive")

project = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction"
)

print("\nGoogle Drive mounted successfully.")
print("Project folder exists:", project.exists())

if project.exists():
    print("\nProject contents:")
    for item in sorted(project.iterdir()):
        print(" -", item.name)

print("\nNo files modified.")

In [ ]:
from pathlib import Path

print("STEP 141 — PORTPY PATIENT INVENTORY")
print("-" * 65)

project = Path(
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction"
)

patient_folders = sorted(
    [
        p for p in project.iterdir()
        if p.is_dir() and p.name.startswith("Patient_")
    ],
    key=lambda p: (
        int(p.name.split("_")[-1])
        if p.name.split("_")[-1].isdigit()
        else float("inf")
    )
)

print("Patient folders found:", len(patient_folders))

for patient in patient_folders:
    print(f"\n{patient.name}")

    raw = patient / "raw_portpy"
    optimized = patient / "optimized_portpy"

    print("  Raw folder exists:", raw.exists())
    print("  Optimized folder exists:", optimized.exists())

    print(
        "  CT available:",
        (raw / "CT_Data.h5").exists()
    )

    print(
        "  Structures available:",
        (raw / "StructureSet_Data.h5").exists()
    )

    print(
        "  Optimized dose available:",
        (optimized / "optimized_dose_3d.npy").exists()
    )

print("\nNo files modified.")
print("No model training performed.")

In [ ]:
import shutil
from pathlib import Path

print("STEP 143 — STORAGE CHECK")
print("-" * 60)

for name, location in [
    ("Colab temporary storage", "/content"),
    ("Mounted Google Drive", "/content/drive/MyDrive")
]:
    path = Path(location)

    if path.exists():
        total, used, free = shutil.disk_usage(path)

        print(f"\n{name}")
        print(f"Total: {total / 1024**3:.2f} GB")
        print(f"Used:  {used / 1024**3:.2f} GB")
        print(f"Free:  {free / 1024**3:.2f} GB")
    else:
        print(f"\n{name}: Not accessible")

print("\nIMPORTANT:")
print("Mounted Drive storage figures may not reflect")
print("your actual Google Drive cloud quota.")

print("\nNo files modified.")

In [ ]:
%pip install -q huggingface_hub

from huggingface_hub import HfApi

print("STEP 144 — INSPECT PORTPY DATASET")
print("-" * 65)

api = HfApi()
repo_id = "PortPy-Project/PortPy_Dataset"

try:
    info = api.repo_info(
        repo_id=repo_id,
        repo_type="dataset",
        files_metadata=True
    )

    files = info.siblings or []

    print("Repository:", repo_id)
    print("Total files:", len(files))

    print("\nFirst 30 files:")
    for file in files[:30]:
        size_mb = (
            file.size / (1024**2)
            if file.size is not None else None
        )

        size_text = (
            f"{size_mb:.2f} MB"
            if size_mb is not None else "Unknown size"
        )

        print(f"{file.rfilename} | {size_text}")

    known_bytes = sum(
        file.size for file in files
        if file.size is not None
    )

    print(
        "\nTotal reported file size:",
        round(known_bytes / (1024**3), 2),
        "GB"
    )

except Exception as e:
    print("Dataset inspection failed:")
    print(type(e).__name__, str(e)[:500])

print("\nNo patient files downloaded.")
print("No existing files modified.")

In [ ]:
from collections import defaultdict
import re

print("STEP 145 — PROSTATE DATASET FILE INVENTORY")
print("-" * 65)

# Reuse the repository metadata from Step 144.
# If the runtime was reset, rerun Step 144 first.
if "info" not in globals():
    raise RuntimeError("Repository metadata missing. Rerun Step 144.")

files = info.siblings or []

patients = defaultdict(list)

for file in files:
    path = file.rfilename

    match = re.match(
        r"^data/(Prostate_Patient_\d+)/",
        path
    )

    if match:
        patients[match.group(1)].append(file)

def patient_number(name):
    return int(name.split("_")[-1])

print("Prostate patients visible:", len(patients))

print("\nFirst 10 patients:")
for name in sorted(patients, key=patient_number)[:10]:
    entries = patients[name]

    total_bytes = sum(
        f.size or 0 for f in entries
    )

    print(
        f"{name}: "
        f"{len(entries)} files, "
        f"{total_bytes / 1024**3:.2f} GB"
    )

print("\nPatient 1 file examples:")
for file in patients.get("Prostate_Patient_1", [])[:30]:
    print(
        file.rfilename,
        "|",
        round((file.size or 0) / 1024**2, 2),
        "MB"
    )

print("\nPotentially useful non-beam files:")
for file in patients.get("Prostate_Patient_1", []):
    if "/Beams/" not in file.rfilename:
        print(
            file.rfilename,
            "|",
            round((file.size or 0) / 1024**2, 2),
            "MB"
        )

print("\nNo patient files downloaded.")
print("No existing files modified.")

In [ ]:
from collections import defaultdict
import re

print("STEP 146 — MULTI-PATIENT DATA AVAILABILITY")
print("-" * 65)

if "info" not in globals():
    raise RuntimeError("Rerun Step 144 to restore repository metadata.")

patients = defaultdict(set)

for file in info.siblings or []:
    match = re.match(
        r"^data/(Prostate_Patient_\d+)/(.+)$",
        file.rfilename
    )

    if match:
        patient, relative_path = match.groups()
        patients[patient].add(relative_path)

counts = {
    "CT": 0,
    "Structures": 0,
    "IMRT dose DICOM": 0,
    "IMRT plan DICOM": 0,
    "CT + Structures + IMRT dose": 0
}

eligible = []
missing = []

for patient, paths in patients.items():
    has_ct = "CT_Data.h5" in paths
    has_structures = "StructureSet_Data.h5" in paths
    has_dose = "DicomFiles/rt_dose_echo_imrt.dcm" in paths
    has_plan = "DicomFiles/rt_plan_echo_imrt.dcm" in paths

    counts["CT"] += has_ct
    counts["Structures"] += has_structures
    counts["IMRT dose DICOM"] += has_dose
    counts["IMRT plan DICOM"] += has_plan

    if has_ct and has_structures and has_dose:
        eligible.append(patient)
        counts["CT + Structures + IMRT dose"] += 1
    else:
        missing.append(patient)

print("Patients visible:", len(patients))

for name, count in counts.items():
    print(f"{name}: {count}")

eligible.sort(key=lambda x: int(x.split("_")[-1]))
missing.sort(key=lambda x: int(x.split("_")[-1]))

print("\nFirst 20 potentially eligible patients:")
print(eligible[:20])

print("\nPatients missing at least one required file:")
print(missing[:30])

print("\nNOTE: This checks filenames only.")
print("Dose validity and alignment are not yet verified.")

print("\nNo patient files downloaded.")
print("No existing files modified.")

In [ ]:
from collections import defaultdict
import re

print("STEP 147 — ESTIMATE MULTI-PATIENT DOWNLOAD")
print("-" * 65)

if "info" not in globals():
    raise RuntimeError("Rerun Step 144 first.")

required_files = [
    "CT_Data.h5",
    "CT_MetaData.json",
    "StructureSet_Data.h5",
    "StructureSet_MetaData.json",
    "OptimizationVoxels_Data.h5",
    "OptimizationVoxels_MetaData.json",
    "DicomFiles/rt_dose_echo_imrt.dcm",
    "DicomFiles/rt_plan_echo_imrt.dcm",
]

patients = defaultdict(dict)

for file in info.siblings or []:
    match = re.match(
        r"^data/(Prostate_Patient_\d+)/(.+)$",
        file.rfilename
    )

    if match:
        patient, relative_path = match.groups()

        if relative_path in required_files:
            patients[patient][relative_path] = file.size

eligible = {
    patient: entries
    for patient, entries in patients.items()
    if all(
        name in entries
        for name in [
            "CT_Data.h5",
            "StructureSet_Data.h5",
            "DicomFiles/rt_dose_echo_imrt.dcm"
        ]
    )
}

total_bytes = sum(
    size
    for entries in eligible.values()
    for size in entries.values()
    if size is not None
)

print("Potentially eligible patients:", len(eligible))

print("\nEstimated total download:")
print(f"{total_bytes / 1024**3:.2f} GB")

print("\nAverage per patient:")
if eligible:
    print(
        f"{total_bytes / len(eligible) / 1024**2:.2f} MB"
    )

print("\nSelected file categories:")
for filename in required_files:
    sizes = [
        entries[filename]
        for entries in eligible.values()
        if filename in entries
        and entries[filename] is not None
    ]

    print(
        f"{filename}: "
        f"{len(sizes)} patients, "
        f"{sum(sizes) / 1024**3:.2f} GB"
    )

print("\nIMPORTANT:")
print("These are estimates from the visible repository listing.")
print("No files have been downloaded.")
print("No existing files have been modified.")

In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download
import h5py

print("STEP 148 — DOWNLOAD PATIENT 2 ESSENTIAL FILES")
print("-" * 65)

repo_id = "PortPy-Project/PortPy_Dataset"

project = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction"
)

destination = project / "Patient_2" / "raw_portpy"
destination.mkdir(parents=True, exist_ok=True)

required_files = [
    "CT_Data.h5",
    "CT_MetaData.json",
    "StructureSet_Data.h5",
    "StructureSet_MetaData.json",
    "OptimizationVoxels_Data.h5",
    "OptimizationVoxels_MetaData.json",
    "DicomFiles/rt_dose_echo_imrt.dcm",
    "DicomFiles/rt_plan_echo_imrt.dcm",
]

success = []
failed = []

for filename in required_files:
    remote_path = f"data/Prostate_Patient_2/{filename}"
    local_path = destination / filename

    local_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    if local_path.exists() and local_path.stat().st_size > 0:
        print("Already exists:", filename)
        success.append(filename)
        continue

    try:
        print("Downloading:", filename)

        downloaded = hf_hub_download(
            repo_id=repo_id,
            repo_type="dataset",
            filename=remote_path,
            local_dir=str(project / "_portpy_download_cache")
        )

        import shutil
        shutil.copy2(downloaded, local_path)

        print(
            "  Saved:",
            round(local_path.stat().st_size / 1024**2, 2),
            "MB"
        )

        success.append(filename)

    except Exception as e:
        print("  FAILED:", type(e).__name__, str(e)[:200])
        failed.append(filename)

print("\nDOWNLOAD SUMMARY")
print("Successful:", len(success))
print("Failed:", len(failed))

if failed:
    print("Failed files:", failed)

ct_path = destination / "CT_Data.h5"
structures_path = destination / "StructureSet_Data.h5"

for label, path in [
    ("CT", ct_path),
    ("Structures", structures_path)
]:
    if path.exists():
        print(f"\n{label} HDF5 contents:")

        with h5py.File(path, "r") as h5:
            def show_dataset(name, obj):
                if isinstance(obj, h5py.Dataset):
                    print(
                        " ",
                        name,
                        "| shape:",
                        obj.shape,
                        "| dtype:",
                        obj.dtype
                    )

            h5.visititems(show_dataset)

print("\nPatient 1 has not been modified.")
print("No preprocessing or model training performed.")

In [ ]:
%pip install -q pydicom

from pathlib import Path
import pydicom
import h5py
import numpy as np

print("STEP 149 — VALIDATE PATIENT 2 DOSE METADATA")
print("-" * 65)

base = Path(
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction/"
    "Patient_2/raw_portpy"
)

dose_path = base / "DicomFiles/rt_dose_echo_imrt.dcm"
ct_path = base / "CT_Data.h5"

dose = pydicom.dcmread(str(dose_path), stop_before_pixels=True)

with h5py.File(ct_path, "r") as f:
    ct_shape = f["ct_hu_3d"].shape

print("\nCT ARRAY")
print("Shape:", ct_shape)

print("\nDICOM DOSE")
print("Modality:", getattr(dose, "Modality", "Missing"))
print("Dose Units:", getattr(dose, "DoseUnits", "Missing"))
print("Dose Type:", getattr(dose, "DoseType", "Missing"))
print("Dose Summation Type:",
      getattr(dose, "DoseSummationType", "Missing"))

print("\nDOSE GRID")
print("Rows:", getattr(dose, "Rows", "Missing"))
print("Columns:", getattr(dose, "Columns", "Missing"))
print("Frames:", getattr(dose, "NumberOfFrames", "Missing"))

print("\nDOSE SCALING")
print("Dose Grid Scaling:",
      getattr(dose, "DoseGridScaling", "Missing"))

print("\nSPATIAL INFORMATION")
print("Image Position:",
      getattr(dose, "ImagePositionPatient", "Missing"))
print("Image Orientation:",
      getattr(dose, "ImageOrientationPatient", "Missing"))
print("Pixel Spacing:",
      getattr(dose, "PixelSpacing", "Missing"))

offsets = getattr(dose, "GridFrameOffsetVector", None)

if offsets is not None:
    print("Number of frame offsets:", len(offsets))
    print("First 5 offsets:", list(offsets[:5]))
else:
    print("Frame offsets: Missing")

print("\nREFERENCE IDENTIFIERS")
print("Frame of Reference UID:",
      getattr(dose, "FrameOfReferenceUID", "Missing"))

print("\nIMPORTANT:")
print("Matching array dimensions alone does not")
print("prove physical alignment.")
print("No files modified.")
print("No model training performed.")

In [ ]:
import json
import h5py
from pathlib import Path

print("STEP 150 — INSPECT PATIENT 2 CT GEOMETRY")
print("-" * 65)

base = Path(
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction/"
    "Patient_2/raw_portpy"
)

ct_json = base / "CT_MetaData.json"
ct_h5 = base / "CT_Data.h5"

print("\n1. CT METADATA JSON")
print("-" * 45)

with open(ct_json, "r") as f:
    metadata = json.load(f)

print(json.dumps(metadata, indent=2, default=str)[:12000])

print("\n2. CT HDF5 DATASETS AND ATTRIBUTES")
print("-" * 45)

with h5py.File(ct_h5, "r") as f:

    def inspect(name, obj):
        print("\nName:", name)
        print("Type:", type(obj).__name__)

        if isinstance(obj, h5py.Dataset):
            print("Shape:", obj.shape)
            print("Data type:", obj.dtype)

        for key, value in obj.attrs.items():
            print("Attribute:", key, "=", str(value)[:500])

    f.visititems(inspect)

    print("\nROOT ATTRIBUTES")
    for key, value in f.attrs.items():
        print(key, "=", str(value)[:500])

print("\nNo files modified.")
print("No model training performed.")

In [ ]:
from google.colab import drive
from pathlib import Path

print("STEP 151A — MOUNT GOOGLE DRIVE")
print("-" * 60)

drive.mount("/content/drive")

project = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction"
)

print("\nProject folder exists:", project.exists())

if project.exists():
    print("\nProject contents:")
    for item in sorted(project.iterdir()):
        print(" -", item.name)

    print("\nPatient 1 exists:", (project / "Patient_1").exists())
    print("Patient 2 exists:", (project / "Patient_2").exists())

print("\nNo files modified.")

In [ ]:
import json
import numpy as np
import pydicom
from pathlib import Path

print("STEP 151B — PATIENT 2 CT AND DOSE COVERAGE")
print("-" * 65)

base = Path(
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction/"
    "Patient_2/raw_portpy"
)

with open(base / "CT_MetaData.json") as f:
    ct = json.load(f)

dose = pydicom.dcmread(
    str(base / "DicomFiles/rt_dose_echo_imrt.dcm"),
    stop_before_pixels=True
)

ct_origin = np.array(ct["origin_xyz_mm"], dtype=float)
ct_spacing = np.array(ct["resolution_xyz_mm"], dtype=float)
ct_size = np.array(ct["size_xyz_mm"], dtype=int)
ct_direction = np.array(ct["direction"], dtype=float).reshape(3, 3)

dose_origin = np.array(dose.ImagePositionPatient, dtype=float)
orientation = np.array(dose.ImageOrientationPatient, dtype=float)
dose_offsets = np.array(dose.GridFrameOffsetVector, dtype=float)

if not np.allclose(ct_direction, np.eye(3)):
    raise ValueError("CT orientation requires general affine mapping.")

if not np.allclose(orientation, [1, 0, 0, 0, 1, 0]):
    raise ValueError("Dose orientation requires general affine mapping.")

ct_end = ct_origin + ct_spacing * (ct_size - 1)

dose_start = dose_origin.copy()
dose_start[2] += dose_offsets[0]

dose_end = np.array([
    dose_origin[0] + (int(dose.Columns) - 1) * float(dose.PixelSpacing[1]),
    dose_origin[1] + (int(dose.Rows) - 1) * float(dose.PixelSpacing[0]),
    dose_origin[2] + dose_offsets[-1]
])

print("\nCT COVERAGE (mm)")
for i, axis in enumerate("XYZ"):
    print(f"{axis}: {ct_origin[i]:.2f} to {ct_end[i]:.2f}")

print("\nDOSE COVERAGE (mm)")
for i, axis in enumerate("XYZ"):
    print(f"{axis}: {dose_start[i]:.2f} to {dose_end[i]:.2f}")

print("\nPHYSICAL OVERLAP")
for i, axis in enumerate("XYZ"):
    overlap = max(
        0,
        min(ct_end[i], dose_end[i]) -
        max(ct_origin[i], dose_start[i])
    )
    print(f"{axis}: {overlap:.2f} mm")

print("\nDOSE FRAME SPACING")
print(np.unique(np.round(np.diff(dose_offsets), 5)))

print("\nDOSE FRAME OF REFERENCE UID")
print(getattr(dose, "FrameOfReferenceUID", "Missing"))

print("\nGeometric overlap does not by itself prove registration.")
print("No files modified.")
print("No model training performed.")

In [ ]:
%pip install -q pydicom

import pydicom

print("STEP 151C — PYDICOM INSTALLATION")
print("-" * 50)
print("pydicom version:", pydicom.__version__)
print("Installation successful!")
print("No patient files modified.")

In [ ]:
import json
import numpy as np
import pydicom
from pathlib import Path

print("STEP 151B — PATIENT 2 CT AND DOSE COVERAGE")
print("-" * 65)

base = Path(
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction/"
    "Patient_2/raw_portpy"
)

with open(base / "CT_MetaData.json") as f:
    ct = json.load(f)

dose = pydicom.dcmread(
    str(base / "DicomFiles/rt_dose_echo_imrt.dcm"),
    stop_before_pixels=True
)

ct_origin = np.array(ct["origin_xyz_mm"], dtype=float)
ct_spacing = np.array(ct["resolution_xyz_mm"], dtype=float)
ct_size = np.array(ct["size_xyz_mm"], dtype=int)
ct_direction = np.array(ct["direction"], dtype=float).reshape(3, 3)

dose_origin = np.array(dose.ImagePositionPatient, dtype=float)
orientation = np.array(dose.ImageOrientationPatient, dtype=float)
dose_offsets = np.array(dose.GridFrameOffsetVector, dtype=float)

if not np.allclose(ct_direction, np.eye(3)):
    raise ValueError("CT orientation requires general affine mapping.")

if not np.allclose(orientation, [1, 0, 0, 0, 1, 0]):
    raise ValueError("Dose orientation requires general affine mapping.")

ct_end = ct_origin + ct_spacing * (ct_size - 1)

dose_start = dose_origin.copy()
dose_start[2] += dose_offsets[0]

dose_end = np.array([
    dose_origin[0] + (int(dose.Columns) - 1) * float(dose.PixelSpacing[1]),
    dose_origin[1] + (int(dose.Rows) - 1) * float(dose.PixelSpacing[0]),
    dose_origin[2] + dose_offsets[-1]
])

print("\nCT COVERAGE (mm)")
for i, axis in enumerate("XYZ"):
    print(f"{axis}: {ct_origin[i]:.2f} to {ct_end[i]:.2f}")

print("\nDOSE COVERAGE (mm)")
for i, axis in enumerate("XYZ"):
    print(f"{axis}: {dose_start[i]:.2f} to {dose_end[i]:.2f}")

print("\nPHYSICAL OVERLAP")
for i, axis in enumerate("XYZ"):
    overlap = max(
        0,
        min(ct_end[i], dose_end[i]) -
        max(ct_origin[i], dose_start[i])
    )
    print(f"{axis}: {overlap:.2f} mm")

print("\nDOSE FRAME SPACING")
print(np.unique(np.round(np.diff(dose_offsets), 5)))

print("\nDOSE FRAME OF REFERENCE UID")
print(getattr(dose, "FrameOfReferenceUID", "Missing"))

print("\nGeometric overlap does not by itself prove registration.")
print("No files modified.")
print("No model training performed.")

In [ ]:
import json
import pydicom
from pathlib import Path

print("STEP 152 — VERIFY PATIENT 2 COORDINATE REFERENCES")
print("-" * 65)

base = Path(
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction/"
    "Patient_2/raw_portpy"
)

dose_path = base / "DicomFiles/rt_dose_echo_imrt.dcm"
plan_path = base / "DicomFiles/rt_plan_echo_imrt.dcm"

dose = pydicom.dcmread(
    str(dose_path),
    stop_before_pixels=True
)

plan = pydicom.dcmread(
    str(plan_path),
    stop_before_pixels=True
)

print("\n1. DOSE IDENTIFIERS")
print("Study UID:", getattr(dose, "StudyInstanceUID", "Missing"))
print("Frame UID:", getattr(dose, "FrameOfReferenceUID", "Missing"))

print("\n2. PLAN IDENTIFIERS")
print("Study UID:", getattr(plan, "StudyInstanceUID", "Missing"))
print("Frame UID:", getattr(plan, "FrameOfReferenceUID", "Missing"))

print("\n3. PLAN FRAME-OF-REFERENCE SEQUENCE")

for item in getattr(plan, "ReferencedFrameOfReferenceSequence", []):
    print(
        "Referenced Frame UID:",
        getattr(item, "FrameOfReferenceUID", "Missing")
    )

print("\n4. DOSE REFERENCED PLAN")

for item in getattr(dose, "ReferencedRTPlanSequence", []):
    print(
        "Referenced Plan UID:",
        getattr(item, "ReferencedSOPInstanceUID", "Missing")
    )

print("\n5. ACTUAL PLAN SOP INSTANCE UID")
print(getattr(plan, "SOPInstanceUID", "Missing"))

print("\n6. CT METADATA")
with open(base / "CT_MetaData.json") as f:
    ct_metadata = json.load(f)

print("CT origin:", ct_metadata.get("origin_xyz_mm"))
print("CT resolution:", ct_metadata.get("resolution_xyz_mm"))
print("CT direction:", ct_metadata.get("direction"))

print("\nIMPORTANT:")
print("The PortPy CT metadata does not contain a Frame UID.")
print("Matching dose and plan references alone cannot")
print("prove CT-to-dose registration.")

print("\nNo files modified.")
print("No model training performed.")

In [ ]:
import pydicom
from pathlib import Path

print("STEP 153 — INSPECT PATIENT 2 PLAN REFERENCES")
print("-" * 65)

base = Path(
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction/"
    "Patient_2/raw_portpy"
)

plan_path = base / "DicomFiles/rt_plan_echo_imrt.dcm"

plan = pydicom.dcmread(
    str(plan_path),
    stop_before_pixels=True
)

print("\n1. PLAN FRAME OF REFERENCE")
print(
    getattr(plan, "FrameOfReferenceUID", "Missing")
)

print("\n2. REFERENCED STRUCTURE SETS")

structure_refs = getattr(
    plan,
    "ReferencedStructureSetSequence",
    []
)

print("Number of references:", len(structure_refs))

for i, item in enumerate(structure_refs):
    print(f"\nStructure reference {i + 1}")
    print(
        "Referenced SOP Class UID:",
        getattr(item, "ReferencedSOPClassUID", "Missing")
    )
    print(
        "Referenced SOP Instance UID:",
        getattr(item, "ReferencedSOPInstanceUID", "Missing")
    )

print("\n3. OTHER FRAME OF REFERENCE REFERENCES")

for item in getattr(
    plan,
    "ReferencedFrameOfReferenceSequence",
    []
):
    print(
        getattr(item, "FrameOfReferenceUID", "Missing")
    )

print("\n4. AVAILABLE LOCAL DICOM FILES")

dicom_folder = base / "DicomFiles"

for file in sorted(dicom_folder.iterdir()):
    print(file.name)

print("\nNo files downloaded.")
print("No existing files modified.")
print("No model training performed.")

In [ ]:
from huggingface_hub import HfApi, hf_hub_download
from pathlib import Path
import pydicom

print("STEP 154 — DOWNLOAD AND VERIFY PATIENT 2 RTSTRUCT")
print("-" * 65)

repo_id = "PortPy-Project/PortPy_Dataset"

base = Path(
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction/"
    "Patient_2/raw_portpy"
)

plan = pydicom.dcmread(
    str(base / "DicomFiles/rt_plan_echo_imrt.dcm"),
    stop_before_pixels=True
)

expected_uid = str(
    plan.ReferencedStructureSetSequence[0].ReferencedSOPInstanceUID
)

print("RTSTRUCT UID required by plan:", expected_uid)

api = HfApi()

# Obtain Patient 2's remote file paths.
remote_files = api.list_repo_tree(
    repo_id=repo_id,
    repo_type="dataset",
    path_in_repo="data/Prostate_Patient_2/DicomFiles",
    recursive=False
)

candidates = [
    item.path
    for item in remote_files
    if item.path.split("/")[-1].startswith("RS.")
]

print("\nRTSTRUCT candidates:", len(candidates))

matched = False

for remote_path in candidates:
    print("Checking:", remote_path.split("/")[-1])

    downloaded = hf_hub_download(
        repo_id=repo_id,
        repo_type="dataset",
        filename=remote_path,
        local_dir=str(base / "_rtstruct_cache")
    )

    ds = pydicom.dcmread(
        downloaded,
        stop_before_pixels=True
    )

    actual_uid = str(getattr(ds, "SOPInstanceUID", ""))

    if actual_uid == expected_uid:
        destination = base / "DicomFiles" / Path(remote_path).name
        destination.parent.mkdir(parents=True, exist_ok=True)

        import shutil
        shutil.copy2(downloaded, destination)

        print("\nMATCH FOUND!")
        print("Saved to:", destination)
        print("RTSTRUCT SOP UID:", actual_uid)

        print("\nRTSTRUCT FRAME REFERENCES")

        for ref in getattr(
            ds,
            "ReferencedFrameOfReferenceSequence",
            []
        ):
            frame_uid = str(
                getattr(ref, "FrameOfReferenceUID", "")
            )

            print("RTSTRUCT Frame UID:", frame_uid)
            print(
                "Matches plan:",
                frame_uid == str(plan.FrameOfReferenceUID)
            )

        matched = True
        break

if not matched:
    print("\nNo matching RTSTRUCT was found.")

print("\nPatient 1 and model checkpoints were not modified.")
print("No model training performed.")

In [ ]:
from huggingface_hub import HfApi, hf_hub_download
from pathlib import Path
import pydicom

print("STEP 155 — VERIFY PATIENT 2 CT FRAME OF REFERENCE")
print("-" * 65)

repo_id = "PortPy-Project/PortPy_Dataset"

base = Path(
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction/"
    "Patient_2/raw_portpy"
)

dose = pydicom.dcmread(
    str(base / "DicomFiles/rt_dose_echo_imrt.dcm"),
    stop_before_pixels=True
)

expected_frame_uid = str(dose.FrameOfReferenceUID)

api = HfApi()

remote_files = api.list_repo_tree(
    repo_id=repo_id,
    repo_type="dataset",
    path_in_repo="data/Prostate_Patient_2/DicomFiles",
    recursive=False
)

ct_files = sorted(
    item.path
    for item in remote_files
    if item.path.split("/")[-1].startswith("CT.")
)

print("CT DICOM files available:", len(ct_files))
print("Expected Frame UID:", expected_frame_uid)

if not ct_files:
    print("\nNo CT DICOM files found.")
else:
    remote_path = ct_files[0]

    print("\nDownloading one CT DICOM header file:")
    print(Path(remote_path).name)

    downloaded = hf_hub_download(
        repo_id=repo_id,
        repo_type="dataset",
        filename=remote_path,
        local_dir="/content/portpy_ct_reference_check"
    )

    ct = pydicom.dcmread(
        downloaded,
        stop_before_pixels=True
    )

    ct_frame_uid = str(
        getattr(ct, "FrameOfReferenceUID", "")
    )

    print("\nCT DICOM INFORMATION")
    print("Modality:", getattr(ct, "Modality", "Missing"))
    print("Frame UID:", ct_frame_uid)
    print("Image Position:",
          getattr(ct, "ImagePositionPatient", "Missing"))
    print("Image Orientation:",
          getattr(ct, "ImageOrientationPatient", "Missing"))
    print("Pixel Spacing:",
          getattr(ct, "PixelSpacing", "Missing"))

    print("\nFRAME OF REFERENCE CHECK")

    if ct_frame_uid and ct_frame_uid == expected_frame_uid:
        print("PASS — CT and dose share the same Frame UID.")
    else:
        print("NOT VERIFIED — CT and dose Frame UIDs differ or are missing.")

    print("\nNote: This checks one CT DICOM instance.")
    print("The full CT series and PortPy CT geometry still")
    print("need consistency checks before dose resampling.")

print("\nNo existing project files modified.")
print("No model training performed.")

In [ ]:
import json
import numpy as np
import pydicom
from pathlib import Path

print("STEP 156 — CHECK PORTPY CT GEOMETRY")
print("-" * 65)

base = Path(
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction/"
    "Patient_2/raw_portpy"
)

with open(base / "CT_MetaData.json") as f:
    metadata = json.load(f)

# Find the CT DICOM file downloaded in Step 155
ct_folder = Path(
    "/content/portpy_ct_reference_check/"
    "data/Prostate_Patient_2/DicomFiles"
)

ct_files = sorted(ct_folder.glob("CT.*"))

if not ct_files:
    raise FileNotFoundError(
        "CT DICOM sample missing. Rerun Step 155."
    )

ct = pydicom.dcmread(
    str(ct_files[0]),
    stop_before_pixels=True
)

portpy_spacing = np.array(
    metadata["resolution_xyz_mm"],
    dtype=float
)

dicom_spacing = np.array(
    [
        float(ct.PixelSpacing[1]),
        float(ct.PixelSpacing[0]),
        float(ct.SliceThickness)
    ]
)

print("\nPORTPY CT")
print("Origin:", metadata["origin_xyz_mm"])
print("Spacing:", portpy_spacing)
print("Size:", metadata["size_xyz_mm"])
print("Direction:", metadata["direction"])

print("\nORIGINAL CT DICOM SAMPLE")
print("Rows:", ct.Rows)
print("Columns:", ct.Columns)
print("Pixel spacing:", ct.PixelSpacing)
print("Slice thickness:", getattr(ct, "SliceThickness", "Missing"))
print("Image position:", ct.ImagePositionPatient)
print("Orientation:", ct.ImageOrientationPatient)

print("\nSPACING COMPARISON")
print("PortPy:", portpy_spacing)
print("DICOM:", dicom_spacing)
print("Absolute difference:", np.abs(portpy_spacing - dicom_spacing))

print("\nIN-PLANE DIMENSION COMPARISON")
print(
    "Rows match:",
    int(ct.Rows) == int(metadata["size_xyz_mm"][1])
)
print(
    "Columns match:",
    int(ct.Columns) == int(metadata["size_xyz_mm"][0])
)

print("\nNOTE:")
print("The downloaded CT is one slice, not necessarily")
print("the first slice in the volume.")
print("Slice thickness is not proof of slice spacing.")
print("Full-series geometry is not yet verified.")

print("\nNo files modified.")
print("No model training performed.")

In [ ]:
import json
import numpy as np
import pydicom
from pathlib import Path

print("STEP 157 — VERIFY CT SLICE POSITION")
print("-" * 65)

base = Path(
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction/"
    "Patient_2/raw_portpy"
)

with open(base / "CT_MetaData.json") as f:
    metadata = json.load(f)

ct_folder = Path(
    "/content/portpy_ct_reference_check/"
    "data/Prostate_Patient_2/DicomFiles"
)

ct_files = sorted(ct_folder.glob("CT.*"))

if not ct_files:
    raise FileNotFoundError(
        "CT DICOM sample missing. Rerun Step 155."
    )

ct = pydicom.dcmread(
    str(ct_files[0]),
    stop_before_pixels=True
)

origin = np.array(metadata["origin_xyz_mm"], dtype=float)
spacing = np.array(metadata["resolution_xyz_mm"], dtype=float)
size = np.array(metadata["size_xyz_mm"], dtype=int)

sample_position = np.array(
    ct.ImagePositionPatient,
    dtype=float
)

z_index_float = (
    sample_position[2] - origin[2]
) / spacing[2]

z_index = int(round(z_index_float))

expected_position = (
    origin[2] + z_index * spacing[2]
)

print("\nPORTPY VOLUME")
print("First Z position:", origin[2])
print("Number of slices:", size[2])
print("Nominal Z spacing:", spacing[2])
print(
    "Last Z position:",
    origin[2] + (size[2] - 1) * spacing[2]
)

print("\nSAMPLED DICOM CT SLICE")
print("Actual Z position:", sample_position[2])
print("Calculated slice index:", z_index_float)
print("Nearest integer index:", z_index)
print("Expected Z position:", expected_position)

print("\nVALIDATION")

within_volume = 0 <= z_index < size[2]

position_matches = np.isclose(
    sample_position[2],
    expected_position,
    atol=1e-3
)

print("Slice index within volume:", within_volume)
print("Sample position matches grid:", position_matches)

if within_volume and position_matches:
    print(
        "\nPASS — Sampled CT slice position is "
        "consistent with PortPy geometry."
    )
else:
    print(
        "\nWARNING — Sampled CT slice position "
        "requires further investigation."
    )

print("\nIMPORTANT:")
print("One matching slice does not verify all 163 slices.")
print("Full-series ordering and spacing remain unverified.")

print("\nNo files modified.")
print("No model training performed.")

In [ ]:
import pydicom
import numpy as np
from pathlib import Path

print("STEP 158 — PATIENT 2 DOSE AND FRACTIONATION CHECK")
print("-" * 65)

base = Path(
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction/"
    "Patient_2/raw_portpy"
)

plan = pydicom.dcmread(
    str(base / "DicomFiles/rt_plan_echo_imrt.dcm"),
    stop_before_pixels=True
)

dose = pydicom.dcmread(
    str(base / "DicomFiles/rt_dose_echo_imrt.dcm")
)

print("\n1. DOSE INFORMATION")
print("Dose units:", getattr(dose, "DoseUnits", "Missing"))
print("Dose type:", getattr(dose, "DoseType", "Missing"))
print(
    "Dose summation type:",
    getattr(dose, "DoseSummationType", "Missing")
)

print("\n2. FRACTIONATION INFORMATION")

fraction_groups = getattr(
    plan,
    "FractionGroupSequence",
    []
)

print("Number of fraction groups:", len(fraction_groups))

fractions = []

for i, group in enumerate(fraction_groups):
    n = getattr(group, "NumberOfFractionsPlanned", None)

    print(f"Group {i + 1}:")
    print("  Planned fractions:", n)
    print(
        "  Number of beams:",
        getattr(group, "NumberOfBeams", "Missing")
    )

    if n is not None:
        fractions.append(int(n))

print("\n3. DOSE STATISTICS")

# Apply the DICOM dose scaling factor.
dose_gy = (
    dose.pixel_array.astype(np.float32)
    * float(dose.DoseGridScaling)
)

print("Dose array shape:", dose_gy.shape)
print("Minimum dose (Gy):", float(dose_gy.min()))
print("Maximum dose (Gy):", float(dose_gy.max()))
print("Mean dose (Gy):", float(dose_gy.mean()))

print("\n4. PER-FRACTION INTERPRETATION")

if (
    len(fractions) == 1
    and fractions[0] > 0
    and getattr(dose, "DoseSummationType", "") == "PLAN"
):
    n = fractions[0]

    print("Planned fractions:", n)
    print(
        "Maximum plan dose / fractions (Gy):",
        float(dose_gy.max()) / n
    )
    print(
        "This division is only an arithmetic comparison."
    )
    print(
        "It does not establish equivalence to "
        "the PortPy-optimized dose."
    )
else:
    print(
        "Cannot establish a single fraction count "
        "for this plan dose."
    )

print("\nNo files modified.")
print("No model training performed.")

In [ ]:
import json
import h5py
from pathlib import Path

print("STEP 159 — INSPECT PATIENT 2 OPTIMIZATION DATA")
print("-" * 65)

base = Path(
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction/"
    "Patient_2/raw_portpy"
)

data_path = base / "OptimizationVoxels_Data.h5"
metadata_path = base / "OptimizationVoxels_MetaData.json"

print("\n1. FILE AVAILABILITY")
print("Optimization HDF5 exists:", data_path.exists())
print("Optimization metadata exists:", metadata_path.exists())

if not data_path.exists() or not metadata_path.exists():
    raise FileNotFoundError(
        "Required optimization files are missing."
    )

print("\n2. OPTIMIZATION METADATA")

with open(metadata_path) as f:
    metadata = json.load(f)

print("Top-level metadata keys:")
for key in metadata:
    print(" -", key)

print("\n3. HDF5 DATASETS")

with h5py.File(data_path, "r") as f:

    def inspect(name, obj):
        if isinstance(obj, h5py.Dataset):
            print("\nDataset:", name)
            print("  Shape:", obj.shape)
            print("  Dtype:", obj.dtype)

    f.visititems(inspect)

print("\n4. INTERPRETATION")
print(
    "Optimization voxel data alone does not establish "
    "that an optimized dose distribution is available."
)
print(
    "We must distinguish optimization inputs "
    "from optimization outputs."
)

print("\nNo files modified.")
print("No optimization performed.")
print("No model training performed.")

In [ ]:
import json
import h5py
import numpy as np
from pathlib import Path

print("STEP 160 — EXAMINE PORTPY DOSE-GRID MAPPING")
print("-" * 65)

base = Path(
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction/"
    "Patient_2/raw_portpy"
)

metadata_path = base / "OptimizationVoxels_MetaData.json"
data_path = base / "OptimizationVoxels_Data.h5"

with open(metadata_path, "r") as f:
    metadata = json.load(f)

print("\n1. OPTIMIZATION GRID METADATA")

for key, value in metadata.items():
    print(f"{key}: {value}")

print("\n2. LOAD GRID MAPPING INFORMATION")

with h5py.File(data_path, "r") as f:
    mapping = f["ct_to_dose_voxel_map"][:]
    coordinates = f["voxel_coordinate_XYZ_mm"][:]

print("CT-to-dose mapping shape:", mapping.shape)
print("Dose voxel coordinates shape:", coordinates.shape)

print("\n3. MAPPING STATISTICS")

unique_values, counts = np.unique(
    mapping,
    return_counts=True
)

print("Minimum mapping index:", int(mapping.min()))
print("Maximum mapping index:", int(mapping.max()))
print("Number of unique mapping values:", len(unique_values))

print("\nMost common mapping values:")

top_indices = np.argsort(counts)[-10:][::-1]

for idx in top_indices:
    print(
        f"  Value {int(unique_values[idx])}: "
        f"{int(counts[idx]):,} CT voxels"
    )

print("\n4. COORDINATE RANGES")

for i, axis in enumerate("XYZ"):
    values = coordinates[:, i]

    print(
        f"{axis}: "
        f"{float(values.min()):.2f} to "
        f"{float(values.max()):.2f} mm"
    )

print("\n5. INDEX VALIDITY")

n_dose_voxels = len(coordinates)

valid = (mapping >= 0) & (mapping < n_dose_voxels)

print("Total CT voxels:", mapping.size)
print("Valid mapped CT voxels:", int(valid.sum()))
print("Other CT voxels:", int((~valid).sum()))

print("\n6. INTERPRETATION")

print(
    "These indices describe the PortPy optimization grid."
)
print(
    "They are not assumed to index the DICOM dose array."
)
print(
    "No dose interpolation has been performed."
)

print("\nNo files modified.")
print("No model training performed.")

In [ ]:
import json
import h5py
import numpy as np
import pydicom
from pathlib import Path

print("STEP 161 — PATIENT 2 PTV AND DOSE COVERAGE")
print("-" * 65)

base = Path(
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction/"
    "Patient_2/raw_portpy"
)

with open(base / "CT_MetaData.json") as f:
    ct_meta = json.load(f)

with h5py.File(base / "StructureSet_Data.h5", "r") as f:
    ptv = f["PTV"][:]

dose = pydicom.dcmread(
    str(base / "DicomFiles/rt_dose_echo_imrt.dcm"),
    stop_before_pixels=True
)

print("\n1. PTV INFORMATION")
print("PTV shape:", ptv.shape)
print("PTV voxel count:", int(np.count_nonzero(ptv)))

z, y, x = np.where(ptv > 0)

if len(x) == 0:
    raise ValueError("PTV mask is empty.")

origin = np.asarray(ct_meta["origin_xyz_mm"], dtype=float)
spacing = np.asarray(ct_meta["resolution_xyz_mm"], dtype=float)

# Use original DICOM in-plane spacing, verified in Step 156.
dicom_xy_spacing = 0.7890625

ptv_min = np.array([
    origin[0] + x.min() * dicom_xy_spacing,
    origin[1] + y.min() * dicom_xy_spacing,
    origin[2] + z.min() * spacing[2]
])

ptv_max = np.array([
    origin[0] + x.max() * dicom_xy_spacing,
    origin[1] + y.max() * dicom_xy_spacing,
    origin[2] + z.max() * spacing[2]
])

print("\n2. PTV PHYSICAL BOUNDING BOX")

for i, axis in enumerate("XYZ"):
    print(
        f"{axis}: {ptv_min[i]:.2f} "
        f"to {ptv_max[i]:.2f} mm"
    )

print("\n3. DICOM DOSE COVERAGE")

dose_origin = np.asarray(
    dose.ImagePositionPatient, dtype=float
)

offsets = np.asarray(
    dose.GridFrameOffsetVector, dtype=float
)

dose_min = np.array([
    dose_origin[0],
    dose_origin[1],
    dose_origin[2] + offsets.min()
])

dose_max = np.array([
    dose_origin[0]
    + (int(dose.Columns) - 1) * float(dose.PixelSpacing[1]),

    dose_origin[1]
    + (int(dose.Rows) - 1) * float(dose.PixelSpacing[0]),

    dose_origin[2] + offsets.max()
])

for i, axis in enumerate("XYZ"):
    print(
        f"{axis}: {dose_min[i]:.2f} "
        f"to {dose_max[i]:.2f} mm"
    )

print("\n4. DOES THE DOSE GRID CONTAIN THE PTV?")

fully_contained = True

for i, axis in enumerate("XYZ"):
    contained = (
        ptv_min[i] >= dose_min[i]
        and ptv_max[i] <= dose_max[i]
    )

    print(f"{axis}: {contained}")
    fully_contained &= contained

print("\nOverall PTV bounding-box containment:", fully_contained)

print("\nIMPORTANT:")
print("This checks physical bounding-box containment only.")
print("It does not verify delivered dose or dose quality.")
print("The original CT series geometry still needs full validation.")

print("\nNo files modified.")
print("No model training performed.")


In [ ]:
import json
import h5py
import numpy as np
import pydicom
from scipy.ndimage import map_coordinates
from pathlib import Path

print("STEP 162 — PATIENT 2 PTV DOSE VALIDATION")
print("-" * 60)

base = Path(
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction/"
    "Patient_2/raw_portpy"
)

with open(base / "CT_MetaData.json") as f:
    meta = json.load(f)

with h5py.File(base / "StructureSet_Data.h5") as f:
    ptv = f["PTV"][:].astype(bool)

dose = pydicom.dcmread(
    str(base / "DicomFiles/rt_dose_echo_imrt.dcm")
)

dose_gy = dose.pixel_array.astype(np.float32)
dose_gy *= float(dose.DoseGridScaling)

# Verified DICOM geometry from earlier steps.
ct_origin = np.array(meta["origin_xyz_mm"], dtype=float)
ct_xy = 0.7890625
ct_z = float(meta["resolution_xyz_mm"][2])

dose_origin = np.array(dose.ImagePositionPatient, dtype=float)
dose_spacing = np.array(dose.PixelSpacing, dtype=float)
offsets = np.array(dose.GridFrameOffsetVector, dtype=float)

# Use only PTV voxels to keep this test efficient.
z, y, x = np.where(ptv)

physical_x = ct_origin[0] + x * ct_xy
physical_y = ct_origin[1] + y * ct_xy
physical_z = ct_origin[2] + z * ct_z

# Dose orientation was verified as axial identity.
dose_x = (physical_x - dose_origin[0]) / dose_spacing[1]
dose_y = (physical_y - dose_origin[1]) / dose_spacing[0]

# Convert physical Z positions to fractional frame indices.
dose_z = np.interp(
    physical_z - dose_origin[2],
    offsets,
    np.arange(len(offsets)),
    left=np.nan,
    right=np.nan
)

valid = (
    np.isfinite(dose_z)
    & (dose_x >= 0)
    & (dose_x <= dose.Columns - 1)
    & (dose_y >= 0)
    & (dose_y <= dose.Rows - 1)
)

print("PTV voxels:", len(x))
print("PTV voxels inside dose grid:", int(valid.sum()))
print("Coverage:", round(100 * valid.mean(), 2), "%")

if not valid.all():
    raise ValueError("Some PTV voxels fall outside the dose grid.")

ptv_dose = map_coordinates(
    dose_gy,
    [dose_z, dose_y, dose_x],
    order=1,
    mode="nearest"
)

fractions = int(
    dose_plan_fractions
) if "dose_plan_fractions" in globals() else 26

print("\nPTV DOSE STATISTICS")
print("Minimum total dose:", round(float(ptv_dose.min()), 3), "Gy")
print("Mean total dose:", round(float(ptv_dose.mean()), 3), "Gy")
print("Maximum total dose:", round(float(ptv_dose.max()), 3), "Gy")

print("\nARITHMETIC DOSE PER PLANNED FRACTION")
print("Mean:", round(float(ptv_dose.mean()) / fractions, 3), "Gy")
print("Maximum:", round(float(ptv_dose.max()) / fractions, 3), "Gy")

print("\nNo files modified.")
print("No training performed.")

In [ ]:
from pathlib import Path

print("STEP 163 — MULTI-PATIENT PREPROCESSING READINESS")
print("-" * 65)

project = Path(
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction"
)

required_files = [
    "CT_Data.h5",
    "CT_MetaData.json",
    "StructureSet_Data.h5",
    "StructureSet_MetaData.json",
    "DicomFiles/rt_dose_echo_imrt.dcm",
    "DicomFiles/rt_plan_echo_imrt.dcm",
]

patient_dirs = sorted(
    [
        p for p in project.iterdir()
        if p.is_dir()
        and p.name.startswith("Patient_")
    ],
    key=lambda p: int(p.name.split("_")[1])
)

print("\nPATIENT READINESS")

ready = []

for patient in patient_dirs:
    raw = patient / "raw_portpy"

    missing = [
        filename
        for filename in required_files
        if not (raw / filename).exists()
    ]

    if not missing:
        ready.append(patient.name)
        print(f"{patient.name}: READY FOR PREPROCESSING TESTS")
    else:
        print(f"{patient.name}: INCOMPLETE")
        print("  Missing:", ", ".join(missing))

print("\nSUMMARY")
print("Patient folders:", len(patient_dirs))
print("Patients with required files:", len(ready))
print("Ready patients:", ready)

print("\nNEXT ACTIONS")
print("1. Automate CT-to-dose geometry validation.")
print("2. Resample clinical dose using physical coordinates.")
print("3. Validate PTV dose and fractionation.")
print("4. Generate consistent Stage 1 training arrays.")
print("5. Scale to additional patients.")

print("\nNo files modified.")
print("No downloads.")
print("No model training.")

In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download
import pydicom
import shutil

print("STEP 164 — COMPLETE PATIENT 1 CLINICAL DICOM FILES")
print("-" * 65)

project = Path(
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction"
)

destination = project / "Patient_1/raw_portpy/DicomFiles"
destination.mkdir(parents=True, exist_ok=True)

repo_id = "PortPy-Project/PortPy_Dataset"

required = [
    "rt_dose_echo_imrt.dcm",
    "rt_plan_echo_imrt.dcm"
]

for filename in required:

    target = destination / filename

    if target.exists():
        print("ALREADY EXISTS:", filename)
        continue

    remote_path = (
        "data/Prostate_Patient_1/"
        "DicomFiles/" + filename
    )

    downloaded = hf_hub_download(
        repo_id=repo_id,
        repo_type="dataset",
        filename=remote_path,
        local_dir="/content/portpy_patient1_download"
    )

    shutil.copy2(downloaded, target)

    print("DOWNLOADED:", filename)
    print("Size:", round(target.stat().st_size / 1e6, 2), "MB")

print("\nVERIFY PATIENT 1 FILES")

dose_path = destination / required[0]
plan_path = destination / required[1]

dose = pydicom.dcmread(
    str(dose_path),
    stop_before_pixels=True
)

plan = pydicom.dcmread(
    str(plan_path),
    stop_before_pixels=True
)

print("Dose modality:", dose.Modality)
print("Plan modality:", plan.Modality)
print(
    "Frame UIDs match:",
    getattr(dose, "FrameOfReferenceUID", None)
    == getattr(plan, "FrameOfReferenceUID", None)
)

print("\nNo existing optimized dose modified.")
print("No checkpoints modified.")
print("No model training performed.")

In [ ]:
# STEP 165 — AUTOMATED MULTI-PATIENT PREPROCESSING TEST

import json
import h5py
import numpy as np
import pydicom
from pathlib import Path
from scipy.ndimage import map_coordinates, zoom

print("STEP 165 — MULTI-PATIENT PREPROCESSING")
print("=" * 65)

project = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction"
)

output_root = project / "Stage1_MultiPatient_Provisional"
output_root.mkdir(parents=True, exist_ok=True)

def preprocess_patient(patient_id):

    name = f"Patient_{patient_id}"
    base = project / name / "raw_portpy"

    print(f"\nPROCESSING {name}")

    with open(base / "CT_MetaData.json") as f:
        meta = json.load(f)

    with h5py.File(base / "CT_Data.h5") as f:
        ct = f["ct_hu_3d"][:]

    with h5py.File(base / "StructureSet_Data.h5") as f:
        ptv = f["PTV"][:].astype(bool)

    dose = pydicom.dcmread(
        str(base / "DicomFiles/rt_dose_echo_imrt.dcm")
    )

    plan = pydicom.dcmread(
        str(base / "DicomFiles/rt_plan_echo_imrt.dcm"),
        stop_before_pixels=True
    )

    assert ct.shape == ptv.shape

    if str(dose.FrameOfReferenceUID) != str(plan.FrameOfReferenceUID):
        raise ValueError("Dose/plan frame mismatch")

    orientation = np.asarray(
        dose.ImageOrientationPatient, dtype=float
    )

    if not np.allclose(
        orientation, [1, 0, 0, 0, 1, 0]
    ):
        raise ValueError("Unsupported dose orientation")

    direction = np.asarray(
        meta["direction"], dtype=float
    ).reshape(3, 3)

    if not np.allclose(direction, np.eye(3)):
        raise ValueError("Unsupported CT orientation")

    groups = plan.FractionGroupSequence

    if len(groups) != 1:
        raise ValueError("Expected one fraction group")

    fractions = int(groups[0].NumberOfFractionsPlanned)

    if fractions <= 0:
        raise ValueError("Invalid fraction count")

    if str(dose.DoseUnits) != "GY":
        raise ValueError("Dose units are not Gy")

    if str(dose.DoseSummationType) != "PLAN":
        raise ValueError("Dose is not a PLAN dose")

    dose_gy = (
        dose.pixel_array.astype(np.float32)
        * float(dose.DoseGridScaling)
    )

    dose_origin = np.asarray(
        dose.ImagePositionPatient, dtype=float
    )

    dose_spacing = np.asarray(
        dose.PixelSpacing, dtype=float
    )

    offsets = np.asarray(
        dose.GridFrameOffsetVector, dtype=float
    )

    if not np.all(np.diff(offsets) > 0):
        raise ValueError("Dose frame offsets not increasing")

    origin = np.asarray(
        meta["origin_xyz_mm"], dtype=float
    )

    spacing = np.asarray(
        meta["resolution_xyz_mm"], dtype=float
    )

    # Metadata spacing is used provisionally.
    # Exact original DICOM geometry requires separate verification.
    z_slices = np.where(ptv.any(axis=(1, 2)))[0]

    if len(z_slices) == 0:
        raise ValueError("Empty PTV")

    first = max(0, int(z_slices.min()) - 5)
    last = min(ct.shape[0] - 1, int(z_slices.max()) + 5)

    yy, xx = np.meshgrid(
        np.arange(ct.shape[1]),
        np.arange(ct.shape[2]),
        indexing="ij"
    )

    physical_x = origin[0] + xx * spacing[0]
    physical_y = origin[1] + yy * spacing[1]

    dose_x = (
        physical_x - dose_origin[0]
    ) / dose_spacing[1]

    dose_y = (
        physical_y - dose_origin[1]
    ) / dose_spacing[0]

    ct_output = []
    ptv_output = []
    dose_output = []

    for z in range(first, last + 1):

        physical_z = origin[2] + z * spacing[2]

        dose_z = np.interp(
            physical_z - dose_origin[2],
            offsets,
            np.arange(len(offsets)),
            left=np.nan,
            right=np.nan
        )

        if not np.isfinite(dose_z):
            raise ValueError(
                f"Dose grid excludes selected CT slice {z}"
            )

        valid = (
            (dose_x >= 0)
            & (dose_x <= dose.Columns - 1)
            & (dose_y >= 0)
            & (dose_y <= dose.Rows - 1)
        )

        # Zero outside dose coverage; don't extrapolate.
        coordinates = [
            np.full(xx.shape, dose_z),
            dose_y,
            dose_x
        ]

        sampled = map_coordinates(
            dose_gy,
            coordinates,
            order=1,
            mode="constant",
            cval=0.0
        )

        sampled[~valid] = 0

        # Provisional preprocessing, not yet paper-verified.
        ct_slice = np.clip(
            (ct[z].astype(np.float32) + 1000) / 4000,
            0,
            1
        )

        ptv_slice = ptv[z].astype(np.float32)

        dose_slice = sampled / fractions

        # Provisional in-plane resizing.
        scale_y = 128 / ct_slice.shape[0]
        scale_x = 128 / ct_slice.shape[1]

        ct_output.append(
            zoom(ct_slice, (scale_y, scale_x), order=1)
        )

        ptv_output.append(
            zoom(ptv_slice, (scale_y, scale_x), order=0)
        )

        dose_output.append(
            zoom(dose_slice, (scale_y, scale_x), order=1)
        )

    ct_output = np.asarray(ct_output, dtype=np.float32)
    ptv_output = np.asarray(ptv_output, dtype=np.float32)
    dose_output = np.asarray(dose_output, dtype=np.float32)

    destination = output_root / name
    destination.mkdir(parents=True, exist_ok=True)

    np.save(destination / "ct.npy", ct_output)
    np.save(destination / "ptv.npy", ptv_output)
    np.save(destination / "dose_gy_per_fraction.npy", dose_output)

    report = {
        "patient": name,
        "slices": int(len(ct_output)),
        "first_slice": first,
        "last_slice": last,
        "fractions": fractions,
        "dose_max_gy_per_fraction": float(dose_output.max()),
        "dose_source": "Clinical DICOM PLAN dose",
        "status": "PROVISIONAL",
        "geometry_note": "CT metadata spacing not fully DICOM verified",
        "sampling_note": "PTV slices plus 5-slice margin",
        "normalization_note": "CT clipped using provisional rule",
        "paper_faithful": False
    }

    with open(destination / "report.json", "w") as f:
        json.dump(report, f, indent=2)

    print("Saved slices:", len(ct_output))
    print("Output shape:", ct_output.shape)
    print("Fractions:", fractions)
    print(
        "Maximum dose per fraction:",
        round(float(dose_output.max()), 3),
        "Gy"
    )

    return report


reports = []

for patient_id in [1, 2]:
    try:
        report = preprocess_patient(patient_id)
        reports.append(report)
        print("STATUS: PROVISIONAL SUCCESS")
    except Exception as e:
        print(f"STATUS: FAILED — {type(e).__name__}: {e}")

print("\n" + "=" * 65)
print("SUMMARY")
print("Patients processed:", len(reports), "/ 2")
print("Output folder:", output_root)
print("Existing checkpoints unchanged.")
print("No model training performed.")

In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download
import shutil
import time

print("STEP 166 — BATCH DOWNLOAD PATIENTS 3–10")
print("=" * 65)

project = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction"
)

repo_id = "PortPy-Project/PortPy_Dataset"

required = [
    "CT_Data.h5",
    "CT_MetaData.json",
    "StructureSet_Data.h5",
    "StructureSet_MetaData.json",
    "DicomFiles/rt_dose_echo_imrt.dcm",
    "DicomFiles/rt_plan_echo_imrt.dcm",
]

cache = Path("/content/portpy_batch_download")

completed = []
failed = []

for patient_id in range(3, 11):

    name = f"Patient_{patient_id}"
    destination = project / name / "raw_portpy"

    print(f"\n{'-' * 50}")
    print("DOWNLOADING", name)

    patient_ok = True

    for filename in required:

        target = destination / filename
        target.parent.mkdir(parents=True, exist_ok=True)

        if target.exists() and target.stat().st_size > 0:
            print("SKIP:", filename)
            continue

        remote_path = (
            f"data/Prostate_Patient_{patient_id}/{filename}"
        )

        try:
            downloaded = hf_hub_download(
                repo_id=repo_id,
                repo_type="dataset",
                filename=remote_path,
                local_dir=str(cache)
            )

            shutil.copy2(downloaded, target)

            print(
                "OK:",
                filename,
                f"({target.stat().st_size / 1e6:.2f} MB)"
            )

        except Exception as e:
            patient_ok = False
            print("FAILED:", filename, str(e)[:180])

    if patient_ok:
        completed.append(name)
        print("PATIENT STATUS: COMPLETE")
    else:
        failed.append(name)
        print("PATIENT STATUS: INCOMPLETE")

print("\n" + "=" * 65)
print("BATCH DOWNLOAD SUMMARY")
print("Completed:", len(completed))
print("Patients:", completed)
print("Incomplete:", failed)
print("No existing checkpoints modified.")
print("No model training performed.")


In [ ]:
print("STEP 167 — BATCH PREPROCESS PATIENTS 3–10")
print("=" * 65)

if "preprocess_patient" not in globals():
    raise RuntimeError(
        "Step 165 function is missing. "
        "Rerun Step 165 to define preprocess_patient()."
    )

successful = []
failed = []

for patient_id in range(3, 11):
    try:
        report = preprocess_patient(patient_id)
        successful.append(report)
        print(f"Patient_{patient_id}: PROVISIONAL SUCCESS")
    except Exception as e:
        failed.append((patient_id, str(e)))
        print(f"Patient_{patient_id}: FAILED — {e}")

print("\n" + "=" * 65)
print("BATCH SUMMARY")
print("Successful:", len(successful), "/ 8")
print("Failed:", len(failed))

print("\nSUCCESSFUL PATIENTS")
for report in successful:
    print(
        report["patient"],
        "| Slices:", report["slices"],
        "| Fractions:", report["fractions"]
    )

if failed:
    print("\nFAILED PATIENTS")
    for patient_id, error in failed:
        print(f"Patient_{patient_id}: {error}")

print("\nTotal new provisional slices:",
      sum(r["slices"] for r in successful))

print("\nNo model training performed.")
print("Existing checkpoints unchanged.")

In [ ]:
import numpy as np
import json
from pathlib import Path

print("STEP 168 — MULTI-PATIENT DATASET AUDIT")
print("=" * 65)

root = Path(
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction/"
    "Stage1_MultiPatient_Provisional"
)

results = []
failed = []

for patient_id in range(1, 11):
    name = f"Patient_{patient_id}"
    folder = root / name

    try:
        ct = np.load(folder / "ct.npy", mmap_mode="r")
        ptv = np.load(folder / "ptv.npy", mmap_mode="r")
        dose = np.load(
            folder / "dose_gy_per_fraction.npy",
            mmap_mode="r"
        )

        with open(folder / "report.json") as f:
            report = json.load(f)

        checks = {
            "matching_shapes": ct.shape == ptv.shape == dose.shape,
            "correct_dimensions": ct.ndim == 3 and ct.shape[1:] == (128, 128),
            "finite_ct": bool(np.isfinite(ct).all()),
            "finite_ptv": bool(np.isfinite(ptv).all()),
            "finite_dose": bool(np.isfinite(dose).all()),
            "nonempty_ptv": bool(np.any(ptv > 0)),
            "nonnegative_dose": bool(np.all(dose >= 0)),
            "positive_dose": bool(np.any(dose > 0)),
            "valid_fractions": int(report["fractions"]) > 0,
        }

        passed = all(checks.values())

        print(
            f"{name}: {'PASS' if passed else 'FAIL'} "
            f"| slices={ct.shape[0]} "
            f"| max dose={float(np.max(dose)):.3f} Gy/fraction"
        )

        if passed:
            results.append((name, int(ct.shape[0])))
        else:
            failed.append((name, checks))

    except Exception as e:
        print(f"{name}: ERROR — {e}")
        failed.append((name, str(e)))

print("\n" + "=" * 65)
print("AUDIT SUMMARY")
print("Patients passing basic array checks:", len(results), "/ 10")
print("Total slices passing:", sum(n for _, n in results))
print("Failed patients:", len(failed))

for name, issue in failed:
    print("\n", name, issue)

print("\nIMPORTANT:")
print("This audit checks basic array integrity only.")
print("It does not validate CT-dose registration,")
print("paper-specific preprocessing, or clinical dose quality.")
print("No files modified. No training performed.")

In [ ]:
# STEP 168 — OPTIMIZED MULTI-PATIENT DATASET AUDIT

import numpy as np
import json
import gc
import time
from pathlib import Path

print("STEP 168 — FAST MULTI-PATIENT DATASET AUDIT")
print("=" * 65, flush=True)

root = Path(
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction/"
    "Stage1_MultiPatient_Provisional"
)

passed = []
failed = []
total_slices = 0

start_time = time.time()

for patient_id in range(1, 11):

    name = f"Patient_{patient_id}"
    folder = root / name

    print(f"\nChecking {name}...", flush=True)

    try:
        # Load each array into memory once
        ct = np.load(folder / "ct.npy")
        ptv = np.load(folder / "ptv.npy")
        dose = np.load(folder / "dose_gy_per_fraction.npy")

        with open(folder / "report.json") as f:
            report = json.load(f)

        # Basic integrity checks
        checks = {
            "shapes_match": ct.shape == ptv.shape == dose.shape,
            "image_size": (
                ct.ndim == 3
                and ct.shape[1:] == (128, 128)
            ),
            "finite_values": (
                np.isfinite(ct).all()
                and np.isfinite(ptv).all()
                and np.isfinite(dose).all()
            ),
            "nonempty_ptv": np.any(ptv > 0),
            "positive_dose": np.any(dose > 0),
            "nonnegative_dose": np.all(dose >= 0),
            "valid_fractions": report["fractions"] > 0
        }

        bad_checks = [
            key for key, value in checks.items()
            if not value
        ]

        if not bad_checks:
            slices = ct.shape[0]
            total_slices += slices
            passed.append(name)

            print(
                f"PASS | {slices} slices | "
                f"Maximum dose: {dose.max():.3f} Gy/fraction",
                flush=True
            )
        else:
            failed.append((name, bad_checks))
            print("FAIL:", bad_checks, flush=True)

        # Release memory before the next patient
        del ct, ptv, dose
        gc.collect()

    except Exception as e:
        failed.append((name, str(e)))
        print("ERROR:", str(e), flush=True)

print("\n" + "=" * 65)
print("FINAL AUDIT SUMMARY")
print("Patients passed:", len(passed), "/ 10")
print("Total slices:", total_slices)
print("Failed:", len(failed))
print("Time:", round(time.time() - start_time, 2), "seconds")

if failed:
    print("\nFAILURE DETAILS")
    for item in failed:
        print(item)

print("\nNo files modified.")
print("No training performed.")
print("Clinical registration and paper fidelity remain unverified.")

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# STEP 169 — REPRODUCIBLE PATIENT-LEVEL SPLIT

import json
import numpy as np
from pathlib import Path

print("STEP 169 — PATIENT-LEVEL DATASET SPLIT")
print("=" * 65)

project = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction"
)

data_root = project / "Stage1_MultiPatient_Provisional"

patients = sorted(
    [
        p.name for p in data_root.iterdir()
        if p.is_dir()
        and (p / "ct.npy").exists()
        and (p / "ptv.npy").exists()
        and (p / "dose_gy_per_fraction.npy").exists()
        and (p / "report.json").exists()
    ],
    key=lambda x: int(x.split("_")[1])
)

if len(patients) != 10:
    raise ValueError(
        f"Expected 10 patients, found {len(patients)}: {patients}"
    )

# Fixed seed for reproducibility
rng = np.random.default_rng(42)
shuffled = list(rng.permutation(patients))

splits = {
    "train": shuffled[:7],
    "validation": shuffled[7:8],
    "test": shuffled[8:10]
}

# Check that no patient appears in multiple groups
all_assigned = sum(splits.values(), [])

assert len(all_assigned) == len(set(all_assigned))
assert set(all_assigned) == set(patients)

slice_counts = {}

for group, names in splits.items():
    slice_counts[group] = sum(
        int(np.load(
            data_root / name / "ct.npy",
            mmap_mode="r"
        ).shape[0])
        for name in names
    )

output = {
    "seed": 42,
    "dataset_status": "PROVISIONAL",
    "split_method": "Patient-level random split",
    "train": splits["train"],
    "validation": splits["validation"],
    "test": splits["test"],
    "slice_counts": slice_counts,
    "paper_faithful_preprocessing": False
}

output_path = data_root / "patient_split_10.json"

if output_path.exists():
    with open(output_path) as f:
        existing = json.load(f)
    if existing != output:
        raise RuntimeError(
            "Existing split differs. Refusing to overwrite it."
        )
    print("Existing identical split retained.")
else:
    with open(output_path, "w") as f:
        json.dump(output, f, indent=2)
    print("New split saved.")

for group, names in splits.items():
    print(f"\n{group.upper()} — {len(names)} patients")
    print("Patients:", names)
    print("Slices:", slice_counts[group])

print("\nTOTAL PATIENTS:", len(patients))
print("TOTAL SLICES:", sum(slice_counts.values()))
print("PATIENT LEAKAGE: NONE")
print("Saved:", output_path)
print("\nNo model training performed.")

In [ ]:
# STEP 170 — PREPROCESSING FIDELITY AUDIT

import json
from pathlib import Path

print("STEP 170 — PREPROCESSING FIDELITY AUDIT")
print("=" * 65)

root = Path(
    "/content/drive/MyDrive/"
    "FluenceFormer_PortPy_Reproduction/"
    "Stage1_MultiPatient_Provisional"
)

with open(root / "patient_split_10.json") as f:
    split = json.load(f)

print("\nDATASET")
print("Patients:", sum(len(split[k]) for k in
                       ["train", "validation", "test"]))
print("Slices:", sum(split["slice_counts"].values()))

print("\nCURRENT PREPROCESSING CHOICES")
print("Stage 1 input channel 1: CT")
print("Stage 1 input channel 2: PTV mask")
print("Target: Clinical DICOM PLAN dose / planned fractions")
print("Resolution: 128 x 128")
print("Slice selection: PTV extent plus 5 slices each side")
print("CT normalization: clip((HU + 1000) / 4000, 0, 1)")
print("Dose interpolation: Linear interpolation in physical coordinates")

print("\nPAPER-FIDELITY STATUS")
checks = {
    "CT input": "Implemented; normalization needs verification",
    "Contour input": "PTV-only; paper equivalence unverified",
    "Dose target": "PortPy clinical PLAN dose; adaptation",
    "Dose normalization": "Per-fraction Gy; paper equivalence unverified",
    "Slice sampling": "PTV plus margin; paper equivalence unverified",
    "Image size": "128 x 128; matches reported training size",
    "CT-dose registration": "Not fully validated",
    "Patient-level split": "Implemented; provisional 10-patient split",
    "Stage 1 model architecture": "Not yet verified for this dataset",
    "Stage 1 training": "Not started"
}

for item, status in checks.items():
    print(f"{item}: {status}")

print("\nPATIENT REPORT STATUS")
for i in range(1, 11):
    with open(root / f"Patient_{i}/report.json") as f:
        report = json.load(f)

    print(
        f"Patient_{i}: "
        f"{report['slices']} slices, "
        f"{report['fractions']} fractions, "
        f"{report['status']}"
    )

print("\nAUDIT COMPLETE")
print("No files modified.")
print("No model training performed.")

In [ ]:
# STEP 171 — INSPECT ORIGINAL FLUENCEFORMER IMPLEMENTATION

import subprocess
from pathlib import Path

print("STEP 171 — ORIGINAL FLUENCEFORMER CODE INSPECTION")
print("=" * 65)

repo = Path("/content/FluenceFormer_Original_Reference")

url = "https://github.com/UJUNWAMGBOH/FluenceFormer.git"
commit = "804ec7799cca82d5341d2c8045943b27b62ce4a1"

if not (repo / ".git").exists():
    subprocess.run(
        ["git", "clone", url, str(repo)],
        check=True
    )

subprocess.run(
    ["git", "-C", str(repo), "checkout", commit],
    check=True
)

actual_commit = subprocess.check_output(
    ["git", "-C", str(repo), "rev-parse", "HEAD"],
    text=True
).strip()

assert actual_commit == commit, "Unexpected repository commit"

print("\nREFERENCE COMMIT:", actual_commit)

keywords = [
    "contour",
    "mask",
    "dataset",
    "preprocess",
    "normalize",
    "dose",
    "slice"
]

candidates = []

for path in repo.rglob("*.py"):
    if ".git" in path.parts:
        continue

    try:
        lines = path.read_text(
            encoding="utf-8",
            errors="replace"
        ).splitlines()

        matches = [
            (i + 1, line.strip())
            for i, line in enumerate(lines)
            if any(word in line.lower() for word in keywords)
        ]

        if matches:
            candidates.append((path, matches))

    except Exception as e:
        print("Could not read:", path, e)

print("\nRELEVANT PYTHON FILES:", len(candidates))

for path, matches in candidates:
    print("\nFILE:", path.relative_to(repo))

    for line_number, line in matches[:12]:
        print(f"  L{line_number}: {line[:160]}")

    if len(matches) > 12:
        print(f"  ... {len(matches) - 12} additional matches")

print("\nINSPECTION COMPLETE")
print("No PortPy files modified.")
print("No model training performed.")

In [ ]:
# STEP 172 — READ ORIGINAL FLUENCEFORMER SOURCE

from pathlib import Path

print("STEP 172 — ORIGINAL SOURCE CODE REVIEW")
print("=" * 65)

repo = Path("/content/FluenceFormer_Original_Reference")

if not repo.exists():
    raise FileNotFoundError(
        "Original repository missing. Rerun Step 171."
    )

source = repo / "src/prostate_fluence"

files = [
    "config.py",
    "dataset.py",
    "utils.py",
]

for filename in files:

    path = source / filename

    print("\n" + "=" * 65)
    print("FILE:", filename)
    print("=" * 65)

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    lines = path.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()

    for number, line in enumerate(lines, start=1):
        print(f"{number:03d}: {line}")

print("\n" + "=" * 65)
print("MODEL FILE CANDIDATES")

for path in sorted(source.rglob("*.py")):
    if any(
        keyword in path.name.lower()
        for keyword in ["model", "network", "train"]
    ):
        print(path.relative_to(repo))

print("\nREVIEW COMPLETE")
print("No project files modified.")
print("No model training performed.")

In [ ]:
# STEP 173 — LOCATE ORIGINAL STAGE 1 IMPLEMENTATION

from pathlib import Path

print("STEP 173 — ORIGINAL STAGE 1 ARCHITECTURE")
print("=" * 65)

repo = Path("/content/FluenceFormer_Original_Reference")
source = repo / "src/prostate_fluence"

if not source.exists():
    raise FileNotFoundError(
        "Original repository not found. Rerun Step 171."
    )

keywords = [
    "stage1",
    "stage_1",
    "stage 1",
    "swinunetr",
    "nnformer",
    "dosepred",
    "dose_predict",
    "mse_loss",
    "in_channels",
    "out_channels",
    "optimizer",
    "epochs"
]

matches_found = 0

for path in sorted(repo.rglob("*.py")):

    if ".git" in path.parts:
        continue

    lines = path.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()

    matches = [
        (i, line)
        for i, line in enumerate(lines)
        if any(k in line.lower() for k in keywords)
    ]

    if not matches:
        continue

    print("\nFILE:", path.relative_to(repo))

    printed = set()

    for i, _ in matches[:15]:

        start = max(0, i - 3)
        end = min(len(lines), i + 5)

        for j in range(start, end):
            if j not in printed:
                print(f"{j+1:03d}: {lines[j]}")
                printed.add(j)

        print("---")
        matches_found += 1

print("\n" + "=" * 65)
print("Matching code sections:", matches_found)
print("No files modified.")
print("No training performed.")

In [ ]:
# STEP 174 — SEARCH REFERENCE DOCUMENTATION

from pathlib import Path

print("STEP 174 — ORIGINAL REPOSITORY DOCUMENTATION")
print("=" * 65)

repo = Path("/content/FluenceFormer_Original_Reference")

keywords = [
    "stage 1", "stage-1", "stage1",
    "dose prediction", "dose estimation",
    "contour", "preprocess", "normalization",
    "fluence", "dataset", "training"
]

documents = [
    p for p in repo.rglob("*")
    if p.is_file()
    and ".git" not in p.parts
    and p.suffix.lower() in {".md", ".txt", ".rst", ".yaml", ".yml"}
]

print("Documentation files:", len(documents))

for path in sorted(documents):

    try:
        lines = path.read_text(
            encoding="utf-8",
            errors="replace"
        ).splitlines()
    except Exception:
        continue

    matches = [
        (i + 1, line.strip())
        for i, line in enumerate(lines)
        if any(k in line.lower() for k in keywords)
    ]

    if not matches:
        continue

    print("\nFILE:", path.relative_to(repo))
    print("Matching lines:", len(matches))

    for number, line in matches[:25]:
        print(f"L{number}: {line[:180]}")

    if len(matches) > 25:
        print(f"... {len(matches) - 25} additional matches")

print("\n" + "=" * 65)
print("DOCUMENTATION SEARCH COMPLETE")
print("No files modified.")
print("No training performed.")

In [ ]:
# STEP 175 — EXTRACT ORIGINAL PAPER METHODOLOGY

import requests
import fitz
from pathlib import Path

print("STEP 175 — ORIGINAL PAPER METHODOLOGY")
print("=" * 65)

paper_path = Path("/content/FluenceFormer_Original_Paper.pdf")
paper_url = "https://arxiv.org/pdf/2512.22425"

if not paper_path.exists():
    response = requests.get(paper_url, timeout=90)
    response.raise_for_status()
    if not response.content.startswith(b"%PDF"):
        raise ValueError("Downloaded file is not a valid PDF.")
    paper_path.write_bytes(response.content)

doc = fitz.open(paper_path)

terms = [
    "stage 1",
    "stage-1",
    "stage i",
    "dose prediction",
    "dose prior",
    "contour",
    "preprocessing",
    "normalization",
    "swin unetr",
    "training configuration"
]

total = 0

for page_num, page in enumerate(doc, start=1):
    text = page.get_text()
    paragraphs = [
        p.strip().replace("\n", " ")
        for p in text.split("\n\n")
        if p.strip()
    ]

    selected = [
        p for p in paragraphs
        if any(term in p.lower() for term in terms)
    ]

    if selected:
        print(f"\n{'=' * 65}")
        print(f"PDF PAGE {page_num}")
        print("=" * 65)

        for paragraph in selected[:8]:
            print(paragraph[:1200])
            print("---")
            total += 1

print("\nTOTAL MATCHING EXCERPTS:", total)
print("TOTAL PDF PAGES:", len(doc))
print("\nNo project files modified.")
print("No model training performed.")

In [ ]:
%pip install -q PyMuPDF

import fitz

print("PyMuPDF installed successfully!")
print("Version:", fitz.VersionBind)

In [ ]:
# STEP 175 — EXTRACT ORIGINAL PAPER METHODOLOGY

import requests
import fitz
from pathlib import Path

print("STEP 175 — ORIGINAL PAPER METHODOLOGY")
print("=" * 65)

paper_path = Path("/content/FluenceFormer_Original_Paper.pdf")
paper_url = "https://arxiv.org/pdf/2512.22425"

if not paper_path.exists():
    response = requests.get(paper_url, timeout=90)
    response.raise_for_status()
    if not response.content.startswith(b"%PDF"):
        raise ValueError("Downloaded file is not a valid PDF.")
    paper_path.write_bytes(response.content)

doc = fitz.open(paper_path)

terms = [
    "stage 1",
    "stage-1",
    "stage i",
    "dose prediction",
    "dose prior",
    "contour",
    "preprocessing",
    "normalization",
    "swin unetr",
    "training configuration"
]

total = 0

for page_num, page in enumerate(doc, start=1):
    text = page.get_text()
    paragraphs = [
        p.strip().replace("\n", " ")
        for p in text.split("\n\n")
        if p.strip()
    ]

    selected = [
        p for p in paragraphs
        if any(term in p.lower() for term in terms)
    ]

    if selected:
        print(f"\n{'=' * 65}")
        print(f"PDF PAGE {page_num}")
        print("=" * 65)

        for paragraph in selected[:8]:
            print(paragraph[:1200])
            print("---")
            total += 1

print("\nTOTAL MATCHING EXCERPTS:", total)
print("TOTAL PDF PAGES:", len(doc))
print("\nNo project files modified.")
print("No model training performed.")

In [ ]:
# STEP 176 — READ COMPLETE METHODOLOGY

import fitz
from pathlib import Path

print("STEP 176 — COMPLETE FLUENCEFORMER METHODOLOGY")
print("=" * 65)

paper_path = Path("/content/FluenceFormer_Original_Paper.pdf")

if not paper_path.exists():
    raise FileNotFoundError(
        "Paper not found. Rerun Step 175."
    )

with fitz.open(paper_path) as doc:

    # PDF pages 3–6 (zero-based indices 2–5)
    for page_index in range(2, 6):

        page = doc[page_index]

        print("\n" + "=" * 65)
        print(f"PDF PAGE {page_index + 1}")
        print("=" * 65)

        print(page.get_text(sort=True))

print("\nMETHODOLOGY EXTRACTION COMPLETE")
print("No project files modified.")
print("No model training performed.")

In [ ]:
# STEP 177 — PORTPY ANATOMICAL STRUCTURE INVENTORY

import json
from pathlib import Path
from collections import Counter

print("STEP 177 — ANATOMICAL STRUCTURE INVENTORY")
print("=" * 65)

project = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction"
)

counts = Counter()
patient_structures = {}

for patient_id in range(1, 11):

    patient = f"Patient_{patient_id}"

    path = (
        project
        / patient
        / "raw_portpy"
        / "StructureSet_MetaData.json"
    )

    print(f"\n{patient}")

    if not path.exists():
        print("Metadata file not found:", path)
        continue

    with open(path) as f:
        metadata = json.load(f)

    # Show top-level structure to avoid assuming
    # the PortPy metadata schema.
    print("Metadata type:", type(metadata).__name__)

    if isinstance(metadata, dict):
        print("Top-level keys:", list(metadata.keys())[:20])

        # Collect recognizable structure-name fields
        names = []

        for key, value in metadata.items():

            if isinstance(value, dict):
                names.extend(str(k) for k in value.keys())

            elif isinstance(value, list):
                for item in value:
                    if isinstance(item, dict):
                        for name_key in [
                            "name",
                            "structure_name",
                            "StructureName",
                            "roi_name"
                        ]:
                            if name_key in item:
                                names.append(str(item[name_key]))

        names = sorted(set(names))

    elif isinstance(metadata, list):

        names = []

        for item in metadata:
            if isinstance(item, dict):
                names.extend(
                    str(item[k])
                    for k in [
                        "name",
                        "structure_name",
                        "StructureName",
                        "roi_name"
                    ]
                    if k in item
                )

        names = sorted(set(names))

    else:
        names = []

    patient_structures[patient] = names

    print("Candidate structure names:", names[:40])

    counts.update(names)

print("\n" + "=" * 65)
print("STRUCTURE FREQUENCY ACROSS PATIENTS")

for name, count in counts.most_common(40):
    print(f"{name}: {count}/10")

print("\nINVENTORY COMPLETE")
print("No files modified.")
print("No training performed.")

In [ ]:
# STEP 178 — VERIFY PORTPY STRUCTURE MASK DATASETS

import h5py
import json
import numpy as np
from pathlib import Path

print("STEP 178 — VERIFY ANATOMICAL MASK DATASETS")
print("=" * 70)

project = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction"
)

structures = [
    "PTV", "CTV", "BLADDER", "RECTUM",
    "FEMUR_L", "FEMUR_R", "BODY", "SKIN"
]

summary = {}

for patient_id in range(1, 11):

    patient = f"Patient_{patient_id}"
    path = project / patient / "raw_portpy" / "StructureSet_Data.h5"

    print(f"\n{patient}")

    if not path.exists():
        print("ERROR: StructureSet_Data.h5 missing")
        continue

    with h5py.File(path, "r") as h5:

        available = []

        def inspect(name, obj):
            if isinstance(obj, h5py.Dataset):
                available.append(
                    (name, tuple(obj.shape), str(obj.dtype))
                )

        h5.visititems(inspect)

        found = []

        for structure in structures:

            matches = [
                item for item in available
                if item[0].split("/")[-1].upper() == structure
            ]

            if matches:
                name, shape, dtype = matches[0]
                found.append(structure)

                print(
                    f"  {structure:<10} "
                    f"shape={shape} dtype={dtype}"
                )
            else:
                print(f"  {structure:<10} NOT FOUND")

        summary[patient] = found

print("\n" + "=" * 70)
print("STRUCTURE DATASET SUMMARY")

for structure in structures:
    count = sum(
        structure in names
        for names in summary.values()
    )
    print(f"{structure:<10}: {count}/10 patients")

print("\nINSPECTION COMPLETE")
print("No files modified.")
print("No model training performed.")

In [ ]:
# STEP 179 — VALIDATE ANATOMICAL MASK VALUES

import h5py
import numpy as np
from pathlib import Path

print("STEP 179 — ANATOMICAL MASK VALIDATION")
print("=" * 70)

project = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction"
)

structures = [
    "PTV", "CTV", "BLADDER", "RECTUM",
    "FEMUR_L", "FEMUR_R", "BODY", "SKIN"
]

results = {}

for patient_id in range(1, 11):

    patient = f"Patient_{patient_id}"
    path = project / patient / "raw_portpy" / "StructureSet_Data.h5"

    with h5py.File(path, "r") as h5:

        depth = h5["PTV"].shape[0]

        # Sample three axial slices
        indices = sorted(set([
            depth // 4,
            depth // 2,
            (3 * depth) // 4
        ]))

        print(f"\n{patient} — sampled slices {indices}")

        patient_result = {}

        for structure in structures:

            if structure not in h5:
                print(f"  {structure:<10} MISSING")
                continue

            dataset = h5[structure]

            values = set()
            positive_voxels = 0

            for z in indices:

                arr = dataset[z]

                values.update(np.unique(arr).tolist())
                positive_voxels += int(np.count_nonzero(arr))

            binary = values.issubset({0, 1})

            patient_result[structure] = {
                "sampled_values": sorted(values),
                "sampled_positive_voxels": positive_voxels,
                "sampled_binary": binary
            }

            print(
                f"  {structure:<10} "
                f"binary={binary} "
                f"values={sorted(values)[:10]} "
                f"positive={positive_voxels:,}"
            )

        results[patient] = patient_result

print("\n" + "=" * 70)
print("VALIDATION SUMMARY")

nonbinary = []

for patient, structures_found in results.items():
    for name, info in structures_found.items():
        if not info["sampled_binary"]:
            nonbinary.append((patient, name))

print("Nonbinary masks in sampled slices:", len(nonbinary))

for patient, name in nonbinary:
    print("  ", patient, name)

print("\nIMPORTANT: This is a sampled-slice check, not a full-volume audit.")
print("No files modified.")
print("No training performed.")

In [ ]:
# STEP 180 — ANATOMICAL COVERAGE ON TREATMENT SLICES

import h5py
import numpy as np
from pathlib import Path

print("STEP 180 — ANATOMICAL COVERAGE AUDIT")
print("=" * 70)

project = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction"
)

structures = [
    "PTV", "CTV", "BLADDER", "RECTUM",
    "FEMUR_L", "FEMUR_R"
]

summary = {}

for patient_id in range(1, 11):

    patient = f"Patient_{patient_id}"

    h5_path = (
        project / patient / "raw_portpy"
        / "StructureSet_Data.h5"
    )

    print(f"\n{patient}")

    with h5py.File(h5_path, "r") as h5:

        ptv = h5["PTV"]

        # Read one axial slice at a time to avoid
        # loading large 3D arrays into memory.
        ptv_indices = []

        for z in range(ptv.shape[0]):
            if np.any(ptv[z]):
                ptv_indices.append(z)

        if not ptv_indices:
            print("WARNING: No PTV-positive slices found")
            continue

        first = min(ptv_indices)
        last = max(ptv_indices)

        selected = list(range(
            max(0, first - 5),
            min(ptv.shape[0], last + 6)
        ))

        print("PTV-positive slice range:", first, "to", last)
        print("Selected treatment slices:", len(selected))

        patient_result = {}

        for structure in structures:

            if structure not in h5:
                print(f"  {structure:<10} MISSING")
                patient_result[structure] = None
                continue

            dataset = h5[structure]

            positive_slices = 0

            for z in selected:
                if np.any(dataset[z]):
                    positive_slices += 1

            patient_result[structure] = positive_slices

            print(
                f"  {structure:<10} "
                f"present on {positive_slices}/{len(selected)} "
                f"selected slices"
            )

        summary[patient] = patient_result

print("\n" + "=" * 70)
print("AUDIT COMPLETE")
print("No files modified.")
print("No training performed.")

In [ ]:
# STEP 181 — CHECK MASK PRESERVATION AFTER RESIZING

import h5py
import numpy as np
from scipy.ndimage import zoom
from pathlib import Path

print("STEP 181 — MASK RESIZING VALIDATION")
print("=" * 70)

project = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction"
)

structures = [
    "PTV", "CTV", "BLADDER",
    "RECTUM", "FEMUR_L", "FEMUR_R"
]

target_size = 128
total_lost = 0

for patient_id in range(1, 11):

    patient = f"Patient_{patient_id}"

    path = (
        project / patient / "raw_portpy"
        / "StructureSet_Data.h5"
    )

    print(f"\n{patient}")

    with h5py.File(path, "r") as h5:

        ptv = h5["PTV"]

        positive_indices = [
            z for z in range(ptv.shape[0])
            if np.any(ptv[z])
        ]

        if not positive_indices:
            print("ERROR: No PTV-positive slices")
            continue

        first = positive_indices[0]
        last = positive_indices[-1]

        selected = range(
            max(0, first - 5),
            min(ptv.shape[0], last + 6)
        )

        for structure in structures:

            if structure not in h5:
                print(f"  {structure:<10} MISSING")
                continue

            dataset = h5[structure]

            original_positive = 0
            resized_positive = 0
            lost_slices = 0

            for z in selected:

                mask = dataset[z]

                if not np.any(mask):
                    continue

                original_positive += 1

                scale = (
                    target_size / mask.shape[0],
                    target_size / mask.shape[1]
                )

                resized = zoom(
                    mask,
                    scale,
                    order=0
                )

                if np.any(resized):
                    resized_positive += 1
                else:
                    lost_slices += 1

            total_lost += lost_slices

            print(
                f"  {structure:<10} "
                f"original={original_positive:>2} "
                f"resized={resized_positive:>2} "
                f"lost={lost_slices}"
            )

print("\n" + "=" * 70)
print("TOTAL STRUCTURE-POSITIVE SLICES LOST:", total_lost)

if total_lost == 0:
    print("PASS: All sampled structure-positive slices retained.")
else:
    print("REVIEW: Some structures disappeared after resizing.")

print("\nNo files modified.")
print("No model training performed.")

In [ ]:
# STEP 182 — COMPARE MASK RESIZING METHODS

import h5py
import numpy as np
from scipy.ndimage import zoom
from skimage.measure import block_reduce
from pathlib import Path

print("STEP 182 — MASK RESIZING COMPARISON")
print("=" * 70)

project = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction"
)

total_nn_lost = 0
total_max_lost = 0
total_checked = 0

for patient_id in range(1, 11):

    patient = f"Patient_{patient_id}"

    path = (
        project / patient / "raw_portpy"
        / "StructureSet_Data.h5"
    )

    with h5py.File(path, "r") as h5:

        ptv = h5["PTV"]

        positive_slices = [
            z for z in range(ptv.shape[0])
            if np.any(ptv[z])
        ]

        if not positive_slices:
            print(patient, "ERROR: No PTV slices")
            continue

        first = positive_slices[0]
        last = positive_slices[-1]

        selected = range(
            max(0, first - 5),
            min(ptv.shape[0], last + 6)
        )

        nn_lost = 0
        max_lost = 0

        for z in selected:

            mask = ptv[z].astype(bool)

            if not np.any(mask):
                continue

            total_checked += 1

            height, width = mask.shape

            # Method 1: Nearest-neighbor resizing
            nn = zoom(
                mask.astype(np.uint8),
                (128 / height, 128 / width),
                order=0
            )

            # Method 2: Maximum-preserving block reduction
            if height % 128 != 0 or width % 128 != 0:
                raise ValueError(
                    f"Unexpected mask dimensions: {mask.shape}"
                )

            factor_h = height // 128
            factor_w = width // 128

            max_resized = block_reduce(
                mask.astype(np.uint8),
                block_size=(factor_h, factor_w),
                func=np.max
            )

            assert max_resized.shape == (128, 128)

            if not np.any(nn):
                nn_lost += 1

            if not np.any(max_resized):
                max_lost += 1

        total_nn_lost += nn_lost
        total_max_lost += max_lost

        print(
            f"{patient:<12} "
            f"nearest-neighbor lost={nn_lost} "
            f"maximum-preserving lost={max_lost}"
        )

print("\n" + "=" * 70)
print("TOTAL PTV-POSITIVE SLICES CHECKED:", total_checked)
print("NEAREST-NEIGHBOR LOST:", total_nn_lost)
print("MAXIMUM-PRESERVING LOST:", total_max_lost)

if total_max_lost == 0:
    print("PASS: Maximum-preserving resizing retained every PTV-positive slice.")

print("\nNo project files modified.")
print("No model training performed.")

In [ ]:
# STEP 183 — BUILD MULTI-PATIENT ANATOMICAL MASK ARRAYS

import h5py
import json
import numpy as np
from pathlib import Path

print("STEP 183 — MULTI-PATIENT ANATOMICAL INPUTS")
print("=" * 70)

project = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction"
)

output_root = project / "Stage1_MultiPatient_Anatomy_Provisional"
output_root.mkdir(parents=True, exist_ok=True)

structures = [
    "PTV", "CTV", "BLADDER",
    "RECTUM", "FEMUR_L", "FEMUR_R"
]

completed = 0
total_slices = 0

for patient_id in range(1, 11):

    patient = f"Patient_{patient_id}"

    source = (
        project / patient / "raw_portpy"
        / "StructureSet_Data.h5"
    )

    destination = output_root / patient
    destination.mkdir(parents=True, exist_ok=True)

    with h5py.File(source, "r") as h5:

        ptv = h5["PTV"]

        positive = [
            z for z in range(ptv.shape[0])
            if np.any(ptv[z])
        ]

        if not positive:
            raise ValueError(f"{patient}: no PTV-positive slices")

        selected = list(range(
            max(0, positive[0] - 5),
            min(ptv.shape[0], positive[-1] + 6)
        ))

        height, width = ptv.shape[1:]

        if height % 128 or width % 128:
            raise ValueError(
                f"{patient}: unsupported dimensions {(height, width)}"
            )

        fh, fw = height // 128, width // 128

        # Shape: [slices, anatomical channels, height, width]
        masks = np.zeros(
            (len(selected), len(structures), 128, 128),
            dtype=np.uint8
        )

        availability = {}

        for channel, structure in enumerate(structures):

            exists = structure in h5
            availability[structure] = exists

            if not exists:
                continue

            dataset = h5[structure]

            for i, z in enumerate(selected):

                original = dataset[z]

                # Exact max-pooling for integer scale factors
                reduced = original.reshape(
                    128, fh, 128, fw
                ).max(axis=(1, 3))

                masks[i, channel] = (reduced > 0).astype(
                    np.uint8
                )

        # Preserve existing output if it already exists
        output_file = destination / "anatomy_masks.npy"

        if output_file.exists():
            raise FileExistsError(
                f"Refusing to overwrite {output_file}"
            )

        np.save(output_file, masks)

        report = {
            "patient": patient,
            "structures": structures,
            "structure_available": availability,
            "selected_slice_indices": selected,
            "shape": list(masks.shape),
            "resizing": "maximum_preserving_block_reduction",
            "paper_faithful_contour_encoding": False,
            "status": "provisional_portpy_adaptation"
        }

        with open(destination / "report.json", "w") as f:
            json.dump(report, f, indent=2)

        print(
            f"{patient}: {masks.shape} "
            f"| missing: "
            f"{[s for s in structures if not availability[s]]}"
        )

        completed += 1
        total_slices += len(selected)

print("\n" + "=" * 70)
print("PATIENTS COMPLETED:", completed)
print("TOTAL SLICES:", total_slices)
print("ANATOMICAL CHANNELS:", len(structures))
print("OUTPUT:", output_root)

print("\nExisting CT and dose arrays were not modified.")
print("No model training performed.")

In [ ]:
# STEP 184 — VALIDATE MULTI-PATIENT STAGE 1 ARRAYS

import json
import numpy as np
from pathlib import Path

print("STEP 184 — MULTI-PATIENT STAGE 1 VALIDATION")
print("=" * 70)

project = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction"
)

old_root = project / "Stage1_MultiPatient_Provisional"
new_root = project / "Stage1_MultiPatient_Anatomy_Provisional"

passed = 0
failed = 0
total_slices = 0

for patient_id in range(1, 11):

    patient = f"Patient_{patient_id}"

    old_dir = old_root / patient
    new_dir = new_root / patient

    print(f"\n{patient}")

    try:
        ct = np.load(old_dir / "ct.npy")
        dose = np.load(old_dir / "dose_gy_per_fraction.npy")
        old_ptv = np.load(old_dir / "ptv.npy")
        anatomy = np.load(new_dir / "anatomy_masks.npy")

        with open(new_dir / "report.json") as f:
            anatomy_report = json.load(f)

        with open(old_dir / "report.json") as f:
            old_report = json.load(f)

        n = anatomy.shape[0]

        checks = {
            "CT shape": ct.shape == (n, 128, 128),
            "Dose shape": dose.shape == (n, 128, 128),
            "Old PTV shape": old_ptv.shape == (n, 128, 128),
            "Anatomy shape": anatomy.shape == (n, 6, 128, 128),
            "CT finite": bool(np.isfinite(ct).all()),
            "Dose finite": bool(np.isfinite(dose).all()),
            "Dose nonnegative": bool((dose >= 0).all()),
            "Anatomy binary": bool(
                np.isin(anatomy, [0, 1]).all()
            ),
            "Slice indices valid": (
                len(anatomy_report["selected_slice_indices"]) == n
                and anatomy_report["selected_slice_indices"]
                == sorted(anatomy_report["selected_slice_indices"])
            ),
        }

        # Compare old PTV with the newly constructed PTV channel.
        # Different resizing methods can produce different boundaries.
        old_binary = old_ptv > 0
        new_binary = anatomy[:, 0] > 0

        intersection = np.logical_and(
            old_binary, new_binary
        ).sum()

        union = np.logical_or(
            old_binary, new_binary
        ).sum()

        iou = (
            float(intersection / union)
            if union > 0 else 1.0
        )

        checks["PTV overlap"] = iou > 0.5

        for name, ok in checks.items():
            print(f"  {name:<23}: {'PASS' if ok else 'FAIL'}")

        print(f"  PTV intersection-over-union: {iou:.4f}")

        # Show report keys so slice-index provenance can be
        # checked without assuming an undocumented schema.
        print(
            "  Old report keys:",
            list(old_report.keys())
        )

        if all(checks.values()):
            passed += 1
            total_slices += n
        else:
            failed += 1

    except Exception as e:
        failed += 1
        print("  ERROR:", type(e).__name__, str(e))

print("\n" + "=" * 70)
print("PATIENTS PASSED:", passed)
print("PATIENTS FAILED:", failed)
print("VALIDATED SLICES:", total_slices)

if passed == 10:
    print("PASS: All 10 patients passed basic array checks.")
else:
    print("REVIEW REQUIRED: Some checks failed.")

print("\nIMPORTANT:")
print("This validates array consistency, not clinical CT-dose registration.")
print("The slice-index correspondence still requires provenance verification.")
print("No project files modified.")
print("No model training performed.")

In [ ]:
# STEP 185 — VERIFY STAGE 1 SLICE CORRESPONDENCE

import json
import numpy as np
from pathlib import Path

print("STEP 185 — SLICE CORRESPONDENCE VERIFICATION")
print("=" * 70)

project = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction"
)

old_root = project / "Stage1_MultiPatient_Provisional"
new_root = project / "Stage1_MultiPatient_Anatomy_Provisional"

passed = 0
failed = 0

for patient_id in range(1, 11):

    patient = f"Patient_{patient_id}"

    old_dir = old_root / patient
    new_dir = new_root / patient

    with open(old_dir / "report.json") as f:
        old_report = json.load(f)

    with open(new_dir / "report.json") as f:
        new_report = json.load(f)

    indices = new_report["selected_slice_indices"]

    old_first = int(old_report["first_slice"])
    old_last = int(old_report["last_slice"])
    old_count = int(old_report["slices"])

    expected_indices = list(
        range(old_first, old_last + 1)
    )

    checks = {
        "Same slice count": len(indices) == old_count,
        "Same first slice": indices[0] == old_first,
        "Same last slice": indices[-1] == old_last,
        "Same slice ordering": indices == expected_indices,
        "No duplicate indices": len(indices) == len(set(indices)),
    }

    ok = all(checks.values())

    print(f"\n{patient}: {'PASS' if ok else 'FAIL'}")
    print(
        f"  Original CT/dose: "
        f"slices {old_first}–{old_last}, count={old_count}"
    )
    print(
        f"  New anatomy:     "
        f"slices {indices[0]}–{indices[-1]}, count={len(indices)}"
    )

    for name, result in checks.items():
        if not result:
            print(f"  FAILED CHECK: {name}")

    if ok:
        passed += 1
    else:
        failed += 1

print("\n" + "=" * 70)
print("PATIENTS PASSED:", passed)
print("PATIENTS FAILED:", failed)

if failed == 0:
    print("PASS: Slice indices agree for all 10 patients.")
else:
    print("STOP: Slice correspondence requires correction.")

print("\nNOTE:")
print("This verifies array slice provenance, not physical CT-dose registration.")
print("No files modified.")
print("No training performed.")

In [ ]:
# STEP 186 — CT–DOSE GEOMETRY AUDIT (ALL 10 PATIENTS)

import json
import numpy as np
import pydicom
import h5py
from pathlib import Path

print("STEP 186 — CT–DOSE GEOMETRY AUDIT")
print("=" * 72)

project = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction"
)

passed = 0
review = 0

for patient_id in range(1, 11):
    patient = f"Patient_{patient_id}"
    raw = project / patient / "raw_portpy"

    print(f"\n{patient}")

    try:
        with open(raw / "CT_MetaData.json") as f:
            meta = json.load(f)

        with h5py.File(raw / "CT_Data.h5", "r") as h5:
            ct_shape = h5["ct_hu_3d"].shape

        ds = pydicom.dcmread(
            raw / "DicomFiles" / "rt_dose_echo_imrt.dcm",
            stop_before_pixels=True
        )

        origin = np.asarray(meta["origin_xyz_mm"], dtype=float)
        spacing = np.asarray(meta["resolution_xyz_mm"], dtype=float)
        direction = np.asarray(meta["direction"], dtype=float).reshape(3, 3)

        dose_origin = np.asarray(
            ds.ImagePositionPatient, dtype=float
        )
        orientation = np.asarray(
            ds.ImageOrientationPatient, dtype=float
        )
        dose_row = orientation[:3]
        dose_col = orientation[3:]

        dose_spacing = np.asarray(
            ds.PixelSpacing, dtype=float
        )

        offsets = np.asarray(
            ds.GridFrameOffsetVector, dtype=float
        )

        normal = np.cross(dose_row, dose_col)

        # DICOM array index order: [frame, row, column].
        # Column index follows the first orientation vector;
        # row index follows the second orientation vector.
        dose_corners = []

        for frame in [0, len(offsets) - 1]:
            for row in [0, int(ds.Rows) - 1]:
                for col in [0, int(ds.Columns) - 1]:
                    point = (
                        dose_origin
                        + col * dose_spacing[1] * dose_row
                        + row * dose_spacing[0] * dose_col
                        + offsets[frame] * normal
                    )
                    dose_corners.append(point)

        dose_corners = np.asarray(dose_corners)

        # PortPy CT coordinates are reported in XYZ order,
        # while CT array dimensions are ZYX.
        ct_extent_xyz = (
            np.asarray(ct_shape[::-1], dtype=float) - 1
        ) * spacing

        ct_corners = []

        for x in [0, ct_extent_xyz[0]]:
            for y in [0, ct_extent_xyz[1]]:
                for z in [0, ct_extent_xyz[2]]:
                    ct_corners.append(
                        origin + direction @ np.array([x, y, z])
                    )

        ct_corners = np.asarray(ct_corners)

        ct_min = ct_corners.min(axis=0)
        ct_max = ct_corners.max(axis=0)
        dose_min = dose_corners.min(axis=0)
        dose_max = dose_corners.max(axis=0)

        overlap = np.minimum(ct_max, dose_max) - np.maximum(
            ct_min, dose_min
        )

        orientation_ok = (
            np.allclose(direction.T @ direction, np.eye(3), atol=1e-3)
            and np.allclose(np.linalg.norm(dose_row), 1, atol=1e-3)
            and np.allclose(np.linalg.norm(dose_col), 1, atol=1e-3)
            and np.allclose(np.dot(dose_row, dose_col), 0, atol=1e-3)
        )

        coverage_ok = bool(np.all(overlap > 0))

        print("  CT shape:", ct_shape)
        print("  CT spacing XYZ (mm):", spacing)
        print("  Dose grid:", (
            len(offsets), int(ds.Rows), int(ds.Columns)
        ))
        print("  CT bounding box (mm):")
        print("    Min:", np.round(ct_min, 2))
        print("    Max:", np.round(ct_max, 2))
        print("  Dose bounding box (mm):")
        print("    Min:", np.round(dose_min, 2))
        print("    Max:", np.round(dose_max, 2))
        print("  Bounding-box overlap (mm):", np.round(overlap, 2))
        print("  Orientation valid:", orientation_ok)
        print("  Bounding boxes overlap:", coverage_ok)

        if orientation_ok and coverage_ok:
            passed += 1
            print("  RESULT: BASIC GEOMETRY CHECK PASSED")
        else:
            review += 1
            print("  RESULT: REVIEW REQUIRED")

    except Exception as e:
        review += 1
        print("  ERROR:", type(e).__name__, str(e))

print("\n" + "=" * 72)
print("BASIC GEOMETRY PASSED:", passed)
print("REQUIRES REVIEW:", review)

print("\nIMPORTANT:")
print("Bounding-box overlap does not prove voxel-level registration.")
print("CT metadata precision and dose-grid sampling still require review.")
print("No files modified.")
print("No model training performed.")

In [ ]:
# STEP 186A — INSTALL PYDICOM

%pip install -q pydicom

import pydicom
import h5py
import numpy as np

print("PyDICOM installed successfully!")
print("PyDICOM version:", pydicom.__version__)
print("H5PY version:", h5py.__version__)
print("NumPy version:", np.__version__)

In [ ]:
# STEP 186 — CT–DOSE GEOMETRY AUDIT (ALL 10 PATIENTS)

import json
import numpy as np
import pydicom
import h5py
from pathlib import Path

print("STEP 186 — CT–DOSE GEOMETRY AUDIT")
print("=" * 72)

project = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction"
)

passed = 0
review = 0

for patient_id in range(1, 11):
    patient = f"Patient_{patient_id}"
    raw = project / patient / "raw_portpy"

    print(f"\n{patient}")

    try:
        with open(raw / "CT_MetaData.json") as f:
            meta = json.load(f)

        with h5py.File(raw / "CT_Data.h5", "r") as h5:
            ct_shape = h5["ct_hu_3d"].shape

        ds = pydicom.dcmread(
            raw / "DicomFiles" / "rt_dose_echo_imrt.dcm",
            stop_before_pixels=True
        )

        origin = np.asarray(meta["origin_xyz_mm"], dtype=float)
        spacing = np.asarray(meta["resolution_xyz_mm"], dtype=float)
        direction = np.asarray(meta["direction"], dtype=float).reshape(3, 3)

        dose_origin = np.asarray(
            ds.ImagePositionPatient, dtype=float
        )
        orientation = np.asarray(
            ds.ImageOrientationPatient, dtype=float
        )
        dose_row = orientation[:3]
        dose_col = orientation[3:]

        dose_spacing = np.asarray(
            ds.PixelSpacing, dtype=float
        )

        offsets = np.asarray(
            ds.GridFrameOffsetVector, dtype=float
        )

        normal = np.cross(dose_row, dose_col)

        # DICOM array index order: [frame, row, column].
        # Column index follows the first orientation vector;
        # row index follows the second orientation vector.
        dose_corners = []

        for frame in [0, len(offsets) - 1]:
            for row in [0, int(ds.Rows) - 1]:
                for col in [0, int(ds.Columns) - 1]:
                    point = (
                        dose_origin
                        + col * dose_spacing[1] * dose_row
                        + row * dose_spacing[0] * dose_col
                        + offsets[frame] * normal
                    )
                    dose_corners.append(point)

        dose_corners = np.asarray(dose_corners)

        # PortPy CT coordinates are reported in XYZ order,
        # while CT array dimensions are ZYX.
        ct_extent_xyz = (
            np.asarray(ct_shape[::-1], dtype=float) - 1
        ) * spacing

        ct_corners = []

        for x in [0, ct_extent_xyz[0]]:
            for y in [0, ct_extent_xyz[1]]:
                for z in [0, ct_extent_xyz[2]]:
                    ct_corners.append(
                        origin + direction @ np.array([x, y, z])
                    )

        ct_corners = np.asarray(ct_corners)

        ct_min = ct_corners.min(axis=0)
        ct_max = ct_corners.max(axis=0)
        dose_min = dose_corners.min(axis=0)
        dose_max = dose_corners.max(axis=0)

        overlap = np.minimum(ct_max, dose_max) - np.maximum(
            ct_min, dose_min
        )

        orientation_ok = (
            np.allclose(direction.T @ direction, np.eye(3), atol=1e-3)
            and np.allclose(np.linalg.norm(dose_row), 1, atol=1e-3)
            and np.allclose(np.linalg.norm(dose_col), 1, atol=1e-3)
            and np.allclose(np.dot(dose_row, dose_col), 0, atol=1e-3)
        )

        coverage_ok = bool(np.all(overlap > 0))

        print("  CT shape:", ct_shape)
        print("  CT spacing XYZ (mm):", spacing)
        print("  Dose grid:", (
            len(offsets), int(ds.Rows), int(ds.Columns)
        ))
        print("  CT bounding box (mm):")
        print("    Min:", np.round(ct_min, 2))
        print("    Max:", np.round(ct_max, 2))
        print("  Dose bounding box (mm):")
        print("    Min:", np.round(dose_min, 2))
        print("    Max:", np.round(dose_max, 2))
        print("  Bounding-box overlap (mm):", np.round(overlap, 2))
        print("  Orientation valid:", orientation_ok)
        print("  Bounding boxes overlap:", coverage_ok)

        if orientation_ok and coverage_ok:
            passed += 1
            print("  RESULT: BASIC GEOMETRY CHECK PASSED")
        else:
            review += 1
            print("  RESULT: REVIEW REQUIRED")

    except Exception as e:
        review += 1
        print("  ERROR:", type(e).__name__, str(e))

print("\n" + "=" * 72)
print("BASIC GEOMETRY PASSED:", passed)
print("REQUIRES REVIEW:", review)

print("\nIMPORTANT:")
print("Bounding-box overlap does not prove voxel-level registration.")
print("CT metadata precision and dose-grid sampling still require review.")
print("No files modified.")
print("No model training performed.")

In [ ]:
# STEP 187 — VERIFY DOSE GRID COVERAGE OF SELECTED CT SLICES

import json
import h5py
import numpy as np
import pydicom
from pathlib import Path

print("STEP 187 — SELECTED CT SLICE / DOSE GRID COVERAGE")
print("=" * 75)

project = Path("/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction")

for patient_id in range(1, 11):

    patient = f"Patient_{patient_id}"
    raw = project / patient / "raw_portpy"

    with open(raw / "CT_MetaData.json") as f:
        meta = json.load(f)

    with open(
        project / "Stage1_MultiPatient_Anatomy_Provisional"
        / patient / "report.json"
    ) as f:
        report = json.load(f)

    ds = pydicom.dcmread(
        raw / "DicomFiles" / "rt_dose_echo_imrt.dcm",
        stop_before_pixels=True
    )

    with h5py.File(raw / "CT_Data.h5", "r") as h5:
        shape = h5["ct_hu_3d"].shape

    origin = np.asarray(meta["origin_xyz_mm"], dtype=float)
    spacing = np.asarray(meta["resolution_xyz_mm"], dtype=float)
    direction = np.asarray(meta["direction"], dtype=float).reshape(3, 3)

    dose_origin = np.asarray(ds.ImagePositionPatient, dtype=float)
    orientation = np.asarray(ds.ImageOrientationPatient, dtype=float)

    dose_x = orientation[:3]
    dose_y = orientation[3:]
    dose_z = np.cross(dose_x, dose_y)

    pixel_spacing = np.asarray(ds.PixelSpacing, dtype=float)
    offsets = np.asarray(ds.GridFrameOffsetVector, dtype=float)

    selected = report["selected_slice_indices"]

    # Evaluate the original CT pixel locations corresponding
    # to the centers of the 128x128 resized image.
    row_positions = (
        (np.arange(128) + 0.5) * shape[1] / 128 - 0.5
    )
    col_positions = (
        (np.arange(128) + 0.5) * shape[2] / 128 - 0.5
    )

    rows, cols = np.meshgrid(
        row_positions, col_positions, indexing="ij"
    )

    total_points = 0
    covered_points = 0
    slice_coverage = []

    for z in selected:

        # CT array order is [z, row, col].
        # PortPy XYZ metadata maps [col, row, z].
        coords = np.stack([
            cols.ravel() * spacing[0],
            rows.ravel() * spacing[1],
            np.full(rows.size, z * spacing[2])
        ], axis=0)

        physical = origin[:, None] + direction @ coords
        relative = physical - dose_origin[:, None]

        dose_col_index = (
            dose_x @ relative
        ) / pixel_spacing[1]

        dose_row_index = (
            dose_y @ relative
        ) / pixel_spacing[0]

        dose_offset = dose_z @ relative

        # Frame offsets may be increasing or decreasing.
        frame_min = min(offsets[0], offsets[-1])
        frame_max = max(offsets[0], offsets[-1])

        inside = (
            (dose_col_index >= 0) &
            (dose_col_index <= ds.Columns - 1) &
            (dose_row_index >= 0) &
            (dose_row_index <= ds.Rows - 1) &
            (dose_offset >= frame_min) &
            (dose_offset <= frame_max)
        )

        fraction = float(inside.mean())

        slice_coverage.append(fraction)
        covered_points += int(inside.sum())
        total_points += inside.size

    overall = covered_points / total_points

    print(
        f"{patient:<12} "
        f"covered={overall * 100:6.2f}% "
        f"min_slice={min(slice_coverage) * 100:6.2f}% "
        f"max_slice={max(slice_coverage) * 100:6.2f}%"
    )

print("=" * 75)
print("NOTE: Full CT coverage is not expected because RTDOSE")
print("grids can cover only part of the CT field of view.")
print("This checks geometric coverage, not dose-value accuracy.")
print("No files modified.")

In [ ]:
# STEP 188 — VERIFY PHYSICAL DOSE-GRID COVERAGE OF PTV

import json
import h5py
import numpy as np
import pydicom
from pathlib import Path

print("STEP 188 — PTV DOSE-GRID COVERAGE")
print("=" * 72)

project = Path("/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction")

results = []

for patient_id in range(1, 11):

    patient = f"Patient_{patient_id}"
    raw = project / patient / "raw_portpy"

    with open(raw / "CT_MetaData.json") as f:
        meta = json.load(f)

    with open(
        project / "Stage1_MultiPatient_Anatomy_Provisional"
        / patient / "report.json"
    ) as f:
        report = json.load(f)

    ds = pydicom.dcmread(
        raw / "DicomFiles" / "rt_dose_echo_imrt.dcm",
        stop_before_pixels=True
    )

    origin = np.asarray(meta["origin_xyz_mm"], dtype=float)
    spacing = np.asarray(meta["resolution_xyz_mm"], dtype=float)
    direction = np.asarray(meta["direction"], dtype=float).reshape(3, 3)

    dose_origin = np.asarray(ds.ImagePositionPatient, dtype=float)
    orientation = np.asarray(ds.ImageOrientationPatient, dtype=float)

    dose_x = orientation[:3]
    dose_y = orientation[3:]
    dose_z = np.cross(dose_x, dose_y)

    pixel_spacing = np.asarray(ds.PixelSpacing, dtype=float)
    offsets = np.asarray(ds.GridFrameOffsetVector, dtype=float)

    # This check assumes conventional relative GridFrameOffsetVector
    # coordinates. Flag other DICOM conventions rather than silently
    # treating them as relative offsets.
    if not np.isclose(offsets[0], 0.0, atol=1e-3):
        print(f"{patient}: REVIEW — nonzero first dose-frame offset")
        results.append((patient, None))
        continue

    selected = report["selected_slice_indices"]

    inside_count = 0
    ptv_count = 0

    with h5py.File(
        raw / "StructureSet_Data.h5", "r"
    ) as h5:

        ptv = h5["PTV"]

        for z in selected:

            mask = ptv[z] > 0

            rows, cols = np.where(mask)

            if len(rows) == 0:
                continue

            coordinates = np.stack([
                cols * spacing[0],
                rows * spacing[1],
                np.full(len(rows), z * spacing[2])
            ], axis=0)

            physical = origin[:, None] + direction @ coordinates
            relative = physical - dose_origin[:, None]

            dose_col = (dose_x @ relative) / pixel_spacing[1]
            dose_row = (dose_y @ relative) / pixel_spacing[0]
            dose_offset = dose_z @ relative

            inside = (
                (dose_col >= 0) &
                (dose_col <= ds.Columns - 1) &
                (dose_row >= 0) &
                (dose_row <= ds.Rows - 1) &
                (dose_offset >= min(offsets)) &
                (dose_offset <= max(offsets))
            )

            inside_count += int(inside.sum())
            ptv_count += len(rows)

    coverage = (
        100 * inside_count / ptv_count
        if ptv_count else 0.0
    )

    results.append((patient, coverage))

    print(
        f"{patient:<12} "
        f"PTV coverage={coverage:7.3f}% "
        f"({inside_count:,}/{ptv_count:,} voxels)"
    )

print("\n" + "=" * 72)

valid = [v for _, v in results if v is not None]

if len(valid) == 10 and all(v >= 99.9 for v in valid):
    print("PASS: All PTV regions are within their dose grids.")
elif valid:
    print("REVIEW: Some PTV regions may extend outside dose coverage.")
else:
    print("REVIEW: PTV coverage could not be established.")

print("\nNo files modified.")
print("No model training performed.")

In [ ]:
# STEP 189A — FIX SPLIT NAME AND COMPLETE DATA LOADERS

from torch.utils.data import DataLoader

# Use the exact keys in the saved split file
train_dataset = Stage1PortPyDataset(split["train"])
val_dataset = Stage1PortPyDataset(split["validation"])
test_dataset = Stage1PortPyDataset(split["test"])

train_loader = DataLoader(
    train_dataset, batch_size=16, shuffle=True, num_workers=0
)

val_loader = DataLoader(
    val_dataset, batch_size=16, shuffle=False, num_workers=0
)

test_loader = DataLoader(
    test_dataset, batch_size=16, shuffle=False, num_workers=0
)

print("STEP 189A — DATA LOADER VERIFICATION")
print("=" * 60)

print("Training slices:", len(train_dataset))
print("Validation slices:", len(val_dataset))
print("Testing slices:", len(test_dataset))

assert len(train_dataset) == 284
assert len(val_dataset) == 37
assert len(test_dataset) == 76

x_batch, y_batch = next(iter(train_loader))

print("\nTraining input shape:", tuple(x_batch.shape))
print("Training target shape:", tuple(y_batch.shape))
print("Input range:", float(x_batch.min()), "to", float(x_batch.max()))
print("Dose range:", float(y_batch.min()), "to", float(y_batch.max()))

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nPASS: All three Stage 1 data loaders are ready.")
print("No model training performed.")

In [ ]:
# STEP 190 — EXTRACT ORIGINAL STAGE 1 ARCHITECTURE DETAILS

import fitz
from pathlib import Path

print("STEP 190 — ORIGINAL FLUENCEFORMER STAGE 1")
print("=" * 75)

paper_path = Path("/content/FluenceFormer_Original_Paper.pdf")

if not paper_path.exists():
    raise FileNotFoundError(
        "Original paper not found. Please tell me if Colab restarted."
    )

doc = fitz.open(str(paper_path))

keywords = [
    "Stage 1",
    "Stage-1",
    "SwinUNETR",
    "nnFormer",
    "dose prediction",
    "encoder",
    "decoder",
]

for page_index in range(min(len(doc), 9)):

    text = doc[page_index].get_text()

    matches = [
        word for word in keywords
        if word.lower() in text.lower()
    ]

    if not matches:
        continue

    print(f"\n{'=' * 75}")
    print(f"PDF PAGE {page_index + 1}")
    print("MATCHED TERMS:", ", ".join(matches))
    print("=" * 75)

    # Print paragraphs containing architecture terms,
    # with surrounding context.
    paragraphs = text.split("\n\n")

    for paragraph in paragraphs:

        if any(
            word.lower() in paragraph.lower()
            for word in keywords
        ):
            print(paragraph[:3500])
            print("-" * 40)

doc.close()

print("\nSTEP 190 COMPLETE")
print("No model training performed.")

In [ ]:
# STEP 191 — BUILD STAGE 1 SWIN UNETR DOSE REGRESSION MODEL

import sys
import subprocess
import torch
import torch.nn as nn

try:
    import monai
    from monai.networks.nets import SwinUNETR
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "monai"
    ])
    import monai
    from monai.networks.nets import SwinUNETR

print("STEP 191 — STAGE 1 SWIN UNETR")
print("=" * 70)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Original paper: CT + anatomical masks -> dose
# PortPy adaptation: 1 CT + 6 contour channels
input_channels = 7
output_channels = 1

# 2D Swin UNETR for axial-slice regression
stage1_model = SwinUNETR(
    in_channels=input_channels,
    out_channels=output_channels,
    feature_size=48,
    spatial_dims=2,
    use_checkpoint=True
).to(device)

# The original paper specifies a ReLU dose regression head.
# MONAI's SwinUNETR output block produces one channel;
# apply ReLU to enforce nonnegative predictions.
class Stage1DoseRegressor(nn.Module):

    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(self.backbone(x))

stage1_model = Stage1DoseRegressor(stage1_model).to(device)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    stage1_model.parameters(),
    lr=1e-4
)

parameters = sum(
    p.numel() for p in stage1_model.parameters()
)

print("MONAI version:", monai.__version__)
print("PyTorch version:", torch.__version__)
print("Device:", device)
print("Input channels:", input_channels)
print("Output channels:", output_channels)
print("Feature size:", 48)
print("Trainable parameters:", f"{parameters:,}")

# Test a small batch without training
stage1_model.eval()

try:
    x_test = torch.randn(
        1, 7, 128, 128,
        device=device
    )

    with torch.no_grad():
        y_test = stage1_model(x_test)

    print("\nFORWARD PASS")
    print("Input shape:", tuple(x_test.shape))
    print("Output shape:", tuple(y_test.shape))
    print("Minimum output:", float(y_test.min()))

    assert y_test.shape == (1, 1, 128, 128)
    assert float(y_test.min()) >= 0

    print("\nPASS: Stage 1 Swin UNETR forward pass successful.")

finally:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("No model training performed.")

In [ ]:
# STEP 192A — SWIN UNETR GPU MEMORY TEST
# No optimizer steps. No model weights changed.

import torch
import gc
import time

print("STEP 192A — GPU MEMORY AND BATCH SIZE TEST")
print("=" * 75)

assert torch.cuda.is_available(), "CUDA GPU not available"

device = torch.device("cuda")
stage1_model = stage1_model.to(device)

print("GPU:", torch.cuda.get_device_name(0))

total_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"Total GPU memory: {total_memory:.2f} GB")

criterion = torch.nn.MSELoss()

results = []

for batch_size in [1, 2, 4, 8, 16]:

    print(f"\nTesting batch size {batch_size}...", flush=True)

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    x = None
    y = None
    prediction = None
    loss = None

    try:
        stage1_model.train()
        stage1_model.zero_grad(set_to_none=True)

        # Use real training samples, repeating only for this memory test
        indices = torch.arange(batch_size) % len(train_dataset)

        x = train_dataset.inputs[indices].to(device)
        y = train_dataset.targets[indices].to(device)

        torch.cuda.synchronize()
        start = time.time()

        with torch.autocast("cuda", dtype=torch.float16):

            prediction = stage1_model(x)
            loss = criterion(prediction.float(), y.float())

        loss.backward()

        torch.cuda.synchronize()

        elapsed = time.time() - start

        peak_memory = (
            torch.cuda.max_memory_allocated() / 1024**3
        )

        print("PASS")
        print(f"Time: {elapsed:.2f} seconds")
        print(f"Peak allocated memory: {peak_memory:.2f} GB")

        results.append({
            "batch_size": batch_size,
            "status": "PASS",
            "peak_memory_gb": round(peak_memory, 2),
            "seconds": round(elapsed, 2)
        })

    except torch.cuda.OutOfMemoryError:

        print("GPU OUT OF MEMORY")

        results.append({
            "batch_size": batch_size,
            "status": "OUT OF MEMORY"
        })

        # Larger batches are unlikely to fit if this one failed.
        break

    finally:
        stage1_model.zero_grad(set_to_none=True)

        del x, y, prediction, loss

        gc.collect()
        torch.cuda.empty_cache()

print("\n" + "=" * 75)
print("GPU TEST SUMMARY")
print("=" * 75)

for result in results:
    print(result)

successful = [
    r["batch_size"]
    for r in results
    if r["status"] == "PASS"
]

if successful:
    print("\nLargest tested successful batch:", max(successful))
else:
    print("\nNo batch size passed.")

print("\nNo optimizer steps performed.")
print("No model weights updated.")
print("No training checkpoints modified.")

In [ ]:
# STEP 192B — COMPLETE TRAINING STEP TEST
# Batch size 16, Adam, MSE, forward + backward + optimizer step

import torch
import copy
import time
import gc

print("STEP 192B — FULL TRAINING STEP TEST")
print("=" * 70)

device = torch.device("cuda")
stage1_model = stage1_model.to(device)

# Preserve the current model and optimizer states
original_model_state = copy.deepcopy(stage1_model.state_dict())
original_optimizer_state = copy.deepcopy(optimizer.state_dict())

try:
    stage1_model.train()
    optimizer.zero_grad(set_to_none=True)

    x = train_dataset.inputs[:16].to(device)
    y = train_dataset.targets[:16].to(device)

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

    start = time.time()

    # Select a parameter that receives gradients
    parameter = next(
        p for p in stage1_model.parameters()
        if p.requires_grad
    )

    before = parameter.detach().clone()

    with torch.autocast("cuda", dtype=torch.float16):
        prediction = stage1_model(x)

    loss = torch.nn.functional.mse_loss(
        prediction.float(), y.float()
    )

    assert torch.isfinite(loss), "Nonfinite training loss"

    loss.backward()

    gradient_present = any(
        p.grad is not None
        and torch.isfinite(p.grad).all().item()
        and torch.count_nonzero(p.grad).item() > 0
        for p in stage1_model.parameters()
    )

    optimizer.step()

    torch.cuda.synchronize()

    elapsed = time.time() - start

    peak_allocated = (
        torch.cuda.max_memory_allocated() / 1024**3
    )

    peak_reserved = (
        torch.cuda.max_memory_reserved() / 1024**3
    )

    parameter_changed = not torch.equal(
        before, parameter.detach()
    )

    print("Batch size:", len(x))
    print("Loss:", float(loss))
    print("Nonzero finite gradients:", gradient_present)
    print("Selected parameter changed:", parameter_changed)
    print(f"Step time: {elapsed:.2f} seconds")
    print(f"Peak allocated: {peak_allocated:.2f} GB")
    print(f"Peak reserved: {peak_reserved:.2f} GB")

    assert gradient_present, "No usable gradients detected"
    assert parameter_changed, "Selected parameter did not update"

    print("\nPASS: Complete batch-16 optimizer step succeeded.")

finally:
    # Restore model and optimizer so this test changes no experiment state
    stage1_model.load_state_dict(original_model_state)
    optimizer.load_state_dict(original_optimizer_state)
    optimizer.zero_grad(set_to_none=True)

    del original_model_state, original_optimizer_state
    gc.collect()
    torch.cuda.empty_cache()

print("Original model and optimizer states restored.")

In [ ]:
# STEP 193 — THREE-EPOCH STAGE 1 TRAINING PILOT

import torch
import numpy as np
import time
import json
import copy
from pathlib import Path
from torch.utils.data import DataLoader

print("STEP 193 — STAGE 1 TRAINING PILOT")
print("=" * 70)

device = torch.device("cuda")
torch.manual_seed(42)

pilot_dir = (
    project / "Stage1_SwinUNETR_Pilot"
)
pilot_dir.mkdir(parents=True, exist_ok=True)

# Keep pilot training separate from the original model state
pilot_model = copy.deepcopy(stage1_model).to(device)

pilot_optimizer = torch.optim.Adam(
    pilot_model.parameters(),
    lr=1e-4
)

pilot_train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0
)

pilot_val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

history = []

for epoch in range(3):

    start = time.time()

    pilot_model.train()

    train_loss_sum = 0.0

    for x, y in pilot_train_loader:

        x = x.to(device)
        y = y.to(device)

        pilot_optimizer.zero_grad(set_to_none=True)

        with torch.autocast("cuda", dtype=torch.float16):
            prediction = pilot_model(x)

        loss = torch.nn.functional.mse_loss(
            prediction.float(),
            y.float()
        )

        if not torch.isfinite(loss):
            raise RuntimeError("Nonfinite training loss")

        loss.backward()
        pilot_optimizer.step()

        train_loss_sum += loss.detach().item() * len(x)

    train_mse = train_loss_sum / len(train_dataset)

    pilot_model.eval()

    val_loss_sum = 0.0

    with torch.no_grad():

        for x, y in pilot_val_loader:

            x = x.to(device)
            y = y.to(device)

            with torch.autocast("cuda", dtype=torch.float16):
                prediction = pilot_model(x)

            loss = torch.nn.functional.mse_loss(
                prediction.float(),
                y.float()
            )

            val_loss_sum += loss.item() * len(x)

    val_mse = val_loss_sum / len(val_dataset)

    record = {
        "epoch": epoch + 1,
        "train_mse": train_mse,
        "val_mse": val_mse,
        "seconds": round(time.time() - start, 2)
    }

    history.append(record)

    print(
        f"Epoch {epoch+1}/3 | "
        f"Train MSE: {train_mse:.6f} | "
        f"Val MSE: {val_mse:.6f} | "
        f"Time: {record['seconds']:.1f}s",
        flush=True
    )

with open(pilot_dir / "pilot_history.json", "w") as f:
    json.dump(history, f, indent=2)

torch.save(
    pilot_model.state_dict(),
    pilot_dir / "pilot_model.pt"
)

print("\n" + "=" * 70)
print("PILOT COMPLETE")
print("History saved:", pilot_dir / "pilot_history.json")
print("Pilot model saved:", pilot_dir / "pilot_model.pt")
print("Full-training model remains unchanged.")
print("Test patients remain untouched.")

In [ ]:
# STEP 194 — FULL 50-EPOCH STAGE 1 TRAINING
# Swin UNETR | Adam | MSE | Batch size 16

import torch
import numpy as np
import json
import time
import copy
from pathlib import Path
from torch.utils.data import DataLoader

print("STEP 194 — FULL STAGE 1 TRAINING")
print("=" * 70)

device = torch.device("cuda")
epochs = 50
batch_size = 16

output_dir = project / "Stage1_SwinUNETR_Full_Training"
output_dir.mkdir(parents=True, exist_ok=True)

latest_path = output_dir / "latest_checkpoint.pt"
best_path = output_dir / "best_model.pt"
history_path = output_dir / "training_history.json"

# Fresh model copy, separate from the three-epoch pilot
model = copy.deepcopy(stage1_model).to(device)

optimizer_full = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

criterion = torch.nn.MSELoss()

train_loader_full = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

val_loader_full = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

history = []
best_val = float("inf")
start_epoch = 0

# Resume an interrupted full-training run
if latest_path.exists():

    checkpoint = torch.load(
        latest_path,
        map_location=device,
        weights_only=False
    )

    model.load_state_dict(checkpoint["model"])
    optimizer_full.load_state_dict(checkpoint["optimizer"])

    start_epoch = checkpoint["epoch"]
    best_val = checkpoint["best_val"]
    history = checkpoint["history"]

    print(f"Resuming from epoch {start_epoch}")

else:
    torch.manual_seed(42)
    np.random.seed(42)
    print("Starting fresh 50-epoch training.")

print("GPU:", torch.cuda.get_device_name(0))
print("Batch size:", batch_size)
print("Epochs:", epochs)
print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

for epoch in range(start_epoch, epochs):

    start = time.time()

    # TRAIN
    model.train()

    train_sum = 0.0

    for x, y in train_loader_full:

        x = x.to(device)
        y = y.to(device)

        optimizer_full.zero_grad(set_to_none=True)

        with torch.autocast("cuda", dtype=torch.float16):
            prediction = model(x)

        loss = criterion(prediction.float(), y.float())

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Nonfinite training loss at epoch {epoch+1}"
            )

        loss.backward()
        optimizer_full.step()

        train_sum += loss.detach().item() * len(x)

    train_mse = train_sum / len(train_dataset)

    # VALIDATE
    model.eval()

    val_sum = 0.0

    with torch.no_grad():

        for x, y in val_loader_full:

            x = x.to(device)
            y = y.to(device)

            with torch.autocast("cuda", dtype=torch.float16):
                prediction = model(x)

            loss = criterion(prediction.float(), y.float())

            val_sum += loss.detach().item() * len(x)

    val_mse = val_sum / len(val_dataset)

    # SAVE BEST MODEL
    if val_mse < best_val:

        best_val = val_mse

        torch.save(
            model.state_dict(),
            best_path
        )

    record = {
        "epoch": epoch + 1,
        "train_mse": float(train_mse),
        "val_mse": float(val_mse),
        "seconds": round(time.time() - start, 2)
    }

    history.append(record)

    # SAVE RESUMABLE CHECKPOINT
    torch.save(
        {
            "epoch": epoch + 1,
            "model": model.state_dict(),
            "optimizer": optimizer_full.state_dict(),
            "best_val": best_val,
            "history": history
        },
        latest_path
    )

    with open(history_path, "w") as f:
        json.dump(history, f, indent=2)

    print(
        f"Epoch {epoch+1:02d}/50 | "
        f"Train MSE: {train_mse:.6f} | "
        f"Val MSE: {val_mse:.6f} | "
        f"Best: {best_val:.6f} | "
        f"{record['seconds']:.1f}s",
        flush=True
    )

print("\n" + "=" * 70)
print("FULL STAGE 1 TRAINING COMPLETE")
print("Best validation MSE:", best_val)
print("Best model:", best_path)
print("Training history:", history_path)
print("Test patients remain held out.")

In [ ]:
# STEP 195 — HELD-OUT STAGE 1 TEST EVALUATION
# Patients 2 and 9 were not used for training or validation.

import torch
import numpy as np
import pandas as pd
from pathlib import Path

print("STEP 195 — HELD-OUT TEST EVALUATION")
print("=" * 75)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

best_path = (
    project / "Stage1_SwinUNETR_Full_Training" / "best_model.pt"
)

assert best_path.exists(), "Best checkpoint not found"

# Reuse the exact architecture from Step 191
stage1_model.load_state_dict(
    torch.load(best_path, map_location=device, weights_only=True)
)

stage1_model = stage1_model.to(device)
stage1_model.eval()

test_patients = split["test"]

results = []

for patient in test_patients:

    ct = np.load(
        data_root / patient / "ct.npy"
    ).astype(np.float32)

    dose = np.load(
        data_root / patient / "dose_gy_per_fraction.npy"
    ).astype(np.float32)

    anatomy = np.load(
        anatomy_root / patient / "anatomy_masks.npy"
    ).astype(np.float32)

    x = np.concatenate(
        [ct[:, None, :, :], anatomy],
        axis=1
    )

    predictions = []

    with torch.no_grad():

        for start in range(0, len(x), 16):

            batch = torch.from_numpy(
                x[start:start+16]
            ).to(device)

            with torch.autocast(
                device_type=device.type,
                enabled=(device.type == "cuda")
            ):
                pred = stage1_model(batch)

            predictions.append(
                pred.float().cpu().numpy()[:, 0]
            )

    predicted = np.concatenate(predictions, axis=0)

    # Whole-image evaluation
    error = predicted - dose

    mse = float(np.mean(error**2))
    mae = float(np.mean(np.abs(error)))
    rmse = float(np.sqrt(mse))

    # Zero-dose baseline
    zero_baseline_mse = float(np.mean(dose**2))

    # PTV region from the anatomy tensor
    ptv_mask = anatomy[:, 0] > 0

    if np.any(ptv_mask):

        ptv_error = error[ptv_mask]

        ptv_mse = float(np.mean(ptv_error**2))
        ptv_mae = float(np.mean(np.abs(ptv_error)))

    else:
        ptv_mse = float("nan")
        ptv_mae = float("nan")

    results.append({
        "patient": patient,
        "slices": len(ct),
        "MSE": mse,
        "MAE": mae,
        "RMSE": rmse,
        "PTV_MSE": ptv_mse,
        "PTV_MAE": ptv_mae,
        "Zero_Baseline_MSE": zero_baseline_mse
    })

    print(
        f"{patient}: "
        f"MSE={mse:.6f}, "
        f"MAE={mae:.6f}, "
        f"PTV_MAE={ptv_mae:.6f}"
    )

df = pd.DataFrame(results)

print("\n" + "=" * 75)
print("TEST RESULTS")
print(df.to_string(index=False))

output_path = (
    project / "Stage1_SwinUNETR_Full_Training"
    / "heldout_test_results.csv"
)

df.to_csv(output_path, index=False)

print("\nSaved:", output_path)
print("No model training performed.")
print("Best validation checkpoint used.")

In [ ]:
# STEP 196 — VISUALIZE STAGE 1 TEST PREDICTIONS

import torch
import numpy as np
import matplotlib.pyplot as plt

print("STEP 196 — PREDICTED VS ACTUAL DOSE")
print("=" * 75)

stage1_model.eval()

test_patients = ["Patient_2", "Patient_9"]

fig, axes = plt.subplots(
    2, 3,
    figsize=(15, 9),
    constrained_layout=True
)

for row, patient in enumerate(test_patients):

    ct = np.load(
        data_root / patient / "ct.npy"
    ).astype(np.float32)

    actual_dose = np.load(
        data_root / patient / "dose_gy_per_fraction.npy"
    ).astype(np.float32)

    anatomy = np.load(
        anatomy_root / patient / "anatomy_masks.npy"
    ).astype(np.float32)

    ptv = anatomy[:, 0]

    # Select the axial slice with the largest PTV area
    slice_index = int(
        np.argmax(ptv.sum(axis=(1, 2)))
    )

    x = np.concatenate(
        [
            ct[slice_index:slice_index+1, None],
            anatomy[slice_index:slice_index+1]
        ],
        axis=1
    )

    x_tensor = torch.from_numpy(x).to(device)

    with torch.no_grad():
        with torch.autocast(
            device_type=device.type,
            enabled=(device.type == "cuda")
        ):
            prediction = stage1_model(x_tensor)

    predicted_dose = (
        prediction.float().cpu().numpy()[0, 0]
    )

    true_dose = actual_dose[slice_index]

    absolute_error = np.abs(
        predicted_dose - true_dose
    )

    shared_max = max(
        float(true_dose.max()),
        float(predicted_dose.max())
    )

    images = [
        true_dose,
        predicted_dose,
        absolute_error
    ]

    titles = [
        f"{patient} — Actual Dose",
        f"{patient} — Predicted Dose",
        f"{patient} — Absolute Error"
    ]

    for col in range(3):

        ax = axes[row, col]

        im = ax.imshow(
            images[col],
            cmap="viridis" if col < 2 else "magma",
            vmin=0,
            vmax=shared_max if col < 2 else None
        )

        # PTV boundary for anatomical reference
        ax.contour(
            ptv[slice_index],
            levels=[0.5],
            colors="white",
            linewidths=0.8
        )

        ax.set_title(titles[col])
        ax.axis("off")

        fig.colorbar(
            im,
            ax=ax,
            fraction=0.046,
            pad=0.04,
            label="Gy per fraction"
        )

    print(
        f"{patient}: selected slice {slice_index}, "
        f"actual max={true_dose.max():.3f}, "
        f"predicted max={predicted_dose.max():.3f}, "
        f"slice MAE={absolute_error.mean():.4f}"
    )

output_path = (
    project / "Stage1_SwinUNETR_Full_Training"
    / "stage1_test_dose_comparison.png"
)

plt.savefig(
    output_path,
    dpi=200,
    bbox_inches="tight"
)

plt.show()

print("\nSaved visualization:", output_path)
print("No model training performed.")

In [ ]:
# STEP 197A — PORTPY BEAM AND FLUENCE DATA DISCOVERY
# Read-only inspection. No training or file modification.

from pathlib import Path
from collections import Counter

print("STEP 197A — PORTPY BEAM AND FLUENCE DISCOVERY")
print("=" * 75)

search_roots = [
    Path("/content/PortPy"),
    Path("/content/portpy"),
    Path("/content/drive/MyDrive/PortPy"),
    Path("/content/drive/MyDrive/PortPy_Data"),
    project
]

keywords = [
    "fluence",
    "beam",
    "angle",
    "influence",
    "optimization",
    "plan"
]

extensions = {
    ".npy", ".npz", ".pkl", ".pickle",
    ".json", ".mat", ".h5", ".hdf5",
    ".csv"
}

found = []
existing_roots = []

for root in search_roots:

    if not root.exists():
        continue

    existing_roots.append(root)

    print("\nSearching:", root)

    for path in root.rglob("*"):

        if not path.is_file():
            continue

        name = path.name.lower()

        if (
            path.suffix.lower() in extensions
            and any(word in name for word in keywords)
        ):
            found.append(path)

print("\n" + "=" * 75)
print("EXISTING SEARCH ROOTS")

for root in existing_roots:
    print(root)

print("\nMATCHING FILES:", len(found))

for path in found[:100]:
    print(path)

if len(found) > 100:
    print(f"... {len(found)-100} additional matches")

print("\nFILE EXTENSIONS")
print(Counter(p.suffix.lower() for p in found))

print("\nIMPORTANT:")
print("This searches filenames only.")
print("No matching files does not prove fluence data is absent.")
print("PortPy may store beam and fluence information inside")
print("patient-specific files or optimization objects.")
print("No files were modified.")

In [ ]:
# STEP 197B — INSPECT PATIENT 1 FLUENCE MAPS AND BEAM METADATA
# Read-only inspection

import numpy as np
import json
from pathlib import Path

print("STEP 197B — FLUENCE AND BEAM INSPECTION")
print("=" * 75)

patient_root = project / "Patient_1"

fluence_dir = patient_root / "optimized_portpy"
raw_dir = patient_root / "raw_portpy"

fluence_files = sorted(
    fluence_dir.glob("beam_*_fluence.npy")
)

print("\nNUMBER OF FLUENCE FILES:", len(fluence_files))

for file in fluence_files:

    arr = np.load(file, allow_pickle=False)

    print(
        f"{file.name:25s} "
        f"shape={str(arr.shape):16s} "
        f"min={np.min(arr):.6f} "
        f"max={np.max(arr):.6f} "
        f"nonzero={np.count_nonzero(arr)} "
        f"finite={np.isfinite(arr).all()}"
    )

print("\n" + "=" * 75)
print("PLANNER BEAM INFORMATION")

planner_path = raw_dir / "PlannerBeams.json"

if planner_path.exists():

    with open(planner_path) as f:
        planner = json.load(f)

    print(
        json.dumps(planner, indent=2, default=str)[:6000]
    )

else:
    print("PlannerBeams.json not found")

print("\n" + "=" * 75)
print("VERIFIED BEAM MAPPING")

mapping_path = patient_root / "verified_beam_mapping.json"

if mapping_path.exists():

    with open(mapping_path) as f:
        mapping = json.load(f)

    print(
        json.dumps(mapping, indent=2, default=str)[:6000]
    )

else:
    print("Verified mapping not found")

print("\n" + "=" * 75)
print("EXAMPLE BEAM METADATA")

metadata_files = sorted(
    raw_dir.glob("Beam_*_MetaData.json")
)

if metadata_files:

    example = metadata_files[0]

    print("File:", example.name)

    with open(example) as f:
        metadata = json.load(f)

    print(
        json.dumps(metadata, indent=2, default=str)[:6000]
    )

else:
    print("No beam metadata found")

print("\nInspection complete. No files modified.")

In [ ]:
# STEP 197C — MULTIPATIENT BEAM AND FLUENCE INVENTORY

from pathlib import Path
import json

print("STEP 197C — MULTIPATIENT INVENTORY")
print("=" * 80)

records = []

for n in range(1, 11):

    patient = f"Patient_{n}"
    patient_root = project / patient

    raw_dir = patient_root / "raw_portpy"
    optimized_dir = patient_root / "optimized_portpy"

    planner_file = raw_dir / "PlannerBeams.json"

    beam_metadata = sorted(
        raw_dir.glob("Beam_*_MetaData.json")
    )

    beam_data = sorted(
        raw_dir.glob("Beam_*_Data.h5")
    )

    fluence_files = sorted(
        optimized_dir.glob("beam_*_fluence.npy")
    )

    beam_ids = []
    angles = []

    for metadata_path in beam_metadata:

        with open(metadata_path) as f:
            metadata = json.load(f)

        beam_ids.append(metadata.get("ID"))
        angles.append(metadata.get("gantry_angle"))

    record = {
        "patient": patient,
        "raw_exists": raw_dir.exists(),
        "planner_exists": planner_file.exists(),
        "beam_metadata_count": len(beam_metadata),
        "beam_data_count": len(beam_data),
        "fluence_count": len(fluence_files),
        "beam_ids": beam_ids,
        "angles": angles
    }

    records.append(record)

    print(f"\n{patient}")
    print("  Raw directory:", record["raw_exists"])
    print("  Planner file:", record["planner_exists"])
    print("  Beam metadata:", record["beam_metadata_count"])
    print("  Beam HDF5 files:", record["beam_data_count"])
    print("  Optimized fluence maps:", record["fluence_count"])
    print("  Gantry angles:", angles)

print("\n" + "=" * 80)
print("SUMMARY")

for record in records:
    print(
        f"{record['patient']:12s} | "
        f"Metadata: {record['beam_metadata_count']:2d} | "
        f"Beam data: {record['beam_data_count']:2d} | "
        f"Fluence: {record['fluence_count']:2d}"
    )

print("\nRead-only inventory complete.")

In [ ]:
# STEP 197D — LOCATE ORIGINAL PORTPY BEAM DATA
# Read-only inventory; no downloads or modifications.

from pathlib import Path
from collections import Counter

print("STEP 197D — ORIGINAL PORTPY DATA INVENTORY")
print("=" * 75)

roots = [
    project / "_portpy_download_cache",
    project
]

for root in roots:

    print("\nROOT:", root)
    print("Exists:", root.exists())

    if not root.exists():
        continue

    patient_dirs = sorted(
        [
            p for p in root.rglob("*")
            if p.is_dir()
            and (
                p.name.startswith("Prostate_Patient_")
                or (
                    p.name.startswith("Patient_")
                    and p.parent == project
                )
            )
        ],
        key=lambda p: str(p)
    )

    print("Patient directories found:", len(patient_dirs))

    for patient_dir in patient_dirs:

        files = [
            p for p in patient_dir.rglob("*")
            if p.is_file()
        ]

        names = [p.name for p in files]

        beam_meta = [
            name for name in names
            if name.startswith("Beam_")
            and name.endswith("_MetaData.json")
        ]

        beam_h5 = [
            name for name in names
            if name.startswith("Beam_")
            and name.endswith("_Data.h5")
        ]

        print("\n", patient_dir.relative_to(project)
              if patient_dir.is_relative_to(project)
              else patient_dir)

        print("  Total files:", len(files))
        print("  Beam metadata:", len(beam_meta))
        print("  Beam HDF5:", len(beam_h5))
        print("  PlannerBeams.json:",
              "PlannerBeams.json" in names)

        print("  Extensions:",
              dict(Counter(p.suffix.lower() for p in files)))

        if len(files) <= 20:
            print("  Files:")
            for p in files:
                print("   ", p.relative_to(patient_dir))

print("\n" + "=" * 75)
print("Inspection complete. No files modified.")

In [ ]:
# STEP 197E — INSPECT DICOM RT PLAN BEAM INFORMATION

import pydicom
from pathlib import Path

print("STEP 197E — DICOM RT PLAN INSPECTION")
print("=" * 80)

for n in range(1, 11):

    patient = f"Patient_{n}"

    plan_path = (
        project
        / patient
        / "raw_portpy"
        / "DicomFiles"
        / "rt_plan_echo_imrt.dcm"
    )

    print(f"\n{patient}")

    if not plan_path.exists():
        print("  RT plan missing")
        continue

    try:
        ds = pydicom.dcmread(
            str(plan_path),
            stop_before_pixels=True
        )

        beams = getattr(ds, "BeamSequence", [])

        print("  Plan label:",
              getattr(ds, "RTPlanLabel", "Unknown"))

        print("  Number of beams:", len(beams))

        angles = []

        for beam in beams:

            beam_number = getattr(
                beam, "BeamNumber", None
            )

            beam_name = getattr(
                beam, "BeamName", "Unknown"
            )

            control_points = getattr(
                beam, "ControlPointSequence", []
            )

            angle = None

            if len(control_points) > 0:
                angle = getattr(
                    control_points[0],
                    "GantryAngle",
                    None
                )

            angles.append(angle)

            print(
                f"    Beam {beam_number}: "
                f"name={beam_name}, "
                f"angle={angle}, "
                f"control_points={len(control_points)}"
            )

        print("  Gantry angles:", angles)

    except Exception as e:

        print("  ERROR:", type(e).__name__, str(e))

print("\n" + "=" * 80)
print("Inspection complete. No files modified.")

In [ ]:
# STEP 197F — INSPECT DICOM BEAM CONTROL POINTS
# Read-only; no model training or file changes.

import pydicom

print("STEP 197F — CONTROL POINT INSPECTION")
print("=" * 75)

for patient in ["Patient_1", "Patient_2"]:

    plan_path = (
        project / patient / "raw_portpy"
        / "DicomFiles" / "rt_plan_echo_imrt.dcm"
    )

    ds = pydicom.dcmread(str(plan_path))

    beam = ds.BeamSequence[0]
    cps = beam.ControlPointSequence

    print(f"\n{patient}")
    print("Beam number:", beam.BeamNumber)
    print("Beam type:", getattr(beam, "BeamType", "Unknown"))
    print("Treatment delivery:",
          getattr(beam, "TreatmentDeliveryType", "Unknown"))
    print("Number of control points:", len(cps))
    print("Final cumulative meterset:",
          getattr(beam, "FinalCumulativeMetersetWeight", None))

    indices = sorted(set([0, 1, len(cps)//2, len(cps)-1]))

    for i in indices:

        cp = cps[i]

        print(f"\n  CONTROL POINT {i}")
        print("  Cumulative meterset:",
              getattr(cp, "CumulativeMetersetWeight", None))
        print("  Gantry angle:",
              getattr(cp, "GantryAngle", None))

        positions = getattr(
            cp, "BeamLimitingDevicePositionSequence", []
        )

        if not positions:
            print("  No device positions explicitly stored")
            print("  (DICOM may inherit them from an earlier point)")

        for item in positions:

            values = list(item.LeafJawPositions)

            print(
                f"  Device: {item.RTBeamLimitingDeviceType}"
            )
            print("  Position count:", len(values))
            print("  First values:", values[:6])

print("\nInspection complete. No files modified.")

In [ ]:
# STEP 197G — VERIFY BEAM METERSETS AND MLC MOTION

import pydicom
import numpy as np

print("STEP 197G — BEAM DELIVERY INFORMATION")
print("=" * 75)

for patient in ["Patient_1", "Patient_2"]:

    plan_path = (
        project / patient / "raw_portpy"
        / "DicomFiles" / "rt_plan_echo_imrt.dcm"
    )

    ds = pydicom.dcmread(str(plan_path))

    print(f"\n{patient}")

    metersets = {}

    for fraction_group in getattr(
        ds, "FractionGroupSequence", []
    ):
        for ref in getattr(
            fraction_group, "ReferencedBeamSequence", []
        ):
            beam_number = int(ref.ReferencedBeamNumber)

            metersets[beam_number] = getattr(
                ref, "BeamMeterset", None
            )

    for beam in ds.BeamSequence:

        number = int(beam.BeamNumber)
        cps = beam.ControlPointSequence

        mlc_positions = []
        weights = []

        for cp in cps:

            weights.append(
                float(cp.CumulativeMetersetWeight)
            )

            for item in getattr(
                cp, "BeamLimitingDevicePositionSequence", []
            ):

                if item.RTBeamLimitingDeviceType == "MLCX":

                    mlc_positions.append(
                        np.array(
                            item.LeafJawPositions,
                            dtype=float
                        )
                    )

        mlc_positions = np.array(mlc_positions)
        weights = np.array(weights)

        if len(mlc_positions) > 1:

            max_mlc_change = float(
                np.max(
                    np.abs(
                        np.diff(mlc_positions, axis=0)
                    )
                )
            )

        else:
            max_mlc_change = float("nan")

        print(
            f"Beam {number:2d} | "
            f"MU={metersets.get(number)} | "
            f"CPs={len(cps)} | "
            f"MLC records={len(mlc_positions)} | "
            f"Max leaf movement={max_mlc_change:.3f} mm | "
            f"Weight monotonic={bool(np.all(np.diff(weights) >= -1e-8))}"
        )

print("\nInspection complete. No files modified.")

In [ ]:
# STEP 197H — INSPECT EXISTING PORTPY OPTIMIZATION ARTIFACTS
# Read-only. No changes to files or models.

from pathlib import Path
import json

print("STEP 197H — OPTIMIZATION PROVENANCE")
print("=" * 75)

patient_root = project / "Patient_1"
optimized_dir = patient_root / "optimized_portpy"

print("\nPATIENT 1 OPTIMIZATION DIRECTORY")

if optimized_dir.exists():
    for path in sorted(optimized_dir.rglob("*")):
        if path.is_file():
            print(
                path.relative_to(project),
                "|",
                path.stat().st_size,
                "bytes"
            )

print("\n" + "=" * 75)
print("POTENTIAL OPTIMIZATION SCRIPTS AND CONFIGURATIONS")

keywords = [
    "optim",
    "fluence",
    "objective",
    "solver",
    "portpy",
    "beamlet"
]

extensions = {
    ".py", ".json", ".yaml", ".yml",
    ".txt", ".md", ".ipynb"
}

matches = []

for path in project.rglob("*"):

    if not path.is_file():
        continue

    if path.suffix.lower() not in extensions:
        continue

    if any(
        word in path.name.lower()
        for word in keywords
    ):
        matches.append(path)

for path in matches[:100]:
    print(path.relative_to(project))

print("\nTotal matching files:", len(matches))

print("\n" + "=" * 75)
print("PATIENT 1 OPTIMIZATION METADATA")

for path in sorted(optimized_dir.glob("*.json")):

    print("\nFILE:", path.name)

    try:
        with open(path) as f:
            data = json.load(f)

        print(json.dumps(data, indent=2, default=str)[:5000])

    except Exception as e:
        print("Could not read:", e)

print("\nInspection complete. No files modified.")

In [ ]:
# STEP 197I — VERIFY OPTIMIZED INTENSITY AND FLUENCE MAPS
# Read-only. No file modifications.

import numpy as np
import json
from pathlib import Path

print("STEP 197I — OPTIMIZATION ARRAY CONSISTENCY")
print("=" * 75)

optimized_dir = project / "Patient_1" / "optimized_portpy"

intensity = np.load(
    optimized_dir / "optimal_intensity.npy"
)

fluence_files = sorted(
    optimized_dir.glob("beam_*_fluence.npy"),
    key=lambda p: int(p.stem.split("_")[1])
)

print("\nOPTIMAL INTENSITY")
print("Shape:", intensity.shape)
print("Size:", intensity.size)
print("Min:", float(np.min(intensity)))
print("Max:", float(np.max(intensity)))
print("Sum:", float(np.sum(intensity)))
print("Nonzero:", int(np.count_nonzero(intensity)))

print("\nBEAM FLUENCE MAPS")

all_maps = []

for path in fluence_files:

    arr = np.load(path)

    all_maps.append(arr.ravel())

    print(
        f"{path.name:25s} "
        f"shape={str(arr.shape):12s} "
        f"size={arr.size:5d} "
        f"nonzero={np.count_nonzero(arr):4d} "
        f"sum={np.sum(arr):.6f}"
    )

combined = np.concatenate(all_maps)

print("\nCOMBINED FLUENCE")
print("Total elements:", combined.size)
print("Sum:", float(combined.sum()))
print("Nonzero:", int(np.count_nonzero(combined)))

print("\nCONSISTENCY CHECKS")

print(
    "Same number of elements:",
    intensity.size == combined.size
)

print(
    "Same sum:",
    bool(np.isclose(
        intensity.sum(),
        combined.sum(),
        rtol=1e-5,
        atol=1e-6
    ))
)

if intensity.size == combined.size:

    print(
        "Direct flattened match:",
        bool(np.allclose(
            intensity.ravel(),
            combined,
            rtol=1e-5,
            atol=1e-6
        ))
    )

    print(
        "Sorted-value match:",
        bool(np.allclose(
            np.sort(intensity.ravel()),
            np.sort(combined),
            rtol=1e-5,
            atol=1e-6
        ))
    )

print("\n" + "=" * 75)
print("Inspection complete. No files modified.")

In [ ]:
# STEP 197J — TEST INTENSITY-TO-FLUENCE DUPLICATION
# Read-only; no changes to project files.

import numpy as np
from pathlib import Path

print("STEP 197J — DUPLICATION TEST")
print("=" * 75)

optimized_dir = project / "Patient_1" / "optimized_portpy"

intensity = np.load(
    optimized_dir / "optimal_intensity.npy"
).ravel()

fluence_files = sorted(
    optimized_dir.glob("beam_*_fluence.npy"),
    key=lambda p: int(p.stem.split("_")[1])
)

combined = np.concatenate([
    np.load(path).ravel()
    for path in fluence_files
])

# Remove zero values for comparison.
intensity_nonzero = intensity[intensity != 0]
fluence_nonzero = combined[combined != 0]

# Repeat every optimized intensity twice.
expected = np.repeat(intensity_nonzero, 2)

print("Optimized nonzero values:", intensity_nonzero.size)
print("Fluence nonzero values:", fluence_nonzero.size)
print("Expected after doubling:", expected.size)

print("\nTOTAL INTENSITY")
print("Original:", float(intensity.sum()))
print("Fluence:", float(combined.sum()))
print("Ratio:", float(combined.sum() / intensity.sum()))

print("\nSORTED-VALUE COMPARISON")

if expected.size == fluence_nonzero.size:

    expected_sorted = np.sort(expected)
    actual_sorted = np.sort(fluence_nonzero)

    print(
        "Every intensity duplicated twice:",
        bool(np.allclose(
            expected_sorted,
            actual_sorted,
            rtol=1e-6,
            atol=1e-8
        ))
    )

    print(
        "Maximum absolute difference:",
        float(np.max(np.abs(
            expected_sorted - actual_sorted
        )))
    )

else:
    print("Counts differ; direct comparison not possible.")

print("\n" + "=" * 75)
print("Test complete. No files modified.")

In [ ]:
# STEP 197K — INSPECT ORIGINAL PORTPY BEAMLET GEOMETRY
# Read-only. No file modifications.

import h5py
import numpy as np
from pathlib import Path

print("STEP 197K — PORTPY BEAMLET GEOMETRY")
print("=" * 75)

raw_dir = project / "Patient_1" / "raw_portpy"

for beam_id in [4, 12, 36]:

    path = raw_dir / f"Beam_{beam_id}_Data.h5"

    print(f"\nBEAM ID {beam_id}")

    if not path.exists():
        print("File missing")
        continue

    with h5py.File(path, "r") as f:

        if "beamlets" not in f:
            print("No beamlets group found")
            print("Available keys:", list(f.keys()))
            continue

        group = f["beamlets"]

        print("Beamlet datasets:", list(group.keys()))

        for key in [
            "id",
            "width_mm",
            "height_mm",
            "position_x_mm",
            "position_y_mm",
            "MLC_leaf_idx"
        ]:

            if key not in group:
                print(f"  {key}: missing")
                continue

            arr = np.asarray(group[key])

            print(f"\n  {key}")
            print("  Shape:", arr.shape)
            print("  First 10:", arr.ravel()[:10])

            if np.issubdtype(arr.dtype, np.number):
                print("  Unique values:",
                      np.unique(arr)[:20])

    fluence = np.load(
        project / "Patient_1" / "optimized_portpy"
        / f"beam_{beam_id}_fluence.npy"
    )

    print("\nSaved fluence grid:", fluence.shape)
    print("Nonzero pixels:", np.count_nonzero(fluence))

print("\n" + "=" * 75)
print("Inspection complete. No files modified.")

In [ ]:
# STEP 197L — VERIFY BEAMLET-TO-FLUENCE GRID MAPPING
# Read-only. No file modifications.

import numpy as np
import h5py

print("STEP 197L — SPATIAL MAPPING VERIFICATION")
print("=" * 75)

raw_dir = project / "Patient_1" / "raw_portpy"
optimized_dir = project / "Patient_1" / "optimized_portpy"

intensity = np.load(
    optimized_dir / "optimal_intensity.npy"
).ravel()

# Examine each beam's full beamlet count.
beam_ids = [36, 44, 52, 60, 68, 4, 12, 20, 28]

counts = []

for beam_id in beam_ids:
    with h5py.File(
        raw_dir / f"Beam_{beam_id}_Data.h5", "r"
    ) as f:
        counts.append(len(f["beamlets"]["id"]))

print("Total physical beamlets:", sum(counts))
print("Optimized intensity values:", len(intensity))

print("\nBEAMLET COUNT COMPARISON")

for beam_id, count in zip(beam_ids, counts):
    print(f"Beam {beam_id}: {count} physical beamlets")

print("\n" + "=" * 75)

if sum(counts) != len(intensity):
    print(
        "IMPORTANT: The full physical beamlet count does not "
        "match the optimized intensity vector length."
    )
    print(
        "This suggests that optimization uses a selected subset "
        "of beamlets or another indexing convention."
    )
    print(
        "We must inspect the beamlet-selection mapping before "
        "attempting direct spatial reconstruction."
    )
else:
    print(
        "Counts match. Direct spatial mapping can be investigated."
    )

print("\nInspection complete. No files modified.")

In [ ]:
# STEP 197M — INSPECT PORTPY BEAM METADATA REFERENCES
# Read-only: no files or model weights are modified.

import json
from pathlib import Path

print("STEP 197M — BEAM METADATA AND INDEXING")
print("=" * 75)

raw_dir = project / "Patient_1" / "raw_portpy"

for beam_id in [36, 4, 12]:

    path = raw_dir / f"Beam_{beam_id}_MetaData.json"

    print(f"\nBEAM ID {beam_id}")

    if not path.exists():
        print("Metadata file missing")
        continue

    with open(path, "r") as f:
        metadata = json.load(f)

    print("\nTOP-LEVEL KEYS:")
    for key in metadata:
        print(" ", key)

    print("\nBEAMLET AND INFLUENCE REFERENCES:")

    for key, value in metadata.items():

        key_lower = key.lower()

        if any(term in key_lower for term in [
            "beamlet",
            "influence",
            "matrix",
            "optimization",
            "sparse"
        ]):

            print(f"\n{key}:")
            print(json.dumps(
                value,
                indent=2,
                default=str
            )[:3000])

print("\n" + "=" * 75)
print("Inspection complete. No files modified.")

In [ ]:
# STEP 197N — IDENTIFY ORIGINAL PORTPY DATA SOURCE
# Read-only: no downloads or file modifications.

from pathlib import Path
import importlib.metadata
import json

print("STEP 197N — PORTPY DATA SOURCE INVESTIGATION")
print("=" * 75)

for package in ["portpy", "portpy-ai", "huggingface_hub"]:

    try:
        version = importlib.metadata.version(package)
        print(f"{package}: installed, version {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package}: not installed under this package name")

print("\nHUGGING FACE CACHE METADATA")
print("-" * 75)

cache_root = project / "_portpy_download_cache"

metadata_files = sorted(
    cache_root.rglob("*.metadata")
)

print("Metadata files found:", len(metadata_files))

for path in metadata_files[:15]:

    print("\nFILE:", path.relative_to(cache_root))

    try:
        print(path.read_text(errors="replace")[:600])
    except Exception as e:
        print("Read error:", e)

print("\nPROJECT CONFIGURATION FILES")
print("-" * 75)

for path in sorted(project.glob("*")):

    if path.is_file() and path.suffix.lower() in {
        ".json", ".txt", ".md", ".py", ".yaml", ".yml"
    }:
        print(path.name)

print("\n" + "=" * 75)
print("Inspection complete. No files modified.")

In [ ]:
# STEP 197O — CHECK OFFICIAL PORTPY DATASET AVAILABILITY
# Read-only remote listing; no large file downloads.

from huggingface_hub import HfApi

repo_id = "PortPy-Project/PortPy_Dataset"
api = HfApi()

print("STEP 197O — OFFICIAL PORTPY FILE AVAILABILITY")
print("=" * 75)

files = api.list_repo_files(
    repo_id=repo_id,
    repo_type="dataset"
)

for n in range(1, 11):
    prefix = f"data/Prostate_Patient_{n}/"

    patient_files = [
        p for p in files if p.startswith(prefix)
    ]

    beam_metadata = [
        p for p in patient_files
        if p.endswith("_MetaData.json")
        and "/Beam_" in p
    ]

    beam_data = [
        p for p in patient_files
        if p.endswith("_Data.h5")
        and "/Beam_" in p
    ]

    planner = [
        p for p in patient_files
        if p.endswith("PlannerBeams.json")
    ]

    print(f"\nProstate_Patient_{n}")
    print("  Total repository files:", len(patient_files))
    print("  Beam metadata files:", len(beam_metadata))
    print("  Beam HDF5 files:", len(beam_data))
    print("  Planner file:", bool(planner))

    if n == 2:
        print("\n  Patient 2 beam metadata examples:")
        for p in beam_metadata[:5]:
            print("   ", p)

print("\n" + "=" * 75)
print("Remote inventory complete. No files downloaded.")

In [ ]:
# STEP 197P — IDENTIFY PATIENT 2'S PLANNER-SELECTED BEAMS
# Downloads only the small planner JSON file.
# Does NOT download beam HDF5 matrices.

from huggingface_hub import hf_hub_download, HfApi
from pathlib import Path
import json

repo_id = "PortPy-Project/PortPy_Dataset"
patient_number = 2

remote_path = (
    f"data/Prostate_Patient_{patient_number}/PlannerBeams.json"
)

print("STEP 197P — PATIENT 2 PLANNER BEAMS")
print("=" * 75)

planner_path = hf_hub_download(
    repo_id=repo_id,
    repo_type="dataset",
    filename=remote_path,
    local_dir=str(project / "_portpy_download_cache")
)

with open(planner_path, "r") as f:
    planner = json.load(f)

print("\nPLANNER JSON CONTENT:")
print(json.dumps(planner, indent=2)[:5000])

api = HfApi()
remote_files = set(
    api.list_repo_files(repo_id=repo_id, repo_type="dataset")
)

print("\nPLANNER-SELECTED BEAM FILE CHECK")

if isinstance(planner, list):
    beam_ids = planner
elif isinstance(planner, dict):
    beam_ids = planner.get("beam_ids", planner.get("beamIDs", []))
else:
    beam_ids = []

print("Automatically extracted beam IDs:", beam_ids)

if beam_ids:
    for beam_id in beam_ids:
        base = (
            f"data/Prostate_Patient_{patient_number}/Beams/"
            f"Beam_{beam_id}"
        )

        metadata_exists = f"{base}_MetaData.json" in remote_files
        h5_exists = f"{base}_Data.h5" in remote_files

        print(
            f"Beam {beam_id}: "
            f"metadata={metadata_exists}, "
            f"HDF5={h5_exists}"
        )
else:
    print(
        "Beam IDs were not automatically extracted. "
        "Inspect the JSON content above before proceeding."
    )

print("\n" + "=" * 75)
print("Step complete. No large beam files downloaded.")

In [ ]:
# STEP 197Q — VERIFY PATIENT 2 PLANNER BEAM METADATA
# Downloads only nine metadata JSON files.
# Does NOT download large beam HDF5 files.

from huggingface_hub import hf_hub_download, HfApi
import json

print("STEP 197Q — PATIENT 2 BEAM VERIFICATION")
print("=" * 75)

repo_id = "PortPy-Project/PortPy_Dataset"
patient_number = 2

beam_ids = [36, 44, 52, 60, 68, 4, 12, 20, 28]

api = HfApi()

for beam_id in beam_ids:

    remote_path = (
        f"data/Prostate_Patient_{patient_number}/Beams/"
        f"Beam_{beam_id}_MetaData.json"
    )

    local_path = hf_hub_download(
        repo_id=repo_id,
        repo_type="dataset",
        filename=remote_path,
        local_dir=str(project / "_portpy_download_cache")
    )

    with open(local_path, "r") as f:
        metadata = json.load(f)

    print(
        f"Beam ID {beam_id:2d} | "
        f"Gantry angle: {metadata.get('gantry_angle')}° | "
        f"Energy: {metadata.get('energy_MV')} MV"
    )

print("\n" + "=" * 75)
print("PATIENT 2 BEAM HDF5 FILE SIZES")

paths = [
    f"data/Prostate_Patient_{patient_number}/Beams/"
    f"Beam_{beam_id}_Data.h5"
    for beam_id in beam_ids
]

total_bytes = 0

for path in paths:

    info = api.get_paths_info(
        repo_id=repo_id,
        paths=[path],
        repo_type="dataset"
    )

    if info:
        size = info[0].size or 0
        total_bytes += size

        print(
            path.split("/")[-1],
            f"{size / (1024**2):.2f} MB"
        )
    else:
        print("NOT FOUND:", path)

print("\nTotal download size:",
      f"{total_bytes / (1024**3):.3f} GB")

print("\nStep complete. No beam HDF5 files downloaded.")

In [ ]:
# STEP 197R — INSPECT PORTPY INFLUENCE MATRIX STORAGE
# Read-only. Uses Patient 1 files already on Drive.

import h5py
import numpy as np

print("STEP 197R — INFLUENCE MATRIX STORAGE")
print("=" * 80)

raw_dir = project / "Patient_1" / "raw_portpy"

for beam_id in [36, 4, 12]:

    path = raw_dir / f"Beam_{beam_id}_Data.h5"

    print(f"\nBEAM {beam_id}")
    print("File size:",
          f"{path.stat().st_size / (1024**2):.2f} MB")

    with h5py.File(path, "r") as f:

        for key in ["inf_matrix_sparse", "inf_matrix_full"]:

            if key not in f:
                print(f"  {key}: NOT FOUND")
                continue

            obj = f[key]

            print(f"\n  {key}")
            print("  HDF5 object type:",
                  type(obj).__name__)

            if isinstance(obj, h5py.Dataset):

                print("  Shape:", obj.shape)
                print("  Data type:", obj.dtype)
                print("  Compression:", obj.compression)

                logical_bytes = (
                    obj.size * obj.dtype.itemsize
                )

                print(
                    "  Uncompressed logical size:",
                    f"{logical_bytes / (1024**2):.2f} MB"
                )

                print(
                    "  Stored size:",
                    f"{obj.id.get_storage_size() / (1024**2):.2f} MB"
                )

            elif isinstance(obj, h5py.Group):

                print("  Group members:")

                def describe(name, item):
                    if isinstance(item, h5py.Dataset):
                        print(
                            f"    {name}: "
                            f"shape={item.shape}, "
                            f"dtype={item.dtype}, "
                            f"stored="
                            f"{item.id.get_storage_size() / (1024**2):.2f} MB"
                        )

                obj.visititems(describe)

print("\n" + "=" * 80)
print("Inspection complete. No files modified.")

In [ ]:
# STEP 197S — VALIDATE PORTPY SPARSE MATRIX FORMAT
# Read-only. Tests Patient 1, Beam 36.
# Does not load the complete dense matrix.

import h5py
import numpy as np

print("STEP 197S — SPARSE MATRIX VALIDATION")
print("=" * 75)

path = (
    project
    / "Patient_1"
    / "raw_portpy"
    / "Beam_36_Data.h5"
)

with h5py.File(path, "r") as f:

    sparse = f["inf_matrix_sparse"]
    full = f["inf_matrix_full"]

    print("Sparse shape:", sparse.shape)
    print("Full shape:", full.shape)

    # Read a small sample of sparse entries.
    sample = sparse[:20]

    print("\nFIRST 10 SPARSE ROWS:")
    print(sample[:10])

    print("\nCOLUMN RANGES IN SAMPLE:")

    for col in range(3):
        print(
            f"Column {col}: "
            f"min={sample[:, col].min()}, "
            f"max={sample[:, col].max()}"
        )

    # Test whether the first two columns look like
    # integer indices and identify their possible roles.

    print("\nPOSSIBLE INDEX COLUMNS:")

    for col in range(3):

        values = sample[:, col]

        integer_like = np.all(
            np.isclose(values, np.round(values))
        )

        print(
            f"Column {col}: "
            f"integer-like={integer_like}"
        )

    print("\nDENSE MATRIX SAMPLE:")
    print(full[:5, :5])

print("\n" + "=" * 75)
print("Inspection complete. No files modified.")

In [ ]:
# STEP 197T — VALIDATE SPARSE AGAINST FULL MATRIX
# Read-only. Patient 1, Beam 36.
# No complete dense matrix loaded.

import h5py
import numpy as np

print("STEP 197T — SPARSE/FULL MATRIX CONSISTENCY")
print("=" * 75)

path = (
    project
    / "Patient_1"
    / "raw_portpy"
    / "Beam_36_Data.h5"
)

with h5py.File(path, "r") as f:

    sparse = f["inf_matrix_sparse"]
    full = f["inf_matrix_full"]

    n_rows, n_cols = full.shape
    n_entries = sparse.shape[0]

    # Sample entries across the entire sparse dataset.
    sample_indices = np.linspace(
        0, n_entries - 1, 100, dtype=int
    )

    sample = sparse[sample_indices]

    voxel_indices = sample[:, 0].astype(np.int64)
    beamlet_indices = sample[:, 1].astype(np.int64)
    sparse_values = sample[:, 2]

    valid_indices = (
        np.all(voxel_indices >= 0)
        and np.all(voxel_indices < n_rows)
        and np.all(beamlet_indices >= 0)
        and np.all(beamlet_indices < n_cols)
    )

    print("Full matrix shape:", full.shape)
    print("Sparse entries:", n_entries)
    print("Sampled entries:", len(sample))
    print("All sampled indices valid:", valid_indices)

    if valid_indices:

        full_values = np.array([
            full[int(r), int(c)]
            for r, c in zip(voxel_indices, beamlet_indices)
        ])

        differences = np.abs(
            full_values - sparse_values
        )

        print("\nCOMPARISON RESULTS")
        print("Maximum absolute difference:",
              differences.max())
        print("Mean absolute difference:",
              differences.mean())

        print(
            "Matching entries (tolerance 1e-6):",
            np.sum(np.isclose(
                full_values,
                sparse_values,
                atol=1e-6,
                rtol=0
            )),
            "/",
            len(sample)
        )

        print("\nFIRST FIVE COMPARISONS")

        for i in range(5):
            print(
                f"Voxel {voxel_indices[i]}, "
                f"Beamlet {beamlet_indices[i]}: "
                f"sparse={sparse_values[i]:.8f}, "
                f"full={full_values[i]:.8f}"
            )

print("\n" + "=" * 75)
print("Validation complete. No files modified.")

In [ ]:
# STEP 197U — DOWNLOAD PATIENT 2 BEAM 36
# Downloads one original PortPy beam HDF5 file.

from huggingface_hub import hf_hub_download
from pathlib import Path
import h5py
import shutil

print("STEP 197U — PATIENT 2 BEAM 36")
print("=" * 75)

repo_id = "PortPy-Project/PortPy_Dataset"

remote_path = (
    "data/Prostate_Patient_2/Beams/"
    "Beam_36_Data.h5"
)

destination = (
    project / "Patient_2" / "raw_portpy"
    / "Beam_36_Data.h5"
)

destination.parent.mkdir(parents=True, exist_ok=True)

if destination.exists():
    print("Existing file found:", destination)
    print("Checking existing HDF5 file...")

    try:
        with h5py.File(destination, "r") as f:
            valid = (
                "inf_matrix_sparse" in f
                and "inf_matrix_full" in f
                and "beamlets" in f
            )

        if not valid:
            raise ValueError(
                "Existing file is missing expected datasets."
            )

        print("Existing file passed structural checks.")

    except Exception as e:
        print("Existing file could not be validated:", e)
        raise RuntimeError(
            "Download stopped to avoid overwriting an existing file."
        ) from e

else:
    print("Downloading official beam file...")

    downloaded = hf_hub_download(
        repo_id=repo_id,
        repo_type="dataset",
        filename=remote_path,
        local_dir=str(project / "_portpy_download_cache")
    )

    shutil.copy2(downloaded, destination)

    print("Download and copy complete.")

print("\nFILE VERIFICATION")

with h5py.File(destination, "r") as f:
    print("Sparse matrix shape:",
          f["inf_matrix_sparse"].shape)
    print("Full matrix shape:",
          f["inf_matrix_full"].shape)
    print("Beamlet count:",
          len(f["beamlets/id"]))

print(
    "File size:",
    f"{destination.stat().st_size / (1024**2):.2f} MB"
)

print("\nSaved to:", destination)
print("\n" + "=" * 75)
print("Step 197U complete.")

In [ ]:
# STEP 197V — DOWNLOAD PATIENT 2'S REMAINING EIGHT BEAMS

from huggingface_hub import hf_hub_download
from pathlib import Path
import h5py
import shutil

print("STEP 197V — PATIENT 2 REMAINING BEAMS")
print("=" * 75)

repo_id = "PortPy-Project/PortPy_Dataset"
remaining_beams = [44, 52, 60, 68, 4, 12, 20, 28]

destination_dir = project / "Patient_2" / "raw_portpy"
destination_dir.mkdir(parents=True, exist_ok=True)

def verify_beam_file(path):
    with h5py.File(path, "r") as f:
        required = [
            "inf_matrix_sparse",
            "inf_matrix_full",
            "beamlets/id"
        ]

        missing = [key for key in required if key not in f]

        if missing:
            raise ValueError(f"Missing datasets: {missing}")

        return {
            "beamlets": len(f["beamlets/id"]),
            "sparse_shape": f["inf_matrix_sparse"].shape,
            "full_shape": f["inf_matrix_full"].shape
        }

for index, beam_id in enumerate(remaining_beams, 1):

    print(f"\n[{index}/8] BEAM {beam_id}")

    filename = f"Beam_{beam_id}_Data.h5"
    destination = destination_dir / filename

    if destination.exists():
        info = verify_beam_file(destination)
        print("Existing file verified. Skipping download.")

    else:
        remote_path = (
            f"data/Prostate_Patient_2/Beams/{filename}"
        )

        downloaded = hf_hub_download(
            repo_id=repo_id,
            repo_type="dataset",
            filename=remote_path,
            local_dir=str(project / "_portpy_download_cache")
        )

        # Copy to a temporary filename before finalizing.
        temporary = destination.with_suffix(".h5.partial")

        try:
            shutil.copy2(downloaded, temporary)
            info = verify_beam_file(temporary)
            temporary.replace(destination)
        finally:
            if temporary.exists():
                temporary.unlink()

        print("Downloaded and verified.")

    print("Beamlets:", info["beamlets"])
    print("Sparse matrix:", info["sparse_shape"])
    print("Full matrix:", info["full_shape"])
    print(
        "File size:",
        f"{destination.stat().st_size / (1024**2):.2f} MB"
    )

print("\n" + "=" * 75)
print("STEP 197V COMPLETE")

all_beams = [36, 44, 52, 60, 68, 4, 12, 20, 28]

missing = [
    beam_id for beam_id in all_beams
    if not (
        destination_dir / f"Beam_{beam_id}_Data.h5"
    ).exists()
]

print("Planner beams present:", len(all_beams) - len(missing), "/9")
print("Missing beams:", missing)

In [ ]:
# STEP 197W — SEARCH FOR EXISTING PORTPY OPTIMIZATION SETUP
# Read-only: no files modified or downloaded.

from pathlib import Path
import json

print("STEP 197W — PORTPY OPTIMIZATION PROVENANCE")
print("=" * 75)

keywords = [
    "optim", "fluence", "solver", "objective",
    "portpy", "intensity", "beamlet", "config"
]

extensions = {
    ".py", ".ipynb", ".json", ".yaml",
    ".yml", ".toml", ".txt", ".md"
}

matches = []

# Search project files without traversing the large download cache.
for path in project.rglob("*"):

    if not path.is_file():
        continue

    if "_portpy_download_cache" in path.parts:
        continue

    if path.suffix.lower() not in extensions:
        continue

    if any(word in path.name.lower() for word in keywords):
        matches.append(path)

print("\nPOTENTIALLY RELEVANT FILES:", len(matches))

for path in sorted(matches)[:80]:
    print(" ", path.relative_to(project))

print("\nPATIENT 1 OPTIMIZATION OUTPUTS")
print("-" * 75)

opt_dir = project / "Patient_1" / "optimized_portpy"

if opt_dir.exists():
    for path in sorted(opt_dir.iterdir()):
        if path.is_file():
            print(
                path.name,
                f"({path.stat().st_size / (1024**2):.2f} MB)"
            )
else:
    print("Optimization directory not found.")

print("\nPATIENT 2 OPTIMIZATION OUTPUTS")
print("-" * 75)

opt_dir2 = project / "Patient_2" / "optimized_portpy"

if opt_dir2.exists():
    for path in sorted(opt_dir2.iterdir()):
        if path.is_file():
            print(path.name)
else:
    print("No Patient 2 optimization directory.")

print("\n" + "=" * 75)
print("Inspection complete. No files modified.")

In [ ]:
# STEP 197X — INSPECT OFFICIAL PORTPY OPTIMIZATION CODE
# Read-only investigation; no project files modified.

import subprocess
from pathlib import Path

print("STEP 197X — OFFICIAL PORTPY OPTIMIZATION SOURCE")
print("=" * 75)

source_dir = Path("/content/portpy_source_inspection")

if not (source_dir / ".git").exists():
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/PortPy-Project/PortPy.git",
            str(source_dir)
        ],
        check=True
    )
else:
    print("Existing source checkout found.")

print("\nRELEVANT SOURCE FILES")
print("-" * 75)

keywords = [
    "optimization",
    "fluence",
    "influence_matrix",
    "beamlet"
]

matches = []

for path in source_dir.rglob("*.py"):
    if any(k in path.name.lower() for k in keywords):
        matches.append(path)

for path in sorted(matches)[:50]:
    print(path.relative_to(source_dir))

print("\nOPTIMIZATION EXAMPLES")
print("-" * 75)

for path in sorted(source_dir.rglob("*.ipynb")):
    name = path.name.lower()
    if any(k in name for k in ["optim", "fluence", "imrt"]):
        print(path.relative_to(source_dir))

print("\n" + "=" * 75)
print("Source inspection complete.")

In [ ]:
# STEP 197Y — INSPECT OFFICIAL PORTPY IMRT OPTIMIZATION EXAMPLE
# Read-only. Does not execute the notebook.

import json
from pathlib import Path

print("STEP 197Y — OFFICIAL IMRT OPTIMIZATION WORKFLOW")
print("=" * 80)

notebook_path = (
    Path("/content/portpy_source_inspection")
    / "examples"
    / "imrt_global_optimal.ipynb"
)

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

code_cells = [
    "".join(cell.get("source", []))
    for cell in notebook.get("cells", [])
    if cell.get("cell_type") == "code"
]

print("Total code cells:", len(code_cells))

keywords = [
    "Optimization(",
    "InfluenceMatrix(",
    "create_cvxpy_problem",
    "solve(",
    "get_optimal",
    "objective",
    "constraint",
    "fluence",
    "beam_ids",
    "down_sample",
]

for i, source in enumerate(code_cells, 1):

    if any(keyword.lower() in source.lower()
           for keyword in keywords):

        print("\n" + "-" * 80)
        print(f"CODE CELL {i}")
        print("-" * 80)
        print(source[:5000])

print("\n" + "=" * 80)
print("Inspection complete. No optimization executed.")

In [ ]:
# STEP 197Z — FIND OFFICIAL PROSTATE OPTIMIZATION CONFIGURATIONS
# Read-only. No optimization or downloads.

from pathlib import Path
import json

print("STEP 197Z — PROSTATE OPTIMIZATION CONFIGURATIONS")
print("=" * 80)

source_dir = Path("/content/portpy_source_inspection")

print("\nPROSTATE-RELATED CONFIGURATION FILES")
print("-" * 80)

matches = []

for path in source_dir.rglob("*"):
    if not path.is_file():
        continue

    if path.suffix.lower() not in {
        ".json", ".py", ".ipynb", ".yaml", ".yml"
    }:
        continue

    if "prostate" in path.name.lower():
        matches.append(path)

for path in sorted(matches)[:60]:
    print(path.relative_to(source_dir))

print("\nPROSTATE REFERENCES IN OPTIMIZATION EXAMPLES")
print("-" * 80)

examples_dir = source_dir / "examples"

for path in sorted(examples_dir.rglob("*.ipynb")):

    try:
        with open(path, encoding="utf-8") as f:
            notebook = json.load(f)

        code = "\n".join(
            "".join(cell.get("source", []))
            for cell in notebook.get("cells", [])
            if cell.get("cell_type") == "code"
        )

        if "prostate" in code.lower():
            print("\nNOTEBOOK:", path.relative_to(source_dir))

            for line in code.splitlines():
                if any(term in line.lower() for term in [
                    "prostate",
                    "protocol_name",
                    "beamlet_width_mm",
                    "beamlet_height_mm",
                    "opt_vox_xyz_res_mm"
                ]):
                    print(" ", line[:180])

    except Exception as e:
        print("Could not inspect:", path.name, str(e)[:120])

print("\n" + "=" * 80)
print("Inspection complete. No files modified.")

In [ ]:
# STEP 198A — INSPECT OFFICIAL PROSTATE IMRT CONFIGURATION
# Read-only. No optimization, downloads or file modifications.

from pathlib import Path
import json

print("STEP 198A — PROSTATE IMRT CONFIGURATION")
print("=" * 80)

root = Path("/content/portpy_source_inspection/portpy/config_files")

config_files = {
    "Clinical criteria — Prostate 26Fx":
        root / "clinical_criteria/Default/Prostate_26Fx.json",

    "Optimization parameters — Prostate 26Fx":
        root / "optimization_params/optimization_params_Prostate_26Fx.json",

    "Clinical criteria — Prostate 5Gy 5Fx":
        root / "clinical_criteria/Default/Prostate_5Gy_5Fx.json"
}

for label, path in config_files.items():

    print("\n" + "=" * 80)
    print(label)
    print("File:", path)

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    with open(path, "r", encoding="utf-8") as f:
        config = json.load(f)

    print("\nTOP-LEVEL KEYS:")
    print(list(config.keys()))

    print("\nCONFIGURATION CONTENT:")
    print(json.dumps(config, indent=2)[:12000])

print("\n" + "=" * 80)
print("Inspection complete. No files modified.")

In [ ]:
# STEP 198B — COMPARE PATIENT 2 STRUCTURES WITH PROSTATE PROTOCOL
# Read-only inspection. No optimization performed.

from pathlib import Path
import json

print("STEP 198B — PATIENT 2 STRUCTURE COMPATIBILITY")
print("=" * 80)

patient_dir = project / "Patient_2" / "raw_portpy"

print("\nPATIENT 2 STRUCTURE FILES")
print("-" * 80)

structure_files = sorted(
    p for p in patient_dir.glob("*")
    if "structure" in p.name.lower()
)

for path in structure_files:
    print(path.name)

print("\nPATIENT 2 STRUCTURE NAMES")
print("-" * 80)

patient_structures = set()

for path in structure_files:
    if path.suffix.lower() != ".json":
        continue

    with open(path, encoding="utf-8") as f:
        metadata = json.load(f)

    if isinstance(metadata, dict):
        for key in ["name", "names", "structure_names", "structures"]:
            values = metadata.get(key, [])

            if isinstance(values, list):
                for item in values:
                    if isinstance(item, str):
                        patient_structures.add(item)
                    elif isinstance(item, dict):
                        for name_key in ["name", "structure_name"]:
                            if name_key in item:
                                patient_structures.add(item[name_key])

print(sorted(patient_structures) if patient_structures
      else "No structure names extracted from available JSON files.")

print("\nPROTOCOL STRUCTURE REQUIREMENTS")
print("-" * 80)

config_path = (
    Path("/content/portpy_source_inspection")
    / "portpy/config_files/optimization_params"
    / "optimization_params_Prostate_26Fx.json"
)

with open(config_path, encoding="utf-8") as f:
    config = json.load(f)

required = set()

for objective in config.get("objective_functions", []):
    name = objective.get("structure_name")
    if name:
        required.add(name)

for constraint in config.get("constraints", []):
    name = constraint.get("parameters", {}).get("structure_name")
    if name:
        required.add(name)

for name in sorted(required):
    if name.startswith("RIND_"):
        status = "DERIVED STRUCTURE"
    elif not patient_structures:
        status = "NOT YET VERIFIED"
    elif name in patient_structures:
        status = "FOUND"
    else:
        status = "NOT FOUND"

    print(f"{name:<22} {status}")

print("\n" + "=" * 80)
print("Inspection complete. No files modified.")

In [ ]:
# STEP 198C — INSPECT PATIENT 2 STRUCTURE METADATA
# Read-only. No files modified.

import json
import h5py
from pathlib import Path

print("STEP 198C — PATIENT 2 STRUCTURE METADATA")
print("=" * 80)

patient_dir = project / "Patient_2" / "raw_portpy"

json_path = patient_dir / "StructureSet_MetaData.json"
h5_path = patient_dir / "StructureSet_Data.h5"

print("\n1. STRUCTURE METADATA JSON")
print("-" * 80)

if json_path.exists():
    with open(json_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)

    print("Top-level type:", type(metadata).__name__)

    if isinstance(metadata, dict):
        print("Top-level keys:", list(metadata.keys()))

    elif isinstance(metadata, list):
        print("Number of entries:", len(metadata))

    print("\nJSON CONTENT (FIRST 12,000 CHARACTERS):")
    print(json.dumps(metadata, indent=2)[:12000])

else:
    print("JSON file not found.")

print("\n2. STRUCTURE HDF5 CONTENT")
print("-" * 80)

if h5_path.exists():

    with h5py.File(h5_path, "r") as f:

        print("Top-level HDF5 keys:", list(f.keys()))

        count = [0]

        def show_item(name, obj):

            if count[0] >= 40:
                return

            if isinstance(obj, h5py.Dataset):
                print(
                    "DATASET:",
                    name,
                    "shape=", obj.shape,
                    "dtype=", obj.dtype
                )
            else:
                print("GROUP:", name)

            count[0] += 1

        f.visititems(show_item)

else:
    print("HDF5 file not found.")

print("\n" + "=" * 80)
print("Inspection complete. No files modified.")

In [ ]:
# STEP 198D — INSPECT PORTPY OPTIMIZATION IMPLEMENTATION
# Read-only. No optimization executed.

from pathlib import Path
import ast

print("STEP 198D — PORTPY OPTIMIZATION IMPLEMENTATION")
print("=" * 80)

source_file = (
    Path("/content/portpy_source_inspection")
    / "portpy/photon/optimization.py"
)

source = source_file.read_text(encoding="utf-8")
tree = ast.parse(source)

target_methods = {
    "__init__",
    "create_cvxpy_problem",
    "solve",
    "get_smoothness_matrix"
}

for node in ast.walk(tree):

    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        if node.name not in target_methods:
            continue

        print("\n" + "-" * 80)
        print("FUNCTION:", node.name)
        print("LINES:", node.lineno, "-", node.end_lineno)
        print("-" * 80)

        lines = source.splitlines()
        function_lines = lines[node.lineno - 1:node.end_lineno]

        for line in function_lines[:100]:
            print(line)

        if len(function_lines) > 100:
            print(
                f"\n... {len(function_lines) - 100} "
                "additional lines omitted ..."
            )

print("\n" + "=" * 80)
print("Inspection complete. No files modified.")

In [ ]:
# STEP 198E — LOCATE OFFICIAL BEAMLET-TO-FLUENCE MAPPING
# Read-only source inspection.

from pathlib import Path
import ast

print("STEP 198E — BEAMLET-TO-FLUENCE MAPPING")
print("=" * 80)

source_file = (
    Path("/content/portpy_source_inspection")
    / "portpy/photon/influence_matrix.py"
)

source = source_file.read_text(encoding="utf-8")
tree = ast.parse(source)

keywords = [
    "fluence",
    "bev",
    "beamlet",
    "2d_grid",
    "solution"
]

matches = []

for node in ast.walk(tree):
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):

        if any(k in node.name.lower() for k in keywords):
            matches.append(node)

print("\nRELEVANT METHODS")
print("-" * 80)

for node in matches:
    print(
        f"{node.name:<45} "
        f"lines {node.lineno}-{node.end_lineno}"
    )

print("\nMAPPING METHODS — SOURCE")
print("-" * 80)

priority = [
    "get_bev_2d_grid",
    "fluence_1d_to_2d",
    "fluence_2d_to_1d"
]

lines = source.splitlines()

for node in matches:

    if node.name not in priority:
        continue

    print("\n" + "-" * 80)
    print("FUNCTION:", node.name)
    print("-" * 80)

    function_lines = lines[node.lineno - 1:node.end_lineno]

    for line in function_lines[:110]:
        print(line)

    if len(function_lines) > 110:
        print(
            f"\n... {len(function_lines)-110} "
            "additional lines omitted ..."
        )

print("\n" + "=" * 80)
print("Inspection complete. No files modified.")

In [ ]:
# STEP 198F — PATIENT 1 FLUENCE VALUE CONSISTENCY
# Read-only. No optimization or large matrix loading.

import numpy as np
from pathlib import Path

print("STEP 198F — PATIENT 1 FLUENCE CONSISTENCY")
print("=" * 80)

opt_dir = project / "Patient_1" / "optimized_portpy"

x = np.load(opt_dir / "optimal_intensity.npy").ravel()

beam_ids = [36, 44, 52, 60, 68, 4, 12, 20, 28]

maps = [
    np.load(opt_dir / f"beam_{beam_id}_fluence.npy")
    for beam_id in beam_ids
]

print("\nOPTIMIZED INTENSITY VECTOR")
print("-" * 80)
print("Length:", len(x))
print("Nonzero values:", np.count_nonzero(x))
print("Minimum:", float(x.min()))
print("Maximum:", float(x.max()))

print("\nSAVED FLUENCE MAPS")
print("-" * 80)

all_pixels = []

for beam_id, fluence in zip(beam_ids, maps):

    print(
        f"Beam {beam_id:<3} "
        f"shape={str(fluence.shape):<12} "
        f"nonzero={np.count_nonzero(fluence):<5} "
        f"sum={fluence.sum():.6f}"
    )

    all_pixels.extend(fluence.ravel())

all_pixels = np.asarray(all_pixels)

print("\nNUMERICAL VALUE MATCHING")
print("-" * 80)

positive_x = x[x > 0]
positive_pixels = all_pixels[all_pixels > 0]

# Compare values after rounding to limit floating-point noise.
# This checks value membership, not spatial correctness.
precision = 6

x_values = set(np.round(positive_x, precision))
pixel_values = np.round(positive_pixels, precision)

matched = sum(value in x_values for value in pixel_values)

print("Positive intensity entries:", len(positive_x))
print("Positive fluence pixels:", len(positive_pixels))
print("Matching fluence pixels:", matched)

if len(positive_pixels):
    percentage = 100 * matched / len(positive_pixels)
    print(f"Value matching rate: {percentage:.2f}%")

print("\n" + "=" * 80)
print("Inspection complete. No files modified.")

In [ ]:
# STEP 198G — VERIFY EXACT FLUENCE VALUE DUPLICATION
# Read-only. No optimization or file modifications.

import numpy as np
from collections import Counter

print("STEP 198G — EXACT FLUENCE VALUE DUPLICATION")
print("=" * 75)

opt_dir = project / "Patient_1" / "optimized_portpy"
beam_ids = [36, 44, 52, 60, 68, 4, 12, 20, 28]

x = np.load(opt_dir / "optimal_intensity.npy").ravel()

pixels = np.concatenate([
    np.load(opt_dir / f"beam_{bid}_fluence.npy").ravel()
    for bid in beam_ids
])

# Use exact float64 values, without rounding.
x_positive = x[x > 0]
pixel_positive = pixels[pixels > 0]

x_counts = Counter(x_positive.tolist())
pixel_counts = Counter(pixel_positive.tolist())

all_values = set(x_counts) | set(pixel_counts)

mismatches = [
    (value, x_counts[value], pixel_counts[value])
    for value in all_values
    if pixel_counts[value] != 2 * x_counts[value]
]

print("Positive intensity entries:", len(x_positive))
print("Positive fluence pixels:", len(pixel_positive))
print("Unique positive intensity values:", len(x_counts))
print("Unique positive fluence values:", len(pixel_counts))
print("Values with incorrect duplication counts:", len(mismatches))

if not mismatches:
    print("\nPASS: Every positive intensity value occurs")
    print("exactly twice as often in the fluence maps.")
else:
    print("\nFirst 10 mismatches:")
    for value, x_count, pixel_count in mismatches[:10]:
        print(
            f"value={value:.12g}, "
            f"intensity_count={x_count}, "
            f"fluence_count={pixel_count}"
        )

print("\nSUM CHECK")
print("-" * 75)
print("Intensity sum:", float(x.sum()))
print("Fluence sum:", float(pixels.sum()))
print("Twice intensity sum:", float(2 * x.sum()))
print(
    "Absolute sum difference:",
    float(abs(pixels.sum() - 2 * x.sum()))
)

print("\n" + "=" * 75)
print("Inspection complete. No files modified.")

In [ ]:
# STEP 198H — LOCATE PATIENT 1 BEAMLET MAPPING DATA
# Read-only. No files modified.

from pathlib import Path
import json

print("STEP 198H — PATIENT 1 BEAMLET MAPPING DATA")
print("=" * 80)

patient_dir = project / "Patient_1"

print("\n1. PATIENT 1 FILES RELEVANT TO MAPPING")
print("-" * 80)

keywords = (
    "beamlet",
    "fluence",
    "intensity",
    "influence",
    "mapping",
    "optimization",
    "solution",
    "plan"
)

matches = []

for path in patient_dir.rglob("*"):
    if path.is_file() and any(
        word in path.name.lower() for word in keywords
    ):
        matches.append(path)

for path in sorted(matches)[:100]:
    print(path.relative_to(patient_dir))

print("\nMatching files:", len(matches))

print("\n2. VERIFIED BEAM MAPPING")
print("-" * 80)

mapping_path = patient_dir / "verified_beam_mapping.json"

if mapping_path.exists():
    with open(mapping_path, encoding="utf-8") as f:
        mapping = json.load(f)

    print(json.dumps(mapping, indent=2)[:6000])
else:
    print("No saved verified beam mapping found.")

print("\n3. PORTPY MAPPING FUNCTION LOCATIONS")
print("-" * 80)

source_path = (
    Path("/content/portpy_source_inspection")
    / "portpy/photon/influence_matrix.py"
)

source_lines = source_path.read_text(
    encoding="utf-8"
).splitlines()

for start, end in [
    (654, 682),
    (684, 723)
]:
    print(f"\nSOURCE LINES {start}-{end}")
    print("-" * 60)

    for line_number in range(start, end + 1):
        print(
            f"{line_number:4}: "
            f"{source_lines[line_number - 1]}"
        )

print("\n" + "=" * 80)
print("Inspection complete. No files modified.")

In [ ]:
# STEP 198I — PATIENT 1 RAW BEAMLET GEOMETRY
# Read-only. No optimization or file modifications.

import h5py
import numpy as np
from pathlib import Path

print("STEP 198I — PATIENT 1 BEAMLET GEOMETRY")
print("=" * 80)

raw_dir = project / "Patient_1" / "raw_portpy"

beam_ids = [36, 44, 52, 60, 68, 4, 12, 20, 28]

for beam_id in beam_ids:

    path = raw_dir / f"Beam_{beam_id}_Data.h5"

    print(f"\nBEAM {beam_id}")
    print("-" * 60)

    if not path.exists():
        print("File not found:", path)
        continue

    with h5py.File(path, "r") as f:

        if "beamlets" not in f:
            print("No beamlets group found.")
            print("Top-level keys:", list(f.keys()))
            continue

        group = f["beamlets"]

        print("Available beamlet fields:", list(group.keys()))

        for field in [
            "width_mm",
            "height_mm",
            "position_x_mm",
            "position_y_mm",
            "id"
        ]:

            if field not in group:
                continue

            data = np.asarray(group[field])

            print(f"\n{field}")
            print("Shape:", data.shape)

            if field in ["width_mm", "height_mm"]:
                unique, counts = np.unique(
                    data,
                    return_counts=True
                )

                print("Unique dimensions and counts:")

                for value, count in zip(unique[:20], counts[:20]):
                    print(f"  {value}: {count}")

            else:
                print("First 10 values:", data.ravel()[:10])

print("\n" + "=" * 80)
print("Inspection complete. No files modified.")

In [ ]:
# STEP 198J — INSPECT INFLUENCE MATRIX DOWNSAMPLING
# Read-only. No files modified.

from pathlib import Path
import ast

print("STEP 198J — INFLUENCE MATRIX DOWNSAMPLING")
print("=" * 80)

source_path = (
    Path("/content/portpy_source_inspection")
    / "portpy/photon/influence_matrix.py"
)

source = source_path.read_text(encoding="utf-8")
lines = source.splitlines()
tree = ast.parse(source)

keywords = (
    "down_sample",
    "beamlet",
    "create_BEV",
    "create_influence"
)

methods = []

for node in ast.walk(tree):
    if isinstance(node, ast.FunctionDef):
        if any(k.lower() in node.name.lower() for k in keywords):
            methods.append(node)

print("\nRELEVANT METHODS")
print("-" * 80)

for node in methods:
    print(
        f"{node.name:<48} "
        f"lines {node.lineno}-{node.end_lineno}"
    )

print("\nDOWNSAMPLING IMPLEMENTATION")
print("-" * 80)

priority = {
    "create_down_sample",
    "down_sample_2d_grid"
}

for node in methods:
    if node.name not in priority:
        continue

    print(f"\nFUNCTION: {node.name}")
    print("-" * 70)

    function_lines = lines[node.lineno - 1:node.end_lineno]

    for line in function_lines[:160]:
        print(line)

    if len(function_lines) > 160:
        print(
            f"\n... {len(function_lines)-160} "
            "additional lines omitted ..."
        )

print("\n" + "=" * 80)
print("Inspection complete. No files modified.")

In [ ]:
# STEP 198K — INSPECT BEAM PREPROCESSING AND INDEXING
# Read-only. No files modified.

from pathlib import Path
import ast

print("STEP 198K — BEAM PREPROCESSING AND INDEXING")
print("=" * 80)

source_path = (
    Path("/content/portpy_source_inspection")
    / "portpy/photon/influence_matrix.py"
)

source = source_path.read_text(encoding="utf-8")
lines = source.splitlines()
tree = ast.parse(source)

target_names = {
    "preprocess_beams",
    "pre_process_beams",
    "sort_beamlets"
}

found = []

for node in ast.walk(tree):
    if isinstance(node, ast.FunctionDef):
        if node.name in target_names:
            found.append(node)

if not found:
    print("No matching methods found.")
    print("\nMethods containing 'beam' or 'process':")

    for node in ast.walk(tree):
        if isinstance(node, ast.FunctionDef):
            if "beam" in node.name or "process" in node.name:
                print(node.name, node.lineno, node.end_lineno)

for node in found:

    print("\n" + "-" * 80)
    print("FUNCTION:", node.name)
    print("LINES:", node.lineno, "-", node.end_lineno)
    print("-" * 80)

    function_lines = lines[node.lineno - 1:node.end_lineno]

    for line in function_lines[:240]:
        print(line)

    if len(function_lines) > 240:
        print(
            f"\n... {len(function_lines) - 240} "
            "additional lines omitted ..."
        )

print("\n" + "=" * 80)
print("Inspection complete. No files modified.")

In [ ]:
# STEP 198L — INSPECT INFLUENCE MATRIX DEFAULT SETTINGS
# Read-only. No files modified.

from pathlib import Path
import ast

print("STEP 198L — INFLUENCE MATRIX DEFAULT SETTINGS")
print("=" * 75)

path = (
    Path("/content/portpy_source_inspection")
    / "portpy/photon/influence_matrix.py"
)

source = path.read_text(encoding="utf-8")
lines = source.splitlines()
tree = ast.parse(source)

for node in ast.walk(tree):
    if isinstance(node, ast.ClassDef) and node.name == "InfluenceMatrix":

        for method in node.body:
            if isinstance(method, ast.FunctionDef) and method.name == "__init__":

                print("\nINFLUENCE MATRIX CONSTRUCTOR")
                print("-" * 75)

                for line in lines[method.lineno - 1:method.end_lineno]:
                    print(line)

                break

print("\nDEFAULT-SETTING REFERENCES")
print("-" * 75)

terms = [
    "opt_beamlets_PTV_margin_mm",
    "beamlet_width_mm",
    "beamlet_height_mm",
    "remove_corner_beamlets"
]

for number, line in enumerate(lines, start=1):

    if any(term in line for term in terms):
        if number <= 210:
            print(f"{number:4}: {line}")

print("\n" + "=" * 75)
print("Inspection complete. No files modified.")

In [ ]:
# STEP 198M-FIX — INSTALL MISSING PORTPY DEPENDENCY

%pip install -q SimpleITK

import SimpleITK as sitk

print("SimpleITK installed successfully.")
print("Version:", sitk.Version_VersionString())

In [ ]:
# STEP 198M-FIX 2 — CORRECT PORTPY DATA DIRECTORY

from pathlib import Path

data_root_portpy = project

print("Correct dataset root:", data_root_portpy)

data = pp.DataExplorer(data_dir=str(data_root_portpy))
data.patient_id = "Patient_1"

print("Expected patient directory:")
print(data_root_portpy / "Patient_1")

print("\nChecking required metadata:")
required_file = data_root_portpy / "Patient_1" / "StructureSet_MetaData.json"

print("Structure metadata exists:", required_file.exists())

if not required_file.exists():
    print("\nMetadata not found. Showing available files:")
    patient_folder = data_root_portpy / "Patient_1"

    for item in sorted(patient_folder.iterdir()):
        print(item.name)
else:
    print("\nPath correction verified.")

In [ ]:
# STEP 198M-FIX 3 — INSPECT PATIENT 1 DATA LOCATIONS
# Read-only. No files modified.

from pathlib import Path

patient = project / "Patient_1"

print("STEP 198M-FIX 3 — PATIENT 1 FILE LOCATIONS")
print("=" * 70)

folders = [
    patient,
    patient / "raw_portpy",
    project / "_portpy_download_cache",
]

for folder in folders:
    print(f"\nFOLDER: {folder}")
    print("Exists:", folder.exists())

    if folder.exists():
        items = sorted(folder.iterdir())

        for item in items[:35]:
            print("  ", "[DIR]" if item.is_dir() else "[FILE]", item.name)

        if len(items) > 35:
            print(f"  ... and {len(items) - 35} more items")

print("\nREQUIRED METADATA SEARCH")
print("-" * 70)

names = [
    "StructureSet_MetaData.json",
    "CT_MetaData.json",
    "OptimizationVoxels_MetaData.json",
    "PlannerBeams.json",
]

for name in names:
    matches = list(project.rglob(name))

    print(f"\n{name}: {len(matches)} match(es)")

    for match in matches[:10]:
        print(" ", match.relative_to(project))

print("\nInspection complete. No files modified.")

In [ ]:
# STEP 198N — CHECK OPTIMIZATION VOXEL REQUIREMENTS
# Read-only. No downloads or file modifications.

from pathlib import Path
import ast

print("STEP 198N — OPTIMIZATION VOXEL REQUIREMENTS")
print("=" * 75)

root = Path("/content/portpy_source_inspection/portpy/photon")

targets = {
    "data_explorer.py": ["load_metadata"],
    "structures.py": ["__init__"],
}

for filename, methods in targets.items():
    path = root / filename
    source = path.read_text(encoding="utf-8")
    lines = source.splitlines()
    tree = ast.parse(source)

    print(f"\nFILE: {filename}")

    for node in ast.walk(tree):
        if isinstance(node, ast.FunctionDef) and node.name in methods:
            print(f"\nFUNCTION: {node.name}")
            print("-" * 65)

            for line in lines[node.lineno - 1:node.end_lineno][:115]:
                print(line)

patient_raw = project / "Patient_1" / "raw_portpy"

print("\nPATIENT 1 VOXEL-RELATED FILES")
print("-" * 65)

for path in sorted(patient_raw.iterdir()):
    if any(term in path.name.lower() for term in
           ["voxel", "structure", "ct_", "planner"]):
        print(path.name)

print("\nPATIENT 2 VOXEL-RELATED FILES FOR COMPARISON")
print("-" * 65)

patient2_raw = project / "Patient_2" / "raw_portpy"

if patient2_raw.exists():
    for path in sorted(patient2_raw.iterdir()):
        if "voxel" in path.name.lower():
            print(path.name)

print("\nInspection complete. No files modified.")

In [ ]:
# STEP 198O — LOCATE MISSING PATIENT 1 FILES
# Read-only. Does not download or modify files.

from huggingface_hub import list_repo_files

print("STEP 198O — OFFICIAL PATIENT 1 FILE CHECK")
print("=" * 75)

repo_id = "PortPy-Project/PortPy_Dataset"

files = list_repo_files(
    repo_id=repo_id,
    repo_type="dataset"
)

patient1_files = [
    name for name in files
    if "Prostate_Patient_1/" in name
]

print("Patient 1 files listed:", len(patient1_files))

print("\nOPTIMIZATION VOXEL FILES")
print("-" * 65)

voxel_files = [
    name for name in patient1_files
    if "OptimizationVoxels" in name
]

if voxel_files:
    for name in voxel_files:
        print(name)
else:
    print("No optimization voxel files found.")

print("\nBEAM FOLDER STRUCTURE")
print("-" * 65)

beam_files = [
    name for name in patient1_files
    if "Beam_36_" in name
]

for name in beam_files[:10]:
    print(name)

print("\nOTHER PATIENT 1 METADATA")
print("-" * 65)

for name in patient1_files:
    if name.endswith(".json") and "Beam_" not in name:
        print(name)

print("\n" + "=" * 75)
print("Inspection complete. No files downloaded.")

In [ ]:
# STEP 198P — DOWNLOAD OFFICIAL PATIENT 1 OPTIMIZATION VOXELS
# Downloads only two missing files.
# Does not overwrite existing files.

from pathlib import Path
from huggingface_hub import hf_hub_download
import shutil

repo_id = "PortPy-Project/PortPy_Dataset"

patient_raw = project / "Patient_1" / "raw_portpy"
patient_raw.mkdir(parents=True, exist_ok=True)

required_files = [
    "OptimizationVoxels_MetaData.json",
    "OptimizationVoxels_Data.h5",
]

print("STEP 198P — PATIENT 1 OPTIMIZATION VOXELS")
print("=" * 75)

for filename in required_files:

    destination = patient_raw / filename

    if destination.exists():
        print(f"ALREADY PRESENT: {filename}")
        continue

    remote_path = f"data/Prostate_Patient_1/{filename}"

    print(f"\nDownloading: {filename}")

    downloaded_path = hf_hub_download(
        repo_id=repo_id,
        repo_type="dataset",
        filename=remote_path,
    )

    shutil.copy2(downloaded_path, destination)

    size_mb = destination.stat().st_size / (1024 ** 2)

    print(f"SAVED: {destination}")
    print(f"SIZE: {size_mb:.2f} MB")

print("\nFINAL VERIFICATION")
print("-" * 75)

for filename in required_files:
    path = patient_raw / filename
    print(f"{filename}: {'FOUND' if path.is_file() else 'MISSING'}")

print("\nSTEP 198P COMPLETE")

In [ ]:
# STEP 198Q — PREPARE PORTPY-COMPATIBLE PATIENT 1 DIRECTORY
# Creates temporary symbolic links only.
# Does not modify original Google Drive files.

from pathlib import Path
import os

print("STEP 198Q — PREPARE PORTPY PATIENT 1")
print("=" * 75)

source = project / "Patient_1" / "raw_portpy"

dataset_root = Path("/content/portpy_patient1_dataset")
patient_dir = dataset_root / "Patient_1"
beams_dir = patient_dir / "Beams"

beams_dir.mkdir(parents=True, exist_ok=True)

linked = 0
skipped = 0

for file in sorted(source.iterdir()):

    if not file.is_file():
        continue

    if file.name.startswith("Beam_"):
        destination = beams_dir / file.name
    else:
        destination = patient_dir / file.name

    if destination.is_symlink():
        if destination.resolve() == file.resolve():
            skipped += 1
            continue
        destination.unlink()

    elif destination.exists():
        raise FileExistsError(
            f"Unexpected existing file: {destination}"
        )

    os.symlink(str(file.resolve()), str(destination))
    linked += 1

print("New symbolic links:", linked)
print("Already linked:", skipped)

print("\nREQUIRED FILE CHECK")
print("-" * 75)

required = [
    patient_dir / "StructureSet_MetaData.json",
    patient_dir / "StructureSet_Data.h5",
    patient_dir / "OptimizationVoxels_MetaData.json",
    patient_dir / "OptimizationVoxels_Data.h5",
    patient_dir / "CT_MetaData.json",
    patient_dir / "PlannerBeams.json",
]

for file in required:
    print(
        "PASS" if file.is_file() else "MISSING",
        file.relative_to(dataset_root)
    )

beam_metadata = list(beams_dir.glob("Beam_*_MetaData.json"))
beam_data = list(beams_dir.glob("Beam_*_Data.h5"))

print("\nBeam metadata files:", len(beam_metadata))
print("Beam HDF5 files:", len(beam_data))

print("\nDATASET ROOT:", dataset_root)

if (
    all(file.is_file() for file in required)
    and len(beam_metadata) == 9
    and len(beam_data) == 9
):
    print("\nSTEP 198Q PASSED")
else:
    print("\nSTEP 198Q INCOMPLETE — REVIEW MISSING FILES")

In [ ]:
# STEP 198R — TEST PATIENT 1 DEFAULT BEAMLET SELECTION
# No optimization or model training.
# May take time and use substantial RAM.

import sys
import numpy as np
from pathlib import Path

source_root = Path("/content/portpy_source_inspection")
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

import portpy.photon as pp

print("STEP 198R — PATIENT 1 BEAMLET SELECTION")
print("=" * 75)

dataset_root = Path("/content/portpy_patient1_dataset")

data = pp.DataExplorer(data_dir=str(dataset_root))
data.patient_id = "Patient_1"

print("\nLoading patient structures...")
structs = pp.Structures(data)

print("\nLoading patient beams...")
beams = pp.Beams(data)

print("\nLoading CT...")
ct = pp.CT(data)

print("\nConstructing default PortPy influence matrix...")

inf_matrix = pp.InfluenceMatrix(
    structs=structs,
    beams=beams,
    ct=ct,
    target_structure="PTV",
    opt_beamlets_PTV_margin_mm=3,
    is_full=False
)

selected = inf_matrix.A.shape[1]

saved_path = (
    project
    / "Patient_1"
    / "optimized_portpy"
    / "optimal_intensity.npy"
)

saved = np.load(saved_path, mmap_mode="r")

print("\nRESULTS")
print("-" * 75)

print("PortPy selected beamlets:", selected)
print("Saved intensity vector length:", saved.size)

print("\nPER-BEAM COUNTS")

total = 0

for beam in inf_matrix.beamlets_dict:
    start = int(beam["start_beamlet_idx"])
    end = int(beam["end_beamlet_idx"])

    count = end - start + 1
    total += count

    print(
        f"Beam {beam['beam_id']}: "
        f"{count} beamlets "
        f"(indices {start}–{end})"
    )

print("\nTotal from beam dictionaries:", total)

print("\nCOMPARISON")
print("-" * 75)

if selected == saved.size and total == selected:
    print("MATCH: Default selection produces the saved vector length.")
    print("Spatial mapping still requires verification.")
else:
    print("NO EXACT MATCH.")
    print("The saved optimization may use different settings.")
    print("Do not assume the original configuration.")

print("\nSTEP 198R COMPLETE")

In [ ]:
# STEP 198S — VALIDATE PATIENT 1 FLUENCE SPATIAL MAPPING
# Read-only. No optimization or model training.

import numpy as np
from pathlib import Path

print("STEP 198S — FLUENCE SPATIAL RECONSTRUCTION")
print("=" * 75)

saved_dir = project / "Patient_1" / "optimized_portpy"

intensity = np.load(saved_dir / "optimal_intensity.npy")

assert intensity.size == inf_matrix.A.shape[1], (
    "Intensity length does not match influence matrix."
)

beam_ids = [
    int(beam["beam_id"])
    for beam in inf_matrix.beamlets_dict
]

print("Reconstructing fluence maps using PortPy...")

reconstructed = inf_matrix.fluence_1d_to_2d(
    fluence_1d=intensity
)

print("\nBEAM-BY-BEAM COMPARISON")
print("-" * 75)

all_match = True

for beam_id, predicted in zip(beam_ids, reconstructed):

    saved_path = saved_dir / f"beam_{beam_id}_fluence.npy"
    reference = np.load(saved_path)

    predicted = np.asarray(predicted)
    reference = np.asarray(reference)

    same_shape = predicted.shape == reference.shape

    if same_shape:
        max_error = float(
            np.max(np.abs(predicted - reference))
        )
        exact_match = np.array_equal(predicted, reference)
    else:
        max_error = float("nan")
        exact_match = False

    all_match = all_match and exact_match

    print(
        f"Beam {beam_id}: "
        f"reconstructed={predicted.shape}, "
        f"saved={reference.shape}, "
        f"exact_match={exact_match}, "
        f"max_error={max_error}"
    )

print("\nFINAL RESULT")
print("-" * 75)

if all_match and len(reconstructed) == len(beam_ids) == 9:
    print("PASS: All nine fluence maps match exactly.")
    print("Spatial indexing has been verified.")
else:
    print("NOT YET VERIFIED.")
    print("Investigate any shape or numerical differences.")

print("\nSTEP 198S COMPLETE")

In [ ]:
# STEP 198T — SAVE PATIENT 1 FLUENCE VALIDATION REPORT
# Writes a small JSON report; does not modify source data.

import json
import numpy as np
from datetime import datetime, timezone

report_dir = project / "Stage2_Fluence_Validation"
report_dir.mkdir(parents=True, exist_ok=True)

intensity = np.load(
    project / "Patient_1" / "optimized_portpy" / "optimal_intensity.npy"
)

reconstructed = inf_matrix.fluence_1d_to_2d(
    fluence_1d=intensity
)

beam_results = []

for beam, reconstructed_map in zip(
    inf_matrix.beamlets_dict, reconstructed
):
    beam_id = int(beam["beam_id"])

    reference = np.load(
        project / "Patient_1" / "optimized_portpy"
        / f"beam_{beam_id}_fluence.npy"
    )

    reconstructed_map = np.asarray(reconstructed_map)

    beam_results.append({
        "beam_id": beam_id,
        "shape": list(reference.shape),
        "exact_match": bool(
            np.array_equal(reconstructed_map, reference)
        ),
        "max_absolute_error": float(
            np.max(np.abs(reconstructed_map - reference))
        )
    })

report = {
    "patient": "Patient_1",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "method": "PortPy InfluenceMatrix.fluence_1d_to_2d",
    "beamlet_selection": {
        "target_structure": "PTV",
        "ptv_margin_mm": 3,
        "beamlet_width_mm": float(inf_matrix.beamlet_width_mm),
        "beamlet_height_mm": float(inf_matrix.beamlet_height_mm)
    },
    "selected_beamlets": int(inf_matrix.A.shape[1]),
    "saved_intensity_count": int(intensity.size),
    "beam_results": beam_results,
    "all_exact": (
        len(beam_results) == 9
        and all(item["exact_match"] for item in beam_results)
    ),
    "limitations": [
        "Saved intensity optimization provenance not yet verified",
        "Full two-stage FluenceFormer reproduction not yet completed"
    ]
}

output = report_dir / "Patient_1_Fluence_Reconstruction_Validation.json"

with open(output, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print("STEP 198T — VALIDATION REPORT")
print("=" * 65)
print("Saved:", output)
print("Maps verified:", len(beam_results))
print("All exact:", report["all_exact"])
print("Selected beamlets:", report["selected_beamlets"])
print("Intensity count:", report["saved_intensity_count"])
print("STEP 198T COMPLETE")

In [ ]:
# STEP 199A — COMPLETE FLUENCEFORMER PROJECT INVENTORY
# READ-ONLY: Does not modify, move or delete files.

from pathlib import Path
from collections import Counter
import os

project = Path(
    "/content/drive/MyDrive/FluenceFormer_PortPy_Reproduction"
)

print("=" * 80)
print("STEP 199A — FLUENCEFORMER PROJECT INVENTORY")
print("=" * 80)

if not project.is_dir():
    raise FileNotFoundError(
        f"Project folder not found: {project}"
    )

print("\nPROJECT ROOT:")
print(project)

# 1. Identify top-level project folders
print("\n" + "=" * 80)
print("1. TOP-LEVEL PROJECT CONTENTS")
print("=" * 80)

for item in sorted(project.iterdir()):
    if item.is_dir():
        print(f"[FOLDER] {item.name}")
    else:
        print(f"[FILE]   {item.name}")

# 2. Inventory all files
files = []

for root, dirs, filenames in os.walk(project):
    dirs[:] = [
        d for d in dirs
        if d not in {".git", ".cache", "__pycache__"}
    ]

    for filename in filenames:
        path = Path(root) / filename

        try:
            size = path.stat().st_size
        except OSError:
            continue

        files.append((path, size))

print("\n" + "=" * 80)
print("2. FILE SUMMARY")
print("=" * 80)

print("Total files:", len(files))
print(
    "Total size:",
    round(sum(size for _, size in files) / (1024**3), 2),
    "GB"
)

extensions = Counter(
    path.suffix.lower() or "[no extension]"
    for path, _ in files
)

print("\nFile types:")
for extension, count in extensions.most_common(20):
    print(f"  {extension:12} {count}")

# 3. Code and notebooks
print("\n" + "=" * 80)
print("3. CODE AND COLAB NOTEBOOKS")
print("=" * 80)

code_extensions = {
    ".py", ".ipynb", ".sh", ".yaml",
    ".yml", ".toml", ".json", ".md"
}

code_files = [
    (path, size)
    for path, size in files
    if path.suffix.lower() in code_extensions
    and "_portpy_download_cache" not in path.parts
]

for path, size in sorted(code_files)[:120]:
    relative = path.relative_to(project)
    print(f"{relative} ({size / 1024:.1f} KB)")

if len(code_files) > 120:
    print(f"... and {len(code_files) - 120} more")

# 4. Models and checkpoints
print("\n" + "=" * 80)
print("4. TRAINED MODELS AND CHECKPOINTS")
print("=" * 80)

model_extensions = {
    ".pt", ".pth", ".ckpt", ".safetensors"
}

models = [
    (path, size)
    for path, size in files
    if path.suffix.lower() in model_extensions
]

for path, size in sorted(models):
    print(
        path.relative_to(project),
        f"({size / (1024**2):.1f} MB)"
    )

# 5. Reports and presentations
print("\n" + "=" * 80)
print("5. REPORTS AND PRESENTATIONS")
print("=" * 80)

document_extensions = {
    ".pdf", ".pptx", ".docx",
    ".xlsx", ".csv", ".txt"
}

documents = [
    (path, size)
    for path, size in files
    if path.suffix.lower() in document_extensions
]

for path, size in sorted(documents)[:100]:
    print(
        path.relative_to(project),
        f"({size / 1024:.1f} KB)"
    )

# 6. Large files
print("\n" + "=" * 80)
print("6. LARGEST FILES — GOOGLE DRIVE CANDIDATES")
print("=" * 80)

for path, size in sorted(
    files, key=lambda x: x[1], reverse=True
)[:20]:
    print(
        f"{size / (1024**3):7.3f} GB",
        path.relative_to(project)
    )

# 7. Check critical known outputs
print("\n" + "=" * 80)
print("7. CRITICAL PROJECT FILES")
print("=" * 80)

critical_files = [
    "Stage1_SwinUNETR_Full_Training/best_model.pt",
    "Patient_1/optimized_portpy/optimal_intensity.npy",
    "Stage2_Fluence_Validation/"
    "Patient_1_Fluence_Reconstruction_Validation.json"
]

for relative in critical_files:
    path = project / relative
    print(
        "FOUND  " if path.is_file() else "MISSING",
        relative
    )

print("\n" + "=" * 80)
print("STEP 199A COMPLETE")
print("No files modified.")
print("=" * 80)

In [ ]:
# STEP 199B — FIND FLUENCEFORMER NOTEBOOKS AND SOURCE CODE
# READ-ONLY. Does not modify, move or delete files.

from pathlib import Path
import os

print("=" * 80)
print("STEP 199B — LOCATE COLAB NOTEBOOKS AND SOURCE CODE")
print("=" * 80)

drive_root = Path("/content/drive/MyDrive")

keywords = [
    "fluence",
    "portpy",
    "capstone",
    "com6000",
    "swinunetr",
    "stage1",
    "stage2",
    "radiotherapy",
]

code_extensions = {
    ".ipynb", ".py", ".sh", ".md",
    ".yaml", ".yml", ".toml"
}

all_notebooks = []
matching_code = []

excluded_dirs = {
    ".git",
    ".cache",
    "__pycache__",
    "node_modules",
    ".ipynb_checkpoints",
}

print("\nSearching Google Drive...")
print("This may take a little time.\n")

for root, dirs, filenames in os.walk(drive_root):

    dirs[:] = [
        d for d in dirs
        if d not in excluded_dirs
    ]

    for filename in filenames:

        path = Path(root) / filename
        extension = path.suffix.lower()

        if extension not in code_extensions:
            continue

        relative = str(path.relative_to(drive_root))
        name_lower = relative.lower()

        if extension == ".ipynb":
            all_notebooks.append(path)

        if any(keyword in name_lower for keyword in keywords):
            matching_code.append(path)

print("=" * 80)
print("1. NOTEBOOKS FOUND IN GOOGLE DRIVE")
print("=" * 80)

print("Total notebooks:", len(all_notebooks))

for path in sorted(all_notebooks)[:100]:
    print(path.relative_to(drive_root))

if len(all_notebooks) > 100:
    print(f"... and {len(all_notebooks) - 100} more")

print("\n" + "=" * 80)
print("2. PROJECT-RELATED CODE FILES")
print("=" * 80)

print("Matching files:", len(matching_code))

for path in sorted(matching_code)[:100]:
    print(path.relative_to(drive_root))

if len(matching_code) > 100:
    print(f"... and {len(matching_code) - 100} more")

print("\n" + "=" * 80)
print("3. CURRENT COLAB RUNTIME CODE")
print("=" * 80)

runtime_folders = [
    Path("/content"),
    Path("/content/portpy_source_inspection"),
]

for folder in runtime_folders:
    print(
        "FOUND" if folder.exists() else "MISSING",
        folder
    )

print("\n" + "=" * 80)
print("STEP 199B COMPLETE")
print("No files modified.")
print("=" * 80)

In [ ]:
# STEP 199C — INSPECT MAIN FLUENCEFORMER NOTEBOOK
# READ-ONLY. Does not execute or modify the notebook.

from pathlib import Path
import json
from collections import Counter

notebook_path = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "FluenceFormer_PortPy_TwoStage_Reproduction.ipynb"
)

print("=" * 80)
print("STEP 199C — MAIN NOTEBOOK INSPECTION")
print("=" * 80)

if not notebook_path.is_file():
    raise FileNotFoundError(notebook_path)

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

cells = notebook.get("cells", [])

print("\nNOTEBOOK:", notebook_path.name)
print("SIZE (MB):", round(notebook_path.stat().st_size / 1024**2, 2))
print("TOTAL CELLS:", len(cells))

cell_types = Counter(
    cell.get("cell_type", "unknown")
    for cell in cells
)

print("CELL TYPES:", dict(cell_types))

print("\n" + "=" * 80)
print("1. NOTEBOOK HEADINGS")
print("=" * 80)

headings = []

for index, cell in enumerate(cells, start=1):
    if cell.get("cell_type") != "markdown":
        continue

    source = "".join(cell.get("source", []))

    for line in source.splitlines():
        if line.strip().startswith("#"):
            headings.append((index, line.strip()))

for index, heading in headings[:80]:
    print(f"Cell {index}: {heading[:150]}")

if len(headings) > 80:
    print(f"... and {len(headings) - 80} more headings")

print("\n" + "=" * 80)
print("2. PROJECT EXPERIMENT REFERENCES")
print("=" * 80)

keywords = [
    "SwinUNETR",
    "FluenceFormer",
    "PortPy",
    "Stage 1",
    "Stage 2",
    "fluence_1d_to_2d",
    "198S",
    "198T",
    "optimization",
    "training_history",
]

for keyword in keywords:
    matching = []

    for index, cell in enumerate(cells, start=1):
        source = "".join(cell.get("source", []))

        if keyword.lower() in source.lower():
            matching.append(index)

    print(
        f"{keyword:22}: "
        f"{len(matching):4} cells "
        f"(latest: {matching[-1] if matching else 'none'})"
    )

print("\n" + "=" * 80)
print("3. LAST 15 CODE CELLS — SHORT PREVIEW")
print("=" * 80)

code_cells = [
    (index, cell)
    for index, cell in enumerate(cells, start=1)
    if cell.get("cell_type") == "code"
]

for index, cell in code_cells[-15:]:
    source = "".join(cell.get("source", []))
    lines = [
        line.strip()
        for line in source.splitlines()
        if line.strip()
    ]

    preview = lines[0][:130] if lines else "[empty]"

    print(f"Cell {index}: {preview}")

print("\n" + "=" * 80)
print("4. OUTPUT AND EXECUTION SUMMARY")
print("=" * 80)

with_outputs = sum(
    bool(cell.get("outputs"))
    for cell in code_cells
)

print("Code cells:", len(code_cells))
print("Code cells with saved outputs:", with_outputs)

print("\n" + "=" * 80)
print("STEP 199C COMPLETE")
print("No files modified.")
print("=" * 80)

In [ ]:
# STEP 199D — NOTEBOOK COMPLETENESS AND SHARING SAFETY
# READ-ONLY. No files modified or uploaded.

import json
import re
from pathlib import Path
from collections import Counter

notebook_path = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "FluenceFormer_PortPy_TwoStage_Reproduction.ipynb"
)

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

cells = nb.get("cells", [])
code_cells = [
    (i, cell)
    for i, cell in enumerate(cells, start=1)
    if cell.get("cell_type") == "code"
]

print("=" * 75)
print("STEP 199D — NOTEBOOK COMPLETENESS AND SHARING SAFETY")
print("=" * 75)

print("\n1. NOTEBOOK CONTENT")
print("Total cells:", len(cells))
print("Code cells:", len(code_cells))

with_outputs = [
    i for i, cell in code_cells
    if cell.get("outputs")
]

with_errors = [
    i for i, cell in code_cells
    if any(
        output.get("output_type") == "error"
        for output in cell.get("outputs", [])
    )
]

print("Cells with saved outputs:", len(with_outputs))
print("Cells with saved errors:", len(with_errors))
print("Saved error cell numbers:", with_errors[:30])

print("\n2. OUTPUT TYPES")

output_types = Counter(
    output.get("output_type", "unknown")
    for _, cell in code_cells
    for output in cell.get("outputs", [])
)

for name, count in output_types.items():
    print(f"{name}: {count}")

print("\n3. EMBEDDED OUTPUT FORMATS")

mime_types = Counter(
    mime
    for _, cell in code_cells
    for output in cell.get("outputs", [])
    for mime in output.get("data", {})
)

for name, count in mime_types.most_common(15):
    print(f"{name}: {count}")

print("\n4. POSSIBLE CREDENTIAL REFERENCES")
print("(Only cell numbers and categories are displayed.)")

patterns = {
    "API key assignment": r"(?i)\bapi[_-]?key\s*=",
    "Token assignment": r"(?i)\b(?:access[_-]?token|auth[_-]?token)\s*=",
    "Password assignment": r"(?i)\bpassword\s*=",
    "Colab authentication": r"auth\.authenticate_user",
    "GitHub authentication": r"(?i)(?:github_pat_|ghp_)",
    "Hugging Face authentication": r"(?i)notebook_login|huggingface_hub\.login",
}

for label, pattern in patterns.items():
    matches = [
        i for i, cell in code_cells
        if re.search(
            pattern,
            "".join(cell.get("source", []))
        )
    ]

    print(f"{label}: {len(matches)} cell(s)")
    if matches:
        print("  Cell numbers:", matches[:20])

print("\n5. FINAL EXPERIMENT CHECK")

required_steps = [
    "198R",
    "198S",
    "198T",
    "199A",
    "199B",
]

for step in required_steps:
    matches = [
        i for i, cell in code_cells
        if re.search(
            rf"(?i)\bSTEP\s+{step}\b",
            "".join(cell.get("source", []))
        )
    ]

    print(
        f"Step {step}:",
        "FOUND" if matches else "NOT FOUND",
        matches[:5]
    )

print("\n" + "=" * 75)
print("STEP 199D COMPLETE")
print("No files modified.")